In [52]:
import sympy
from sympy import *
import numpy as np

In [53]:
# Define the letters that we will use to create the expressions to be processed by Sympy

x,a,b,c,d = symbols('x a b c d')


In [54]:
'''
In this function, we run Newton approximation to find the root of a differentiable function closest to a prescribed value of $x$.
'''
def newton_approx(

           # The Sympy expression of the function for which we are running Newton's method on
           function, 

           # The Sympy expression for the derivative of the function above
           deriv,

           # The initial value of $x$ for Newton's algorithm
           starting_x, 

           # Max number of iterations
           N,

           # Margin of error (for reciprocal)
           epsilon):
    
    # We store the approximation found in each iteration of Newton's algorithm in an array. 
    cumulative = [starting_x]
    
    for _iteration in range(N):
            
        f_0 = function.subs([(x, cumulative[-1])]).evalf()
        
        df_0 = deriv.subs([(x, cumulative[-1])]).evalf()

        # Obtain the next value from Newton's algorithm
        new_value = (cumulative[-1] - f_0/df_0).evalf()

        # We get an error if new_value does not lie in the domain of function
        if(new_value.is_real == False or new_value == sympy.nan):
            print(f"Error in newton_approx: new_value is not real number. function = {function}, cumulated values = {cumulative}")
            return
        else:
            # If new_value is valid, we append it to our array of values found by Newton's algorithm
            cumulative.append(new_value)

        xm = 1/(1/cumulative[-1] - epsilon)
        xp = 1/(1/cumulative[-1] + epsilon)

        if(xm > 0):
            f_m = function.subs([(x, xm)]).evalf()
            f_p = function.subs([(x, xp)]).evalf()
            # We use the intermediate value theorem to certify that we have found a root within the specified margin
            if((f_m < 0 and f_p > 0) or (f_m > 0 and f_p < 0)):
                return cumulative[-1]

    # Message if max number of iterations exceeded without getting a certified approximation of a root
    print(f"Problem with newton_approx: Could not certify root within margin in {_iteration} iterations, function = {function}, f_0 = {f_0}, f_m = {f_m}, f_p = {f_p}, x_0 = {cumulative[-1]}, x_m = {xm}, x_p = {xp}, returned an approximated root")
    return cumulative[-1]

In [55]:
'''
In this function, we find the minimum PF root of a clique polynomial.
From the given basepoint a_start,..., the function iteratively:
- Checks that the basepoint is 'optimal enough', in the sense that the root and parameters at this point 
can be certified to be within the specified margin of the actual minimum, if yes, the loop is terminated
- If not, prepare a grid centered at the basepoint, evaluate the root at each gridpoint,
and move the basepoint to the gridpoint with the minimum PF root
'''
def MinPFRootApprox(function,
                    starting_x = 0.01, 
                    newton_iterations = 20,
                    gridsize = 4,
                    margin_x = 0.01,
                    margin_param = 0.001,
                    a_start = 0.5,
                    a_range = 0.1,
                    b_start = 0.5,
                    b_range = 0.1,
                    c_start = 0.5,
                    c_range = 0.1,
                    d_start = 0.5,
                    d_range = 0.1):

    deriv = diff(function, x)
    
    # This variable records the largest least positive root we've found so far while varying $a$,...
    cumulative_max = 0

    # These variables record the values of $a$,... that gave rise to the above root
    max_param_a = 0
    max_param_b = 0
    max_param_c = 0
    max_param_d = 0

    a_center = a_start
    a_min = a_start
    a_max = a_start
    a_step = 1
    
    b_center = b_start
    b_min = b_start
    b_max = b_start
    b_step = 1
    
    c_center = c_start
    c_min = c_start
    c_max = c_start
    c_step = 1
    
    d_center = d_start
    d_min = d_start
    d_max = d_start
    d_step = 1
    
    margin_newton = margin_x/10

    for j in range(10):

        function_sub = function.subs([(a, a_center), (b, b_center), (c, c_center), (d, d_center)])
        deriv_sub = deriv.subs([(a, a_center), (b, b_center), (c, c_center), (d, d_center)])
        root = newton_approx(function_sub, deriv_sub, starting_x, newton_iterations, margin_newton)
        center_value = 1/root

        # If there are no parameters, then we just need to run Newton's method once
        if (function.free_symbols == {x}):
            print(f"The minimum PF zero is {center_value} within margin of {margin_newton} \n")
            return

        # Setting up the grid
        if(a in function.free_symbols):
            a_min = max(0,a_center-a_range/(2**j))
            a_max = min(1,a_center+a_range/(2**j))
            a_step = (a_max-a_min)/gridsize
        if(b in function.free_symbols):
            b_min = max(0,b_center-b_range/(2**j))
            b_max = min(1,b_center+b_range/(2**j))
            b_step = (b_max-b_min)/gridsize
        if(c in function.free_symbols):
            c_min = max(0,c_center-c_range/(2**j))
            c_max = min(2,c_center+c_range/(2**j))
            c_step = (c_max-c_min)/gridsize
        if(d in function.free_symbols):
            d_min = max(0,d_center-d_range/(2**j))
            d_max = min(2,d_center+d_range/(2**j))
            d_step = (d_max-d_min)/gridsize
            
        corner_values = []

        # Computing the corner values
        for s in [-1,1]:
            if(a in function.free_symbols):
                function_sub = function.subs([(a, a_center+s*margin_param), (b, b_center), (c, c_center), (d, d_center)])
                deriv_sub = deriv.subs([(a, a_center+s*margin_param), (b, b_center), (c, c_center), (d, d_center)])
                root = newton_approx(function_sub, deriv_sub, starting_x, newton_iterations, margin_newton)
                corner_values.append(1/root)
            if(b in function.free_symbols):
                function_sub = function.subs([(a, a_center), (b, b_center+s*margin_param), (c, c_center), (d, d_center)])
                deriv_sub = deriv.subs([(a, a_center), (b, b_center+s*margin_param), (c, c_center), (d, d_center)])
                root = newton_approx(function_sub, deriv_sub, starting_x, newton_iterations, margin_newton)
                corner_values.append(1/root)
            if(c in function.free_symbols):
                function_sub = function.subs([(a, a_center), (b, b_center), (c, c_center+s*margin_param), (d, d_center)])
                deriv_sub = deriv.subs([(a, a_center), (b, b_center), (c, c_center+s*margin_param), (d, d_center)])
                root = newton_approx(function_sub, deriv_sub, starting_x, newton_iterations, margin_newton)
                corner_values.append(1/root)
            if(d in function.free_symbols):
                function_sub = function.subs([(a, a_center), (b, b_center), (c, c_center), (d, d_center+s*margin_param)])
                deriv_sub = deriv.subs([(a, a_center), (b, b_center), (c, c_center), (d, d_center+s*margin_param)])
                root = newton_approx(function_sub, deriv_sub, starting_x, newton_iterations, margin_newton)
                corner_values.append(1/root)
        
        # Check if the basepoint is 'optimal enough'
        if((min(corner_values)>center_value+2*margin_newton) and (max(corner_values)<center_value-2*margin_newton+margin_x)):
            print(f"The minimum PF zero is {center_value} within margin of {margin_x}, attained around a={a_center}, b={b_center}, c={c_center}, d={d_center} within margin of {margin_param}\n")
            return
        else:
            if(min(corner_values)>center_value):
                # If the basepoint seems close to the optimal point, but margin_newton is too large for certification, we lower margin_newton
                margin_newton = (min(corner_values) - center_value)/10
        
        # Looping through gridpoints
        for val_a in np.arange(a_min,a_max+a_step,a_step):
            for val_b in np.arange(b_min,b_max+b_step,b_step):
                for val_c in np.arange(c_min,c_max+c_step,c_step):
                    for val_d in np.arange(d_min,d_max+d_step,d_step):

                        function_sub = function.subs([(a, val_a), (b, val_b), (c, val_c), (d, val_d)])
                        deriv_sub = deriv.subs([(a, val_a), (b, val_b), (c, val_c), (d, val_d)])
                        root = newton_approx(function_sub, deriv_sub, starting_x, newton_iterations, margin_newton)
                        
                        # If we encounter invalid values, we exit the loop and return the last valid value
                        if(root == sympy.nan):
                            break
    
                        # If we find a least positive root that is higher than the maximum we've found so far, we record that
                        if(root > 0 and root > cumulative_max):
                            cumulative_max = root
                            max_param_a = val_a
                            max_param_b = val_b
                            max_param_c = val_c
                            max_param_d = val_d

        # Reset basepoint to gridpoint with largest root
        a_center = max_param_a
        b_center = max_param_b
        c_center = max_param_c
        d_center = max_param_d
        
    if(cumulative_max == 0):
        print(f"Error in MinPFRootApprox: Could not find a root \n")
        return
    else:
        print(f"Problem with MinPFRootApprox: Could not get certified root/parameters within margin. Approximations: Zero={center_value}, a={max_param_a}, b={max_param_b}, c={max_param_c}, d={max_param_d}. Corner_values = {corner_values}, margin_x = {margin_x}, margin_newton = {margin_newton}.\n")
        return

In [56]:
Case_I = [   
("Lemma 7.7 - mu meets beta and gamma", 
 ((1 - 2 * (x ** a)) ** 2 - x) * (1 - x ** (1 - 2 * a)) - 2 * (x ** (2 - 2 * a)),
 20.7, 0.4, 1),

("Lemma 7.7 - mu meets beta but not gamma", 
 ((1 - 2 * x ** a) * (1 - 2 * x ** b) - x) * (1 - x ** (1 - a - b)) - 2 * x ** (2 - a - 2 * b) * (1 - 2 * x ** b),
 19.9, 0.45, 0.32), 

("Lemma 7.7 - mu does not meet beta and gamma", 
 ((1 - 2 * x ** a) ** 2 - x) * (1 - x ** (1 - 2 * a)) - 2 * x ** (2 - 4 * a) * (1 - 2 * x ** a) ** 2,
 21.2, 0.34, 1),

("Lemma 7.9", 
 (1 - 2 * x ** a) ** 2 * (1 - x ** (2 * a) - 4 * x) - x ** (1 - 2 * a) * (1 - x ** (2 * a)), 
 68.2, 0.33, 1),
]

In [57]:
for i, (name, function, zero_start, a_start, b_start) in enumerate(Case_I):
    print(f"Evaluating {name}: {function} ...")
    MinPFRootApprox(function, starting_x = 1/zero_start, a_start = a_start, b_start = b_start, a_range = 0.01, b_range = 0.01)

Evaluating Lemma 7.7 - mu meets beta and gamma: -2*x**(2 - 2*a) + (1 - x**(1 - 2*a))*(-x + (1 - 2*x**a)**2) ...
The minimum PF zero is 20.7138211534012 within margin of 0.01, attained around a=0.39749999999999996, b=1, c=0.5, d=0.5 within margin of 0.001

Evaluating Lemma 7.7 - mu meets beta but not gamma: -2*x**(-a - 2*b + 2)*(1 - 2*x**b) + (1 - x**(-a - b + 1))*(-x + (1 - 2*x**a)*(1 - 2*x**b)) ...
The minimum PF zero is 19.9282292364218 within margin of 0.01, attained around a=0.4518749999999999, b=0.31625000000000003, c=0.5, d=0.5 within margin of 0.001

Evaluating Lemma 7.7 - mu does not meet beta and gamma: -2*x**(2 - 4*a)*(1 - 2*x**a)**2 + (1 - x**(1 - 2*a))*(-x + (1 - 2*x**a)**2) ...
The minimum PF zero is 21.2747611932598 within margin of 0.01, attained around a=0.3425000000000001, b=1, c=0.5, d=0.5 within margin of 0.001

Evaluating Lemma 7.9: -x**(1 - 2*a)*(1 - x**(2*a)) + (1 - 2*x**a)**2*(-4*x - x**(2*a) + 1) ...
The minimum PF zero is 68.2660686379711 within margin of 0.01,

In [58]:
Case_II = [
("Lemma A.4.5 - mu meets beta and petal curves other than gamma", 
 (1 - x) * (1 - x ** b) * (1 - x ** (1 - a - 2 * b)) - 2 * x ** a * (1 - x ** b) - 2 * x ** (2 - a - b),
 18.9, 0.49, 0.06),

("Lemma A.4.5 - mu meets petal curves other than gamma, but not beta", 
 (1 - x) * (1 - x ** b) * (1 - x ** (1 - a - 2 * b)) - 2 * x ** a * (1 - x ** b) - 2 * x ** (2 - 2 * a - b) * (1 - 2 * x ** a),
 21.2, 0.38, 0.06),

("Lemma A.4.5 - mu does not meet petal curves other than gamma", 
 (1 - x) * (1 - x ** b) * (1 - x ** (1 - a - 2 * b)) - 2 * x ** a * (1 - x ** b) - 2 * x ** (1 + b) * (1 - 2 * x ** a - x ** (1 - a - 2 * b)), 
 14.5, 0.59, 0.07),
]

In [59]:
for i, (name, function, zero_start, a_start, b_start) in enumerate(Case_II):
    print(f"Evaluating {name}: {function} ...")
    MinPFRootApprox(function, starting_x = 1/zero_start, a_start = a_start, b_start = b_start, a_range = 0.01, b_range = 0.01)

Evaluating Lemma A.4.5 - mu meets beta and petal curves other than gamma: -2*x**a*(1 - x**b) - 2*x**(-a - b + 2) + (1 - x)*(1 - x**b)*(1 - x**(-a - 2*b + 1)) ...
The minimum PF zero is 18.9331965223783 within margin of 0.01, attained around a=0.49, b=0.065, c=0.5, d=0.5 within margin of 0.001

Evaluating Lemma A.4.5 - mu meets petal curves other than gamma, but not beta: -2*x**a*(1 - x**b) - 2*x**(-2*a - b + 2)*(1 - 2*x**a) + (1 - x)*(1 - x**b)*(1 - x**(-a - 2*b + 1)) ...
The minimum PF zero is 21.2775451798003 within margin of 0.01, attained around a=0.37781249999999994, b=0.06312499999999999, c=0.5, d=0.5 within margin of 0.001

Evaluating Lemma A.4.5 - mu does not meet petal curves other than gamma: -2*x**a*(1 - x**b) - 2*x**(b + 1)*(-2*x**a - x**(-a - 2*b + 1) + 1) + (1 - x)*(1 - x**b)*(1 - x**(-a - 2*b + 1)) ...
The minimum PF zero is 14.5670811896825 within margin of 0.01, attained around a=0.5924999999999999, b=0.074375, c=0.5, d=0.5 within margin of 0.001



In [60]:
Case_III = [
("Lemma A.5.4 - mu meets beta", 
 (1 - x - 2 * x ** a) * (1 - x ** b) - x ** (1 - a - b) * (1 - x - 2 * x ** a) - 2 * x ** (2 - a - b),
 16.9, 0.37, 0.24), 

("Lemma A.5.4 - mu does not meet beta", 
 (1 - x - 2 * x ** a) * (1 - x ** b) - x ** (1 - a - b) * (1 - x - 2 * x ** a) - 2 * x ** (2 - 2 * a - b) * (1 - 2 * x ** a),
 14.8, 0.32, 0.26),
]

In [61]:
for i, (name, function, zero_start, a_start, b_start) in enumerate(Case_III):
    print(f"Evaluating {name}: {function} ...")
    MinPFRootApprox(function, starting_x = 1/zero_start, a_start = a_start, b_start = b_start, a_range = 0.01, b_range = 0.01)

Evaluating Lemma A.5.4 - mu meets beta: -x**(-a - b + 1)*(-x - 2*x**a + 1) - 2*x**(-a - b + 2) + (1 - x**b)*(-x - 2*x**a + 1) ...
The minimum PF zero is 17.0000139686639 within margin of 0.01, attained around a=0.36749999999999994, b=0.245, c=0.5, d=0.5 within margin of 0.001

Evaluating Lemma A.5.4 - mu does not meet beta: -2*x**(-2*a - b + 2)*(1 - 2*x**a) - x**(-a - b + 1)*(-x - 2*x**a + 1) + (1 - x**b)*(-x - 2*x**a + 1) ...
The minimum PF zero is 14.8173236579180 within margin of 0.01, attained around a=0.3218750000000001, b=0.25718749999999996, c=0.5, d=0.5 within margin of 0.001



In [62]:
Prop_A_6_2 = [
("Lemma A.6.3 - mu and nu meet beta", 
 (1 - x - 2 * x ** a) * (1 - x ** b) * (1 - x ** (1 - a - 2 * b)) 
 - 4 * x ** (2 - a - b), # mu, mu', nu, nu'
 17.4, 0.43, 0.12),

("Lemma A.6.3 - mu meets beta, nu does not meet beta", 
 (1 - x - 2 * x ** a) * (1 - x ** b) * (1 - x ** (1 - a - 2 * b)) 
 - 2 * x ** (2 - a - b) # mu, mu'
 - 2 * x ** (2 - 2 * a - b) * (1 - 2 * x ** a), # nu, nu'
 17.5, 0.36, 0.14),

("Lemma A.6.3 - mu and nu do not meet beta", 
 (1 - x - 2 * x ** a) * (1 - x ** b) * (1 - x ** (1 - a - 2 * b)) 
 - 4 * x ** (2 - 2 * a - b) * (1 - 2 * x ** a), # mu, mu', nu, nu'
 15.8, 0.32, 0.14),

###
    
("Lemma A.6.4 - mu and nu meet beta", 
 (1 - x - 2 * x ** a) * (1 - x ** b) * (1 - x ** (1 - a - 2 * b)) 
 - 2 * x ** (2 - a - b) # mu, mu'
 - 2 * x ** (1 + b) * (1 - x ** (1 - a - 2 * b)), # nu, nu'
 17.6, 0.46, 0.17),

("Lemma A.6.4 - mu meets beta, nu does not meet beta",
 (1 - x - 2 * x ** a) * (1 - x ** b) * (1 - x ** (1 - a - 2 * b)) 
 - 2 * x ** (2 - a - b) # mu, mu'
 - 2 * x ** (1 - a + b) * (1 - 2 * x ** a) * (1 - x ** (1 - a - 2 * b)), # nu, nu'
 18.6, 0.37, 0.21),
    
("Lemma A.6.4 - mu does not meet beta, nu meets beta",
 (1 - x - 2 * x ** a) * (1 - x ** b) * (1 - x ** (1 - a - 2 * b)) 
 - 2 * x ** (2 - 2 * a - b) * (1 - 2 * x ** a) # mu, mu'
 - 2 * x ** (1 + b) * (1 - x ** (1 - a - 2 * b)), # nu, nu'
 18.5, 0.36, 0.21),

("Lemma A.6.4 - mu and nu do not meet beta", 
 (1 - x - 2 * x ** a) * (1 - x ** b) * (1 - x ** (1 - a - 2 * b)) 
 - 2 * x ** (2 - 2 * a - b) * (1 - 2 * x ** a) # mu, mu'
 - 2 * x ** (1 - a + b) * (1 - 2 * x ** a) * (1 - x ** (1 - a - 2 * b)), # nu, nu'
 16.8, 0.32, 0.23),

####

("Lemma A.6.5 - omega meets beta", 
 (1 - x - 2 * x ** a) * (1 - x ** b) * (1 - 2 * x ** (1 - a - 2 * b)) 
 - 4 * x ** (1 + b) * (1 - 2 * x ** a) * (1 - 2 * x ** (1 - a - 2 * b)) # mu, mu', nu, nu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b - 4 * x ** (1 + b)), # omega, omega'
 16, 0.41, 0.12),

("Lemma A.6.5 - omega does not meet beta", 
 (1 - x - 2 * x ** a) * (1 - x ** b) * (1 - 2 * x ** (1 - a - 2 * b)) 
 - 4 * x ** (1 + b) * (1 - 2 * x ** a) * (1 - 2 * x ** (1 - a - 2 * b)) # mu, mu', nu, nu'
 - 2 * x ** (2 - 2 * a - 2 * b) * (1 - 2 * x ** a) * (1 - x ** b - 4 * x ** (1 + b)), # omega, omega'
 15.7, 0.36, 0.13),

#####
    
("Lemma A.6.6 - omega and chi meet beta", 
 (1 - x - 2 * x ** a) * (1 - x ** b) * (1 - x ** (1 - a - 2 * b)) 
 - 4 * x ** (1 + b) * (1 - 2 * x ** a) * (1 - x ** (1 - a - 2 * b)) # mu, mu', nu, nu'
 - 4 * x ** (2 - a - 2 * b) * (1 - x ** b - 4 * x ** (1 + b)), # omega, omega', chi, chi'
 14.8, 0.48, 0.13),

("Lemma A.6.6 - mu and nu meet beta", 
 (1 - x - 2 * x ** a) * (1 - x ** b) * (1 - x ** (1 - a - 2 * b)) 
 - 4 * x ** (1 + b) * (1 - x ** (1 - a - 2 * b)) # mu, mu', nu, nu'
 - 4 * x ** (2 - a - 2 * b) * ((1 - 2 * x ** a) * (1 - x ** b) - 4 * x ** (1 + b)), # omega, omega', chi, chi'
 16.2, 0.50, 0.18),

("Lemma A.6.6 - mu and omega do not meet beta", (1 - x - 2 * x ** a) * (1 - x ** b) * (1 - x ** (1 - a - 2 * b)) 
 - 2 * x ** (1 - a + b) * (1 - 2 * x ** a) * (1 - x ** (1 - a - 2 * b)) # mu, mu'
 - 2 * x ** (1 + b) * (1 - 2 * x ** a) * (1 - x ** (1 - a - 2 * b)) # nu, nu'
 - 2 * x ** (2 - 2 * a - 2 * b) * (1 - 2 * x ** a) * (1 - x ** b - 2 * x ** (1 - a + b) - 2 * x ** (1 + b)) # omega, omega'
 - 2 * x ** (2 - a - 2 * b) * (1 - 2 * x ** a) * (1 - x ** b - 2 * x ** (1 - a + b) - 2 * x ** (1 + b)), # chi, chi'
 16.6, 0.35, 0.20),

######
    
("Lemma A.6.7", 
 (1 - x ** a - 2 * x ** (1 - 2 * b)) * (1 - x ** (1 - a)) * (1 - x ** b) 
 - 2 * x ** (a + b) * (1 - 2 * x ** (1 - 2 * b)) * (1 - x ** (1 - a)) # mu, mu'
 - 2 * x ** (1 - a + b) * (1 - x ** a - 2 * x ** (1 - 2 * b)), # nu, nu'
 19.5, 0.61, 0.32),
]

In [63]:
for i, (name, function, zero_start, a_start, b_start) in enumerate(Prop_A_6_2):
    print(f"Evaluating {name}: {function} ...")
    MinPFRootApprox(function, starting_x = 1/zero_start, a_start = a_start, b_start = b_start, a_range = 0.01, b_range = 0.01)

Evaluating Lemma A.6.3 - mu and nu meet beta: -4*x**(-a - b + 2) + (1 - x**b)*(1 - x**(-a - 2*b + 1))*(-x - 2*x**a + 1) ...
The minimum PF zero is 17.4258840128918 within margin of 0.01, attained around a=0.42812500000000003, b=0.125, c=0.5, d=0.5 within margin of 0.001

Evaluating Lemma A.6.3 - mu meets beta, nu does not meet beta: -2*x**(-2*a - b + 2)*(1 - 2*x**a) - 2*x**(-a - b + 2) + (1 - x**b)*(1 - x**(-a - 2*b + 1))*(-x - 2*x**a + 1) ...
The minimum PF zero is 17.5728509773644 within margin of 0.01, attained around a=0.36000000000000004, b=0.1375, c=0.5, d=0.5 within margin of 0.001

Evaluating Lemma A.6.3 - mu and nu do not meet beta: -4*x**(-2*a - b + 2)*(1 - 2*x**a) + (1 - x**b)*(1 - x**(-a - 2*b + 1))*(-x - 2*x**a + 1) ...
The minimum PF zero is 15.8683879621599 within margin of 0.01, attained around a=0.321875, b=0.14468750000000002, c=0.5, d=0.5 within margin of 0.001

Evaluating Lemma A.6.4 - mu and nu meet beta: -2*x**(b + 1)*(1 - x**(-a - 2*b + 1)) - 2*x**(-a - b + 2) + 

In [64]:
Case_IV_a = [
("mu and omega are disjoint", 
 (1 - x - 2 * x ** (1 - 2 * b)) * (1 - x ** b) 
 - 2 * x ** ((1 + b) / 2) * (1 - 2 * x ** (1 - 2 * b)) # mu, mu'
 - 2 * x ** (1 + b) * (1 - 2 * x ** (1 - 2 * b)) # nu, nu'
 - 2 * x ** ((1 + b) / 2) * (1 - 2 * x ** (1 - 2 * b)) * (1 - 2 * x ** ((1 + b) / 2) - 2 * x ** (1 + b)) # omega, omega'
 - 2 * x ** (1 + b) * (1 - 2 * x ** (1 - 2 * b)) * (1 - 2 * x ** ((1 + b) / 2) - 2 * x ** (1 + b)), # chi, chi'
 20.2, 1, 0.33),

("mu and nu and omega and chi meet beta", 
 (1 - x - 2 * x ** (1 - 2 * b)) * (1 - x ** b) 
 - 8 * x ** (1 + b), # mu, mu', nu, nu', omega, omega', chi, chi'
 16.9, 1, 0.24),

("mu and nu and omega meet beta, chi does not meet beta", 
 (1 - x - 2 * x ** (1 - 2 * b)) * (1 - x ** b) 
 - 6 * x ** (1 + b) # mu, mu', nu, nu', omega, omega'
 - 2 * x ** (3 * b) * (1 - 2 * x ** (1 - 2 * b)), # chi, chi'
 18.4, 1, 0.29),

("mu and nu meet beta, omega and chi do not meet beta", 
 (1 - x - 2 * x ** (1 - 2 * b)) * (1 - x ** b)
 - 4 * x ** (1 + b) # mu, mu', nu, nu'
 - 4 * x ** (3 * b) * (1 - 2 * x ** (1 - 2 * b)), # omega, omega', chi, chi'
 18.8, 1, 0.31),

("mu meets beta, nu and omega and chi do not meet beta", 
 (1 - x - 2 * x ** (1 - 2 * b)) * (1 - x ** b) 
 - 2 * x ** (1 + b) # mu, mu'
 - 6 * x ** (3 * b) * (1 - 2 * x ** (1 - 2 * b)), # nu, nu', omega, omega', chi, chi'
 18.5, 1, 0.33),

("mu and nu and omega and chi do not meet beta", 
 (1 - x - 2 * x ** (1 - 2 * b)) * (1 - x ** b) 
 - 8 * x ** (3 * b) * (1 - 2 * x ** (1 - 2 * b)), # mu, mu', nu, nu', omega, omega', chi, chi'
 17.2, 1, 0.34),
]

In [65]:
for i, (name, function, zero_start, a_start, b_start) in enumerate(Case_IV_a):
    print(f"Evaluating {name}: {function} ...")
    MinPFRootApprox(function, starting_x = 1/zero_start, a_start = a_start, b_start = b_start, a_range = 0.01, b_range = 0.01)

Evaluating mu and omega are disjoint: -2*x**(b/2 + 1/2)*(1 - 2*x**(1 - 2*b))*(-2*x**(b/2 + 1/2) - 2*x**(b + 1) + 1) - 2*x**(b/2 + 1/2)*(1 - 2*x**(1 - 2*b)) - 2*x**(b + 1)*(1 - 2*x**(1 - 2*b))*(-2*x**(b/2 + 1/2) - 2*x**(b + 1) + 1) - 2*x**(b + 1)*(1 - 2*x**(1 - 2*b)) + (1 - x**b)*(-x - 2*x**(1 - 2*b) + 1) ...
The minimum PF zero is 20.2893511257014 within margin of 0.01, attained around a=1, b=0.329375, c=0.5, d=0.5 within margin of 0.001

Evaluating mu and nu and omega and chi meet beta: -8*x**(b + 1) + (1 - x**b)*(-x - 2*x**(1 - 2*b) + 1) ...
The minimum PF zero is 17.0000392325682 within margin of 0.01, attained around a=1, b=0.245, c=0.5, d=0.5 within margin of 0.001

Evaluating mu and nu and omega meet beta, chi does not meet beta: -2*x**(3*b)*(1 - 2*x**(1 - 2*b)) - 6*x**(b + 1) + (1 - x**b)*(-x - 2*x**(1 - 2*b) + 1) ...
The minimum PF zero is 18.4805058424063 within margin of 0.01, attained around a=1, b=0.29, c=0.5, d=0.5 within margin of 0.001

Evaluating mu and nu meet beta, om

In [66]:
Case_IV_b = [
("mu and nu meet beta", 
 (1 - x - 2 * x ** (1/3)) * (1 - x ** (1/3)) 
 - 4 * x ** (4/3), # mu, mu', nu, nu'
 16.5, 1, 1),

("omega meets beta and gamma", 
 (1 - x - 2 * x ** (1/3)) * (1 - x ** (1/3)) 
 - 4 * x * (1 - 2 * x ** (1/3)) # mu, mu', nu, nu'
 - 2 * x ** (4/3), # omega, omega'
 16.8, 1, 1),

("omega meets beta but not gamma", 
 (1 - x - 2 * x ** (1/3)) * (1 - x ** (1/3)) 
 - 4 * x * (1 - 2 * x ** (1/3)) # mu, mu', nu, nu'
 - 2 * x * (1 - x ** (1/3)), # omega, omega'
 18.6, 1, 1),

("omega meets gamma but not beta", 
 (1 - x - 2 * x ** (1/3)) * (1 - x ** (1/3)) 
 - 4 * x * (1 - 2 * x ** (1/3)) # mu, mu', nu, nu'
 - 2 * x * (1 - 2 * x ** (1/3)), # omega, omega'
 15.2, 1, 1),

("omega does not meet beta nor gamma",
 (1 - x - 2 * x ** (1/3)) * (1 - x ** (1/3)) 
 - 4 * x * (1 - 2 * x ** (1/3)) # mu, mu', nu, nu'
 - 2 * x ** (2/3) * (1 - 2 * x ** (1/3)) * (1 - x ** (1/3)), # omega, omega'
 16.3, 1, 1),
]

In [67]:
for i, (name, function, zero_start, a_start, b_start) in enumerate(Case_IV_b):
    print(f"Evaluating {name}: {function} ...")
    MinPFRootApprox(function, starting_x = 1/zero_start, a_start = a_start, b_start = b_start, a_range = 0.01, b_range = 0.01)

Evaluating mu and nu meet beta: -4*x**1.33333333333333 + (1 - x**0.333333333333333)*(-2*x**0.333333333333333 - x + 1) ...
The minimum PF zero is 16.5890801013864 within margin of 0.001 

Evaluating omega meets beta and gamma: -4*x*(1 - 2*x**0.333333333333333) - 2*x**1.33333333333333 + (1 - x**0.333333333333333)*(-2*x**0.333333333333333 - x + 1) ...
The minimum PF zero is 16.8868304627129 within margin of 0.001 

Evaluating omega meets beta but not gamma: -4*x*(1 - 2*x**0.333333333333333) - 2*x*(1 - x**0.333333333333333) + (1 - x**0.333333333333333)*(-2*x**0.333333333333333 - x + 1) ...
The minimum PF zero is 18.6357142000821 within margin of 0.001 

Evaluating omega meets gamma but not beta: -6*x*(1 - 2*x**0.333333333333333) + (1 - x**0.333333333333333)*(-2*x**0.333333333333333 - x + 1) ...
The minimum PF zero is 15.2331434520469 within margin of 0.001 

Evaluating omega does not meet beta nor gamma: -2*x**0.666666666666667*(1 - 2*x**0.333333333333333)*(1 - x**0.333333333333333) - 4*x*

In [68]:
Case_V_a = [
("Lemma A.7.4 - mu meets gamma",
 (1 - x) * (1 - x ** a - x ** (1 - 2 * a - b)) * (1 - x ** b) 
 - 2 * x ** (1 + a + b), # mu, mu'
 16, 0.22, 0.17, 1), 

("Lemma A.7.4 - mu does not meet gamma", 
 (1 - x) * (1 - x ** a - x ** (1 - 2 * a - b)) * (1 - x ** b) 
 - 2 * x ** (3 * a + 2 * b) * (1 - x ** (1 - 2 * a - b)), # mu, mu'
 18, 0.28, 0.17, 1), 

####
    
("Lemma A.7.5 - mu meets gamma and nu - 1", 
 (1 - x) * (1 - x ** a - x ** (1 - 2 * a - b)) * (1 - x ** b) 
 - 2 * x ** (1 + a) * (1 - x ** b) # mu, mu'
 - 2 * x ** (1 + b) * (1 - x ** a - x ** (1 - 2 * a - b)), # nu, nu'
 14.4, 0.24, 0.13, 1),

("Lemma A.7.5 - mu meets gamma and nu - 2", # Using the inequality q \leq p
 (1 - x) * (1 - x ** a - x ** a) * (1 - x ** (1 - 3 * a)) 
 - 2 * x ** (1 + a) * (1 - x ** (1 - 3 * a)) # mu, mu'
 - 2 * x ** (2 - 3 * a) * (1 - x ** a - x ** a), # nu, nu'
 15.1, 0.3, 0.1, 1),

("Lemma A.7.5 - mu does not meet gamma, mu meets nu", 
 (1 - x) * (1 - x ** a - x ** (1 - 2 * a - b)) * (1 - x ** b) 
 - 2 * x ** (3 * a + b) * (1 - x ** (1 - 2 * a - b)) * (1 - x ** b) # mu, mu'
 - 2 * x ** (1 + b) * (1 - x ** a - x ** (1 - 2 * a - b)), # nu, nu'
 15.8, 0.31, 0.11, 1),

("Lemma A.7.5 - mu does not meet nu", 
 (1 - x) * (1 - x ** a - x ** (1 - 2 * a - b)) * (1 - x ** b) 
 - 2 * x ** c * (1 - x ** (1 - 2 * a - b)) * (1 - x ** b) # mu, mu'
 - 2 * x ** (1 + a + b - c) * ((1 - x ** (1 - 2 * a - b)) * (1 - 2 * x ** c) - x ** a), # nu, nu'
 19.6, 0.25, 0.19, 0.83),

#####
    
("Case Va", 
 (1 - x) * (1 - x ** a - x ** b) 
 - 2 * x ** (3/2 - 3/2 * a - 3/2 * b) * (1 - x ** b) # beta', (beta')'
 - 2 * x ** (3/2 - 3/2 * a - 3/2 * b) * (1 - 2 * x ** (3/2 - 3/2 * a - 3/2 * b)) * (1 - x ** b), # gamma', (gamma')'
 29.2, 0.37, 0.21, 1),
]

In [69]:
for i, (name, function, zero_start, a_start, b_start, c_start) in enumerate(Case_V_a):
    print(f"Evaluating {name}: {function} ...")
    MinPFRootApprox(function, starting_x = 1/zero_start, a_start = a_start, b_start = b_start, c_start = c_start, a_range = 0.01, b_range = 0.01, c_range = 0.01)

Evaluating Lemma A.7.4 - mu meets gamma: -2*x**(a + b + 1) + (1 - x)*(1 - x**b)*(-x**a - x**(-2*a - b + 1) + 1) ...
The minimum PF zero is 16.0942575656572 within margin of 0.01, attained around a=0.22, b=0.165, c=1, d=0.5 within margin of 0.001

Evaluating Lemma A.7.4 - mu does not meet gamma: -2*x**(3*a + 2*b)*(1 - x**(-2*a - b + 1)) + (1 - x)*(1 - x**b)*(-x**a - x**(-2*a - b + 1) + 1) ...
The minimum PF zero is 18.0365924218145 within margin of 0.01, attained around a=0.28375000000000006, b=0.16249999999999998, c=1, d=0.5 within margin of 0.001

Evaluating Lemma A.7.5 - mu meets gamma and nu - 1: -2*x**(a + 1)*(1 - x**b) - 2*x**(b + 1)*(-x**a - x**(-2*a - b + 1) + 1) + (1 - x)*(1 - x**b)*(-x**a - x**(-2*a - b + 1) + 1) ...
The minimum PF zero is 14.4309316150742 within margin of 0.01, attained around a=0.235, b=0.11750000000000001, c=1, d=0.5 within margin of 0.001

Evaluating Lemma A.7.5 - mu meets gamma and nu - 2: -2*x**(2 - 3*a)*(1 - 2*x**a) - 2*x**(a + 1)*(1 - x**(1 - 3*a)) + (

In [70]:
Lemma_A_7_7 = [
# If nu meets beta

("nu meets gamma and mu", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - 2 * x ** (1 - 2 * a - b)) 
 - 2 * x ** (2 - a - b), # nu, nu'
 15.4, 0.06, 0.25, 1),

("nu does not meet gamma, nu meets mu", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - 2 * x ** (1 - 2 * a - b)) 
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b), # nu, nu'
 14.5, 0.05, 0.18, 1),

("nu does not meet mu", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - b) - x ** c) 
 - 2 * x ** (2 - a - b - c) * (1 - x ** b - x ** c), # nu, nu'
 17.8, 0.08, 0.24, 0.55),

# If nu does not meet beta
    
("nu meets gamma and mu", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - 2 * x ** (1 - 2 * a - b)) 
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) # nu, nu'
 - 2 * x ** (1 + a) * (1 - x ** b - 2 * x ** (1 - 2 * a - b)), # omega, omega'
 14.5, 0.08, 0.26, 1),
    
("nu does not meet mu", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - b) - x ** c) 
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** b - x ** c) # nu, nu'
 - 2 * x ** (1 + a) * (1 - x ** b - x ** (1 - 2 * a - b) - x ** c), # omega, omega'
 15.2, 0.09, 0.25, 0.64),

("nu does not meet gamma, nu meets mu", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - 2 * x ** (1 - 2 * a - b)) 
 - 2 * x ** (2 - 2 * a - 2 * b) * (1 - x ** a) * (1 - x ** b) # nu, nu'
 - 4 * x ** (1 + a) * (1 - x ** b - 2 * x ** (1 - 2 * a - b)), # omega, omega', chi, chi'
 17.1, 0.1, 0.19, 1),
]

In [71]:
for i, (name, function, zero_start, a_start, b_start, c_start) in enumerate(Lemma_A_7_7):
    print(f"Evaluating {name}: {function} ...")
    MinPFRootApprox(function, starting_x = 1/zero_start, a_start = a_start, b_start = b_start, c_start = c_start, a_range = 0.01, b_range = 0.01, c_range = 0.01)

Evaluating nu meets gamma and mu: -2*x**(-a - b + 2) + (1 - x)*(1 - x**a)*(-x**b - 2*x**(-2*a - b + 1) + 1) ...
The minimum PF zero is 15.4189103981342 within margin of 0.01, attained around a=0.05749999999999998, b=0.25312499999999993, c=1, d=0.5 within margin of 0.001

Evaluating nu does not meet gamma, nu meets mu: -2*x**(-a - 2*b + 2)*(1 - x**b) + (1 - x)*(1 - x**a)*(-x**b - 2*x**(-2*a - b + 1) + 1) ...
The minimum PF zero is 14.5867729245882 within margin of 0.01, attained around a=0.04875, b=0.17750000000000002, c=1, d=0.5 within margin of 0.001

Evaluating nu does not meet mu: -2*x**(-a - b - c + 2)*(-x**b - x**c + 1) + (1 - x)*(1 - x**a)*(-x**b - x**c - x**(-2*a - b + 1) + 1) ...
The minimum PF zero is 17.8353803416192 within margin of 0.01, attained around a=0.07875000000000003, b=0.24062500000000003, c=0.5534375, d=0.5 within margin of 0.001

Evaluating nu meets gamma and mu: -2*x**(a + 1)*(-x**b - 2*x**(-2*a - b + 1) + 1) - 2*x**(-2*a - b + 2)*(1 - x**a) + (1 - x)*(1 - x**a)

In [72]:
Lemma_A_7_8 = [
# If nu meets beta
("nu meets gamma and other petal curves", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c)
 - x ** (1 - 2 * a - 2 * b) * (1 - x) * (1 - x ** a) * (1 - x ** b) # mu
 - 2 * x ** (2 - a - b) * (1 - x ** (1 - 2 * a - 2* b)), # nu, nu'
 23.7, 0.04, 0.15, 0.23),

("nu meets delta but not gamma, nu meets other petal curves", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c)
 - x ** (1 - 2 * a - 2 * b) * (1 - x) * (1 - x ** a) * (1 - x ** b) # mu
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2* b)), # nu, nu'
 20.7, 0.04, 0.11, 0.20),

("nu does not meet delta, nu meets other petal curves", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c)
 - x ** (1 - 2 * a - 2 * b) * (1 - x) * (1 - x ** a) * (1 - x ** b) # mu
 - 2 * x ** (1 + a + c) * ((1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b)) - x ** (1 - 2 * a - b - c)), # nu, nu'
 21.1, 0.04, 0.13, 0.3),

("nu meets gamma, nu does not meet other petal curves", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c)
 - x ** (1 - 2 * a - 2 * b) * (1 - x) * (1 - x ** a) * (1 - x ** b) # mu
 - 2 * x ** (2 - a - b - c) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)), # nu, nu'
 21.1, 0.04, 0.13, 0.13),

("nu meets delta but not gamma, nu does not meet other petal curves", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c)
 - x ** (1 - 2 * a - 2 * b) * (1 - x) * (1 - x ** a) * (1 - x ** b) # mu
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)), # nu, nu'
 18.4, 0.03, 0.11, 0.13),

# If nu does not meet beta
    
("nu meets gamma and other petal curves", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c)
 - x ** (1 - 2 * a - 2 * b) * (1 - x) * (1 - x ** a) * (1 - x ** b) # mu
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * b)) # nu, nu'
 - 2 * x ** (1 + a) * ((1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c) - x ** (1 - 2 * a - 2 * b) * (1 - x ** b)), # omega, omega'
 20.8, 0.05, 0.14, 0.21),

("nu meets delta but not gamma, nu meets other petal curves", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c)
 - x ** (1 - 2 * a - 2 * b) * (1 - x) * (1 - x ** a) * (1 - x ** b) # mu
 - 2 * x ** (2 - 2 * a - 2 * b) * (1 - x ** a) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b)) # nu, nu'
 - 2 * x ** (1 + a) * ((1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c) - x ** (1 - 2 * a - 2 * b) * (1 - x ** b)), # omega, omega'
 19.4, 0.05, 0.12, 0.19),

("nu does not meet delta, nu meets other petal curves", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c)
 - x ** (1 - 2 * a - 2 * b) * (1 - x) * (1 - x ** a) * (1 - x ** b) # mu
 - 2 * x ** (1 + c) * (1 - x ** a) * ((1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b)) - x ** (1 - 2 * a - b - c)) # nu, nu'
 - 2 * x ** (1 + a) * ((1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c) - x ** (1 - 2 * a - 2 * b) * (1 - x ** b)), # omega, omega'
 19.5, 0.05, 0.13, 0.24),

("nu meets gamma, nu does not meet other petal curves", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c)
 - x ** (1 - 2 * a - 2 * b) * (1 - x) * (1 - x ** a) * (1 - x ** b) # mu
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)) # nu, nu'
 - 2 * x ** (1 + a) * ((1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c) - x ** (1 - 2 * a - 2 * b) * (1 - x ** b)), # omega, omega'
 19.5, 0.05, 0.13, 0.15),

("nu meets delta but not gamma, nu does not meet other petal curves", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c)
 - x ** (1 - 2 * a - 2 * b) * (1 - x) * (1 - x ** a) * (1 - x ** b) # mu
 - 2 * x ** (2 - 2 * a - 2 * b - c) * ((1 - x ** a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b))) # nu, nu'
 - 2 * x ** (1 + a) * ((1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c) - x ** (1 - 2 * a - 2 * b) * (1 - x ** b)), # omega, omega'
 18.5, 0.05, 0.11, 0.15),
]

In [73]:
for i, (name, function, zero_start, a_start, b_start, c_start) in enumerate(Lemma_A_7_8):
    print(f"Evaluating {name}: {function} ...")
    MinPFRootApprox(function, starting_x = 1/zero_start, a_start = a_start, b_start = b_start, c_start = c_start, a_range = 0.01, b_range = 0.01, c_range = 0.01)

Evaluating nu meets gamma and other petal curves: -x**(-2*a - 2*b + 1)*(1 - x)*(1 - x**a)*(1 - x**b) - 2*x**(-a - b + 2)*(1 - x**(-2*a - 2*b + 1)) + (1 - x)*(1 - x**a)*(1 - x**c)*(-x**b - x**(-2*a - b - c + 1) + 1) ...
The minimum PF zero is 23.7118036634707 within margin of 0.01, attained around a=0.042187499999999996, b=0.14968749999999997, c=0.22906250000000003, d=0.5 within margin of 0.001

Evaluating nu meets delta but not gamma, nu meets other petal curves: -x**(-2*a - 2*b + 1)*(1 - x)*(1 - x**a)*(1 - x**b) - 2*x**(-a - 2*b + 2)*(1 - x**b)*(1 - x**(-2*a - 2*b + 1)) + (1 - x)*(1 - x**a)*(1 - x**c)*(-x**b - x**(-2*a - b - c + 1) + 1) ...
The minimum PF zero is 20.6999052446940 within margin of 0.01, attained around a=0.035625000000000004, b=0.10812500000000001, c=0.20000000000000004, d=0.5 within margin of 0.001

Evaluating nu does not meet delta, nu meets other petal curves: -x**(-2*a - 2*b + 1)*(1 - x)*(1 - x**a)*(1 - x**b) - 2*x**(a + c + 1)*(-x**(-2*a - b - c + 1) + (1 - x**b)*

In [74]:
Lemma_A_7_9 = [
("mu meets beta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c) 
 - 2 * x ** (2 - a - b), # mu, mu'
 17.8, 0.08, 0.24, 0.18),

# If mu does not meet beta
    
("nu meets gamma and other petal curves", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c) 
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) # mu, mu'
 - 2 * x ** (2 - a - b), # nu, nu'
 18.9, 0.07, 0.25, 0.19),
    
("nu meets delta but not gamma, nu meets other petal curves", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c) 
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) # mu, mu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b), # nu, nu'
 17.5, 0.06, 0.15, 0.18),
    
("nu does not meet delta nor other petal curves", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c) 
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) # mu, mu'
 - 2 * x ** (1 + a + c) * (1 - x ** b - x ** (1 - 2 * a - b - c)), # nu, nu'
 15.8, 0.06, 0.25, 0.29),
    
("nu meets gamma, nu does not meet other petal curves", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c) 
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) # mu, mu'
 - 2 * x ** (2 - a - b - c) * (1 - x ** c), # nu, nu'
 15.8, 0.07, 0.24, 0.11),

("nu meets delta but not gamma, nu does not meet other petal curves", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c) 
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c), # nu, nu'
 14.9, 0.05, 0.17, 0.1),

("nu does not meet delta nor other petal curves", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c)
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) # mu, mu'
 - 2 * x ** (1 + a) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c), # nu, nu'
 15.2, 0.09, 0.25, 0.16),
]

In [75]:
for i, (name, function, zero_start, a_start, b_start, c_start) in enumerate(Lemma_A_7_9):
    print(f"Evaluating {name}: {function} ...")
    MinPFRootApprox(function, starting_x = 1/zero_start, a_start = a_start, b_start = b_start, c_start = c_start, a_range = 0.01, b_range = 0.01, c_range = 0.01)

Evaluating mu meets beta: -2*x**(-a - b + 2) + (1 - x)*(1 - x**a)*(1 - x**c)*(-x**b - x**(-2*a - b - c + 1) + 1) ...
The minimum PF zero is 17.8353797546030 within margin of 0.01, attained around a=0.07875000000000003, b=0.24062500000000003, c=0.18062499999999998, d=0.5 within margin of 0.001

Evaluating nu meets gamma and other petal curves: -2*x**(-2*a - b + 2)*(1 - x**a) - 2*x**(-a - b + 2) + (1 - x)*(1 - x**a)*(1 - x**c)*(-x**b - x**(-2*a - b - c + 1) + 1) ...
The minimum PF zero is 18.9785912637713 within margin of 0.01, attained around a=0.06750000000000002, b=0.23562500000000003, c=0.19687500000000002, d=0.5 within margin of 0.001

Evaluating nu meets delta but not gamma, nu meets other petal curves: -2*x**(-2*a - b + 2)*(1 - x**a) - 2*x**(-a - 2*b + 2)*(1 - x**b) + (1 - x)*(1 - x**a)*(1 - x**c)*(-x**b - x**(-2*a - b - c + 1) + 1) ...
The minimum PF zero is 17.5663655272638 within margin of 0.01, attained around a=0.057499999999999996, b=0.14875000000000002, c=0.1837499999999999

In [76]:
Lemma_A_7_10 = [
# If nu meets beta
    
("nu meets gamma", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c) 
 - 2 * x ** (2 - a - b - c) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - b), # nu, nu'
 20.4, 0.09, 0.21, 0.13),

("nu meets delta but not gamma", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c) 
 - 2 * x ** (2 - a - b - c) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b), # nu, nu'
 19.2, 0.09, 0.15, 0.12),

("nu does not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c) 
 - 2 * x ** (2 - a - b - c) * (1 - x ** c) # mu, mu'
 - 2 * x ** (1 + a + c) * (1 - x ** b - x ** (1 - 2 * a - b - c)), # nu, nu'
 18.8, 0.13, 0.23, 0.12),

# If nu does not meet beta
    
("nu meets gamma", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c) 
 - 2 * x ** (2 - a - b - c) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a), # nu, nu'
 15.8, 0.07, 0.24, 0.11),

("nu meets delta but not gamma", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c) 
 - 2 * x ** (2 - a - b - c) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - 2 * a - 2 * b) * (1 - x ** a) * (1 - x ** b), # nu, nu'
 15.4, 0.07, 0.2, 0.07),

("nu does not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c) 
 - 2 * x ** (2 - a - b - c) * (1 - x ** c) # mu, mu'
 - 2 * x ** (1 + c) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - b - c)), # nu, nu' 
 15.7, 0.09, 0.24, 0.11),
]

In [77]:
for i, (name, function, zero_start, a_start, b_start, c_start) in enumerate(Lemma_A_7_10):
    print(f"Evaluating {name}: {function} ...")
    MinPFRootApprox(function, starting_x = 1/zero_start, a_start = a_start, b_start = b_start, c_start = c_start, a_range = 0.01, b_range = 0.01, c_range = 0.01)

Evaluating nu meets gamma: -2*x**(-a - b + 2) - 2*x**(-a - b - c + 2)*(1 - x**c) + (1 - x)*(1 - x**a)*(1 - x**c)*(-x**b - x**(-2*a - b - c + 1) + 1) ...
The minimum PF zero is 20.5030319457142 within margin of 0.01, attained around a=0.09296874999999996, b=0.22953124999999994, c=0.13203125000000002, d=0.5 within margin of 0.001

Evaluating nu meets delta but not gamma: -2*x**(-a - 2*b + 2)*(1 - x**b) - 2*x**(-a - b - c + 2)*(1 - x**c) + (1 - x)*(1 - x**a)*(1 - x**c)*(-x**b - x**(-2*a - b - c + 1) + 1) ...
The minimum PF zero is 19.2767187147929 within margin of 0.01, attained around a=0.08999999999999998, b=0.15125000000000002, c=0.110625, d=0.5 within margin of 0.001

Evaluating nu does not meet delta: -2*x**(a + c + 1)*(-x**b - x**(-2*a - b - c + 1) + 1) - 2*x**(-a - b - c + 2)*(1 - x**c) + (1 - x)*(1 - x**a)*(1 - x**c)*(-x**b - x**(-2*a - b - c + 1) + 1) ...
The minimum PF zero is 18.8502622512601 within margin of 0.01, attained around a=0.13187500000000002, b=0.23625000000000004, c

In [78]:
Lemma_A_7_11 = [
# If nu meets beta and a petal curve other than beta and delta
    
("nu meets gamma", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c) 
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - b), # nu, nu'
 18.6, 0.07, 0.22, 0.17),

("nu meets delta but not gamma", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c) 
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b), # nu, nu'
 16.9, 0.07, 0.14, 0.14),

("nu does not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c) 
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) # mu, mu'
 - 2 * x ** (1 + a + c) * (1 - x ** b - x ** (1 - 2 * a - b - c)), # nu, nu'
 15.7, 0.09, 0.24, 0.21),

# If nu meets beta, omega meets a petal curve other than beta and delta
    
("nu meets gamma, omega meets gamma", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c) 
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - b - c) * (1 - x ** c) # nu, nu'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a), # omega, omega'
 16.5, 0.06, 0.24, 0.08),
    
("nu meets delta but not gamma, omega meets gamma", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c) 
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # nu, nu'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a), # omega, omega'
 15.6, 0.05, 0.17, 0.07),

("nu does not meet delta, omega meets gamma", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c) 
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) # mu, mu'
 - 2 * x ** (1 + a) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c) # nu, nu'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a), # omega, omega'
 16.6, 0.08, 0.24, 0.13),

("nu meets gamma, omega meets delta but not gamma", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c) 
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - b - c) * (1 - x ** c) # nu, nu'
 - 2 * x ** (2 - 2 * a - 2 * b) * (1 - x ** b) * (1 - x ** a), # omega, omega'
 16, 0.06, 0.19, 0.06),

("nu does not meet delta, omega meets delta but not gamma", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c) 
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) # mu, mu'
 - 2 * x ** (1 + a) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c) # nu, nu'
 - 2 * x ** (2 - 2 * a - 2 * b) * (1 - x ** b) * (1 - x ** a), # omega, omega'
 15.9, 0.09, 0.18, 0.11),

("nu meets gamma, omega does not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c) 
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - b - c) * (1 - x ** c) # nu, nu'
 - 2 * x ** (1 + c) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** a), # omega, omega'
 16.8, 0.08, 0.24, 0.1),

("nu meets delta but not gamma, omega does not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c) 
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # nu, nu'
 - 2 * x ** (1 + c) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** a), # omega, omega'
 15.6, 0.07, 0.17, 0.12),

("nu does not meet delta, omega does not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c) 
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) # mu, mu'
 - 2 * x ** (1 + a) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c) # nu, nu'
 - 2 * x ** (1 + c) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** a), # omega, omega'
 15.4, 0.1, 0.25, 0.15),

("nu meets delta but not gamma, omega meets delta but not gamma", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - b - c)) * (1 - x ** c) 
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) # mu, mu'
 - 4 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # nu, nu', chi, chi'
 - 4 * x ** (2 - 2 * a - 2 * b) * (1 - x ** b) * (1 - x ** a), # omega, omega', upsilon, upsilon'
 17.4, 0.05, 0.12, 0.07),
]

In [79]:
for i, (name, function, zero_start, a_start, b_start, c_start) in enumerate(Lemma_A_7_11):
    print(f"Evaluating {name}: {function} ...")
    MinPFRootApprox(function, starting_x = 1/zero_start, a_start = a_start, b_start = b_start, c_start = c_start, a_range = 0.01, b_range = 0.01, c_range = 0.01)

Evaluating nu meets gamma: -2*x**(-a - b + 2) - 2*x**(-2*a - b - c + 2)*(1 - x**a)*(1 - x**c) + (1 - x)*(1 - x**a)*(1 - x**c)*(-x**b - x**(-2*a - b - c + 1) + 1) ...
The minimum PF zero is 18.6623679655388 within margin of 0.01, attained around a=0.07468750000000002, b=0.23687500000000006, c=0.16999999999999998, d=0.5 within margin of 0.001

Evaluating nu meets delta but not gamma: -2*x**(-a - 2*b + 2)*(1 - x**b) - 2*x**(-2*a - b - c + 2)*(1 - x**a)*(1 - x**c) + (1 - x)*(1 - x**a)*(1 - x**c)*(-x**b - x**(-2*a - b - c + 1) + 1) ...
The minimum PF zero is 16.9966984309024 within margin of 0.01, attained around a=0.065625, b=0.14125, c=0.14687499999999998, d=0.5 within margin of 0.001

Evaluating nu does not meet delta: -2*x**(a + c + 1)*(-x**b - x**(-2*a - b - c + 1) + 1) - 2*x**(-2*a - b - c + 2)*(1 - x**a)*(1 - x**c) + (1 - x)*(1 - x**a)*(1 - x**c)*(-x**b - x**(-2*a - b - c + 1) + 1) ...
The minimum PF zero is 15.7997192964797 within margin of 0.01, attained around a=0.0893749999999999

In [80]:
Lemma_A_7_12 = [
("mu meets beta but not nu", 
 (1 - x) * (1 - x ** a) * ((1 - x ** b) * (1 - x ** c) - x ** (1 - 2 * a - b)) 
 - 2 * x ** (2 - a - b - c) * (1 - x ** c), # mu, mu'
 17.8, 0.08, 0.24, 0.31),

# If mu meets beta and nu
    
("omega meets gamma, omega meets nu", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - 2 * b)) 
 - 4 * x ** (2 - a - b), # mu, mu', omega, omega'
 17.2, 0.08, 0.17, 1),

("omega does not meet gamma, omega meets nu", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - 2 * b)) 
 - 2 * x ** (2 - a - b) # mu, mu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b), # omega, omega'
 16, 0.07, 0.16, 1),

("omega meets gamma, omega does not meet nu", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - 2 * b)) 
 - 2 * x ** (2 - a - b) # mu, mu'
 - 2 * x ** (1 + a + b) * (1 - x ** (1 - 2 * a - 2 * b)), # omega, omega'
 20.5, 0.1, 0.22, 1),

("omega meets delta, omega does not meet gamma, omega does not meet nu", 
 (1 - x) * (1 - x ** a) * ((1 - x ** b) * (1 - x ** c) - x ** (1 - 2 * a - b)) 
 - 2 * x ** (2 - a - b) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c), # omega, omega'
 19.2, 0.09, 0.15, 0.43),

("omega does not meet delta", 
 (1 - x) * (1 - x ** a) * ((1 - x ** b - x ** (1 - 2 * a - 2 * b)))
 - 2 * x ** (2 - a - b) # mu, mu'
 - 2 * x ** (1 + a) * (1 - x ** b - x ** (1 - 2 * a - 2 * b)), # omega, omega'
 16.7, 0.1, 0.16, 1),

# If mu does not meet beta nor nu
    
("omega meets gamma, omega meets nu", 
 (1 - x) * (1 - x ** a) * ((1 - x ** b) * (1 - x ** c) - x ** (1 - 2 * a - b))
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - b) * (1 - 2 * x ** (2 - 2 * a - b - c)), # omega, omega'
 15.5, 0.06, 0.26, 0.57),

("omega does not meet gamma, omega meets nu, omega meets mu", 
 (1 - x) * (1 - x ** a) * ((1 - x ** b) * (1 - x ** c) - x ** (1 - 2 * a - b))
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b), # omega, omega'
 15, 0.05, 0.17, 0.54),

("omega does not meet gamma, omega meets nu, omega does not meet mu", 
 (1 - x) * (1 - x ** a) * ((1 - x ** b) * (1 - x ** c) - x ** (1 - 2 * a - b))
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) # mu, mu'
 - 2 * x ** (a + c) * (1 - x ** b - 2 * x ** (2 - 2 * a - b - c)), # omega, omega'
 23.2, 0.08, 0.22, 0.93),

("omega meets gamma, omega does not meet nu", 
 (1 - x) * (1 - x ** a) * ((1 - x ** b) * (1 - x ** c) - x ** (1 - 2 * a - b))
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - b - c) * (1 - x ** c) * (1 - 2 * x ** (2 - 2 * a - b - c)), # omega, omega'
 18.7, 0.07, 0.23, 0.28),

("omega meets delta, omega does not meet gamma, omega does not meet nu", 
 (1 - x) * (1 - x ** a) * ((1 - x ** b) * (1 - x ** c) - x ** (1 - 2 * a - b))
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b - 2 * x ** (2 - 2 * a - b - c)) * (1 - x ** c), # omega, omega'
 17, 0.06, 0.15, 0.33),

("omega does not meet delta, omega meets mu", 
 (1 - x) * (1 - x ** a) * ((1 - x ** b) * (1 - x ** c) - x ** (1 - 2 * a - b))
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) # mu, mu'
 - 2 * x ** (1 + a) * ((1 - x ** b) * (1 - x ** c) - x ** (1 - 2 * a - b)), # omega, omega'
 15.2, 0.08, 0.25, 0.37),
    
("omega does not meet delta, omega does not meet mu", 
 (1 - x) * (1 - x ** a) * ((1 - x ** b) * (1 - x ** c) - x ** (1 - 2 * a - b))
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) # mu, mu'
 - 2 * x ** (a + c) * ((1 - x ** b - 2 * x ** (2 - 2 * a - b - c)) * (1 - x ** c) - x ** (1 - 2 * a - b)), # omega, omega'
 16.9, 0.09, 0.25, 0.77),

# If mu meets nu but not beta
    
("omega meets gamma, omega meets nu, chi meets gamma, chi meets nu", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - 2 * b)) 
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) # mu, mu'
 - 2 * x ** (2 - a - b) * (1 - 2 * x ** (2 - 2 * a - b)) # omega, omega'
 - 2 * x ** (2 - a - b) * (1 - 2 * x ** (2 - 2 * a - b)), # chi, chi'
 17.7, 0.07, 0.17, 1),

("omega does not meet gamma, omega meets nu, chi meets gamma, chi meets nu", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - 2 * b)) 
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) # mu, mu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b - 2 * x ** (2 - 2 * a - b)) # omega, omega'
 - 2 * x ** (2 - a - b) * (1 - 2 * x ** (2 - 2 * a - b)), # chi, chi'
 16.5, 0.07, 0.15, 1),

("omega meets gamma, omega does not meet nu, chi meets gamma, chi meets nu", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - 2 * b)) 
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) # mu, mu'
 - 2 * x ** (1 + a + b) * (1 - x ** (1 - 2 * a - 2 * b) - 2 * x ** (2 - 2 * a - b)) # omega, omega'
 - 2 * x ** (2 - a - b) * (1 - 2 * x ** (2 - 2 * a - b)), # chi, chi'
 21, 0.09, 0.23, 1),

("omega meets delta, omega does not meet gamma, omega does not meet nu, chi meets gamma, chi meets nu", 
 (1 - x) * (1 - x ** a) * ((1 - x ** b) * (1 - x ** c) - x ** (1 - 2 * a - b)) 
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - c) * ((1 - x ** b) * (1 - x ** c) - 2 * x ** (2 - 2 * a - b)) # omega, omega'
 - 2 * x ** (2 - a - b) * (1 - 2 * x ** (2 - 2 * a - b)), # chi, chi'
 19.7, 0.09, 0.15, 0.43),

("omega meets gamma, omega does not meet nu, chi meets gamma, chi meets nu", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - 2 * b)) 
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) # mu, mu'
 - 2 * x ** (1 + a) * (1 - x ** b - x ** (1 - 2 * a - 2 * b) - 2 * x ** (2 - 2 * a - b)) # omega, omega'
 - 2 * x ** (2 - a - b) * (1 - 2 * x ** (2 - 2 * a - b)), # chi, chi'
 17.3, 0.09, 0.16, 1),

("omega does not meet gamma, omega meets nu, chi does not meet gamma, chi meets nu", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - 2 * b)) 
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) # mu, mu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b - 2 * x ** (2 - 2 * a - b)) # omega, omega'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b - 2 * x ** (2 - 2 * a - b)), # chi, chi'
 14.6, 0.06, 0.12, 1),

("omega meets gamma, omega does not meet nu, chi does not meet gamma, chi meets nu", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - 2 * b)) 
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) # mu, mu'
 - 2 * x ** (1 + a + b) * (1 - x ** (1 - 2 * a - 2 * b) - 2 * x ** (2 - 2 * a - b)) # omega, omega'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b - 2 * x ** (2 - 2 * a - b)), # chi, chi'
 20.8, 0.09, 0.21, 1),

("omega meets delta, omega does not meet gamma, omega does not meet nu, chi does not meet gamma, chi meets nu", 
 (1 - x) * (1 - x ** a) * ((1 - x ** b) * (1 - x ** c) - x ** (1 - 2 * a - b)) 
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - c) * ((1 - x ** b) * (1 - x ** c) - 2 * x ** (2 - 2 * a - b)) # omega, omega'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b - 2 * x ** (2 - 2 * a - b)), # chi, chi'
 18.4, 0.07, 0.12, 0.43),

("omega does not meet delta, chi does not meet gamma, chi meets nu", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - 2 * b)) 
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) # mu, mu'
 - 2 * x ** (1 + a) * (1 - x ** b - x ** (1 - 2 * a - 2 * b) - 2 * x ** (2 - 2 * a - b)) # omega, omega'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b - 2 * x ** (2 - 2 * a - b)), # chi, chi'
 15.3, 0.09, 0.13, 1),

("omega meets gamma, omega does not meet nu, chi meets gamma, chi does not meet nu", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - 2 * b)) 
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) # mu, mu'
 - 2 * x ** (1 + a + b) * (1 - x ** (1 - 2 * a - 2 * b) - 2 * x ** (2 - 2 * a - b)) # omega, omega'
 - 2 * x ** (1 + a + b) * (1 - x ** (1 - 2 * a - 2 * b) - 2 * x ** (2 - 2 * a - b)), # chi, chi'
 22.7, 0.09, 0.26, 1),

("omega meets delta, omega does not meet gamma, omega does not meet nu, chi meets gamma, chi does not meet nu", 
 (1 - x) * (1 - x ** a) * ((1 - x ** b) * (1 - x ** c) - x ** (1 - 2 * a - b)) 
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - c) * ((1 - x ** b) * (1 - x ** c) - 2 * x ** (2 - 2 * a - b)) # omega, omega'
 - 2 * x ** (2 - a - b - c) * (1 - x ** c - 2 * x ** (2 - 2 * a - b)), # chi, chi'
 21.8, 0.08, 0.17, 0.28),

("omega does not meet delta, chi meets gamma, chi does not meet nu", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - 2 * b)) 
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) # mu, mu'
 - 2 * x ** (1 + a) * (1 - x ** b - x ** (1 - 2 * a - 2 * b) - 2 * x ** (2 - 2 * a - b)) # omega, omega'
 - 2 * x ** (1 + a + b) * (1 - x ** (1 - 2 * a - 2 * b) - 2 * x ** (2 - 2 * a - b)), # chi, chi'
 20.6, 0.1, 0.23, 1),

("omega meets delta, omega does not meet gamma, omega does not meet nu, chi meets delta, chi does not meet gamma, chi does not meet nu", 
 (1 - x) * (1 - x ** a) * ((1 - x ** b) * (1 - x ** c) - x ** (1 - 2 * a - b)) 
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - c) * ((1 - x ** b) * (1 - x ** c) - 2 * x ** (2 - 2 * a - b)) # omega, omega'
 - 2 * x ** (2 - a - 2 * b - c) * ((1 - x ** b) * (1 - x ** c) - 2 * x ** (2 - 2 * a - b)), # chi, chi'
 19.8, 0.07, 0.12, 0.32),

("omega does not meet delta, chi meets delta, chi does not meet gamma, chi does not meet nu", 
 (1 - x) * (1 - x ** a) * ((1 - x ** b) * (1 - x ** c) - x ** (1 - 2 * a - b)) 
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) # mu, mu'
 - 2 * x ** (1 + a) * ((1 - x ** b) * (1 - x ** c) - x ** (1 - 2 * a - b) - 2 * x ** (2 - 2 * a - b)) # omega, omega'
 - 2 * x ** (2 - a - 2 * b - c) * ((1 - x ** b) * (1 - x ** c) - 2 * x ** (2 - 2 * a - b)), # chi, chi'
 18.9, 0.10, 0.14, 0.39),

("omega does not meet delta, chi does not meet delta, omega meets mu, chi meets mu", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - 2 * b)) 
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) # mu, mu'
 - 2 * x ** (1 + a) * (1 - x ** b - x ** (1 - 2 * a - 2 * b)) # omega, omega'
 - 2 * x ** (1 + a) * (1 - x ** b - x ** (1 - 2 * a - 2 * b)), # chi, chi'
 16.3, 0.11, 0.16, 1),

("omega does not meet delta, chi does not meet delta, at least one of omega and chi does not meet mu", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - 2 * a - 2 * b)) 
 - 2 * x ** (2 - a - b - c) * (1 - x ** a) # mu, mu'
 - 2 * x ** c * (1 - x ** b - x ** (1 - 2 * a - 2 * b) - 2 * x ** (2 - a - b - c)) # omega, omega'
 - 2 * x ** (1 + a) * (1 - x ** b - x ** (1 - 2 * a - 2 * b) - 2 * x ** (2 - a - b - c)), # chi, chi'
 21.9, 0.11, 0.15, 0.8),
]

In [81]:
for i, (name, function, zero_start, a_start, b_start, c_start) in enumerate(Lemma_A_7_12):
    print(f"Evaluating {name}: {function} ...")
    MinPFRootApprox(function, starting_x = 1/zero_start, a_start = a_start, b_start = b_start, c_start = c_start, a_range = 0.01, b_range = 0.01, c_range = 0.01)

Evaluating mu meets beta but not nu: -2*x**(-a - b - c + 2)*(1 - x**c) + (1 - x)*(1 - x**a)*(-x**(-2*a - b + 1) + (1 - x**b)*(1 - x**c)) ...
The minimum PF zero is 17.8353801438979 within margin of 0.01, attained around a=0.07875000000000003, b=0.24062500000000003, c=0.313125, d=0.5 within margin of 0.001

Evaluating omega meets gamma, omega meets nu: -4*x**(-a - b + 2) + (1 - x)*(1 - x**a)*(-x**b - x**(-2*a - 2*b + 1) + 1) ...
The minimum PF zero is 17.2281721791154 within margin of 0.01, attained around a=0.075625, b=0.16968750000000005, c=1, d=0.5 within margin of 0.001

Evaluating omega does not meet gamma, omega meets nu: -2*x**(-a - 2*b + 2)*(1 - x**b) - 2*x**(-a - b + 2) + (1 - x)*(1 - x**a)*(-x**b - x**(-2*a - 2*b + 1) + 1) ...
The minimum PF zero is 16.0942134059324 within margin of 0.01, attained around a=0.07375, b=0.145625, c=1, d=0.5 within margin of 0.001

Evaluating omega meets gamma, omega does not meet nu: -2*x**(-a - b + 2) - 2*x**(a + b + 1)*(1 - x**(-2*a - 2*b + 1))

In [82]:
Case_V_b = [
    
# If beta' passes through filaments
    
("beta' meets beta, beta' meets delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** c) 
 - 2 * x ** (3 - 3 * a - 3 * b - 3 * c) * (1 - x ** b), # beta', (beta')'
 22, 0.09, 0.22, 0.31),

("beta' does not meet beta, beta' meets delta", 
 (1 - x) * (1 - x ** b - x ** c) 
 - 2 * x ** (3/2 - 3/2 * b - 3/2 * c) * (1 - x ** b), # beta', (beta')'
 20, 1, 0.23, 0.4),

("beta' meets beta, beta' does not meet delta, nu meets beta, nu meets beta'", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** c) 
 - 2 * x ** (3 - 3 * a - 3 * b - 3 * c) * (1 - x ** b - x ** c) # beta', (beta')'
 - 2 * x ** (1 + a + c), # nu, nu'
 19.2, 0.19, 0.23, 0.33),

("beta' meets beta, beta' does not meet delta, nu meets beta, nu does not meet beta'", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - a - b - 1/3 * c)) 
 - 2 * x ** c * (1 - x ** b - x ** (1 - a - b - 1/3 * c)) # beta', (beta')'
 - 2 * x ** (1 + a + b - c) * (1 - 2 * x ** c), # nu, nu'
 25.7, 0.31, 0.33, 0.46),

("beta' meets beta, beta' does not meet delta, nu does not meet beta, nu meets beta'", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** c) 
 - 2 * x ** (3 - 3 * a - 3 * b - 3 * c) * (1 - x ** b - x ** c) # beta', (beta')'
 - 2 * x ** (1 + c) * (1 - x ** a), # nu, nu'
 16.9, 0.1, 0.24, 0.34),

("beta' meets beta, beta' does not meet delta, nu does not meet beta, nu does not meet beta'", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (1 - a - b - 1/3 * c)) 
 - 2 * x ** c * (1 - x ** b - x ** (1 - a - b - 1/3 * c)) # beta', (beta')'
 - 2 * x ** (1 + a + b - c) * (1 - x ** a - 2 * x ** c), # nu, nu'
 18.5, 0.22, 0.36, 0.56),

# If c' is nonsingular
    
("mu meets beta, nu meets delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (5/7 - 4/7 * a - b)) 
 - 2 * x ** (1 + a + b) # mu, mu'
 - 2 * x ** (1 + a + b) * (1 - x ** b), # nu, nu'
 15.6, 0.2, 0.34, 1),

("mu meets beta, nu does not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (5/7 - 4/7 * a - b)) 
 - 2 * x ** (1 + a + b) # mu, mu'
 - 2 * x ** (1 + a) * (1 - x ** b - x ** (5/7 - 4/7 * a - b)), # nu, nu'
 14.7, 0.19, 0.34, 1),

("mu does not meet beta, nu meets delta, omega meets delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (5/7 - 4/7 * a - b)) 
 - 2 * x ** (1 + b) * (1 - x ** a) # mu, mu'
 - 4 * x ** (1 + a + b) * (1 - x ** b - 2 * x ** (1 + b)), # nu, nu'
 16, 0.17, 0.34, 1),

("mu does not meet beta, nu meets delta, omega does not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (5/7 - 4/7 * a - b)) 
 - 2 * x ** (1 + b) * (1 - x ** a) # mu, mu'
 - 2 * x ** (1 + a + b) * (1 - x ** b - 2 * x ** (1 + b)) 
 - 2 * x ** (1 + a) * (1 - x ** b - x ** (5/7 - 4/7 * a - b)), # nu, nu'
 15.4, 0.17, 0.34, 1),
    
("mu does not meet beta, nu does not meet delta, omega does not meet delta, nu meets mu, omega meets mu", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (5/7 - 4/7 * a - b)) 
 - 2 * x ** (1 + b) * (1 - x ** a) # mu, mu'
 - 4 * x ** (1 + a) * (1 - x ** b - x ** (5/7 - 4/7 * a - b)), # nu, nu', omega, omega'
 14.5, 0.18, 0.35, 1),

("mu does not meet beta, nu does not meet delta, omega does not meet delta, at least one of nu and omega does not meet mu", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** (5/7 - 4/7 * a - b)) 
 - 2 * x ** c * (1 - x ** a) # mu, mu'
 - 2 * x ** (1 + a + b - c) * (1 - x ** b - x ** (5/7 - 4/7 * a - b) - 2 * x ** c) # nu, nu'
 - 2 * x ** (1 + a) * (1 - x ** b - x ** (5/7 - 4/7 * a - b) - 2 * x ** c), # omega, omega'
 19.3, 0.25, 0.34, 1.01),

# If b' singular, beta' passes through filaments, c' singular, gamma' passes through filaments
    
("mu meets beta, mu meets delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** c) 
 - 2 * x ** (4 - 2 * a - 3 * b - 6 * c) * (1 - x ** b), # mu, mu'
 16.7, 0.14, 0.39, 0.2),

("mu does not meet beta, mu meets delta", 
 (1 - x) * (1 - x ** b - x ** c) 
 - 2 * x ** (3/2 - 3/2 * b - 3/2 * c) * (1 - x ** b), # mu, mu'
 20.1, 1, 0.23, 0.4),

("mu meets beta, mu does not meet delta, nu meets beta - 1", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** c) 
 - 2 * x ** (4 - 2 * a - 3 * b - 6 * c) * (1 - x ** b - x ** c) # mu, mu'
 - 2 * x ** (1 + a + c), # nu, nu'
 14.2, 0.38, 0.41, 0.22),

("mu meets beta, mu does not meet delta, nu meets beta - 2", # Using the inequality q \leq s
 (1 - x) * (1 - x ** a) * (1 - x ** c - x ** c) 
 - 2 * x ** (4 - 2 * a - 3 * c - 6 * c) * (1 - x ** c - x ** c) # mu, mu'
 - 2 * x ** (1 + a + c), # nu, nu'
 15, 0.347, 1, 0.291),

("mu meets beta, mu does not meet delta, nu does not meet beta - 1", 
 (1 - x) * (1 - x ** a) * (1 - x ** b - x ** c) 
 - 2 * x ** (4 - 2 * a - 3 * b - 6 * c) * (1 - x ** b - x ** c) # mu, mu'
 - 2 * x ** (1 + c) * (1 - x ** a), # nu, nu'
 14.3, 0.15, 0.41, 0.24),

("mu meets beta, mu does not meet delta, nu does not meet beta - 2", # Using the inequality q \leq s
 (1 - x) * (1 - x ** a) * (1 - x ** c - x ** c) 
 - 2 * x ** (4 - 2 * a - 3 * c - 6 * c) * (1 - x ** c - x ** c) # mu, mu'
 - 2 * x ** (1 + c) * (1 - x ** a), # nu, nu'
 14.9, 0.15, 1, 0.3),
]

In [83]:
for i, (name, function, zero_start, a_start, b_start, c_start) in enumerate(Case_V_b):
    print(f"Evaluating {name}: {function} ...")
    MinPFRootApprox(function, starting_x = 1/zero_start, a_start = a_start, b_start = b_start, c_start = c_start, a_range = 0.01, b_range = 0.01, c_range = 0.01)

Evaluating beta' meets beta, beta' meets delta: -2*x**(-3*a - 3*b - 3*c + 3)*(1 - x**b) + (1 - x)*(1 - x**a)*(-x**b - x**c + 1) ...
The minimum PF zero is 22.9146914938988 within margin of 0.01, attained around a=0.09187499999999998, b=0.22125000000000003, c=0.3131250000000001, d=0.5 within margin of 0.001

Evaluating beta' does not meet beta, beta' meets delta: -2*x**(-1.5*b - 1.5*c + 1.5)*(1 - x**b) + (1 - x)*(-x**b - x**c + 1) ...
The minimum PF zero is 20.1657734102614 within margin of 0.01, attained around a=1, b=0.23062500000000002, c=0.400625, d=0.5 within margin of 0.001

Evaluating beta' meets beta, beta' does not meet delta, nu meets beta, nu meets beta': -2*x**(a + c + 1) - 2*x**(-3*a - 3*b - 3*c + 3)*(-x**b - x**c + 1) + (1 - x)*(1 - x**a)*(-x**b - x**c + 1) ...
The minimum PF zero is 19.2399847298655 within margin of 0.01, attained around a=0.195, b=0.23500000000000001, c=0.33, d=0.5 within margin of 0.001

Evaluating beta' meets beta, beta' does not meet delta, nu meets b

In [84]:
Lemma_A_8_3 = [
    
# If mu and nu meet gamma
    
("mu meets other petal curves", 
 (1 - x) * (1 - x ** a) * (1 - x ** a) * (1 - x ** c - x ** (1 - 2 * a - 2 * a - c)) 
 - 2 * x ** (2 - a - a) * (1 - x ** c - x ** (1 - 2 * a - 2 * a - c)) # mu, mu'
 - 2 * x ** (2 - a - a) * (1 - x ** c - x ** (1 - 2 * a - 2 * a)), # nu, nu'
 15.1, 0.09, 1, 0.2, 0.01),

("mu and nu do not meet other petal curves", # then there exists omega meeting a petal curve other than (beta and gamma), and filaments 
 (1 - x) * (1 - x ** a) * (1 - x ** a) * (1 - x ** c - x ** (1 - 2 * a - 2 * a - c)) 
 - 4 * x ** (1 + a + a) * (1 - x ** c - x ** (1 - 2 * a - 2 * a - c)) # mu, mu', nu, nu'
 - 2 * x ** (1 + a + a + c) * (1 - x ** a) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * a - c) - x ** (1 - 2 * a - 2 * a)), # omega, omega'
 20.8, 0.129, 1, 0.29, 0.05),

# If mu meets gamma, nu does not meet gamma
# then there exists omega exiting gamma at the same vertex as mu
# We can assume omega does not meet beta

("mu and nu and omega meet other petal curves, there is a proper nonempty subset of petal curves met by mu and nu and omega", 
 (1 - x) * (1 - x ** a) * (1 - x ** a) * (1 - x ** c - x ** (1 - 2 * a - 2 * a - c)) 
 - 2 * x ** (2 - a - a) * (1 - x ** c - x ** (1 - 2 * a - 2 * a)) # mu, mu'
 - 2 * x ** (2 - a - 2 * a) * (1 - x ** a) * (1 - x ** c - x ** (1 - 2 * a - 2 * a)) # nu, nu'
 - 2 * x ** (2 - 2 * a - a) * (1 - x ** a) * (1 - x ** c - x ** (1 - 2 * a - 2 * a)), # omega, omega'
 15, 0.08, 1, 0.18, 0.01),

("mu and nu and omega meet other petal curves, there is a proper nonempty subset of petal curves met by nu and omega but not mu", 
 (1 - x) * (1 - x ** a) * (1 - x ** a) * (1 - x ** c - x ** (1 - 2 * a - 2 * a - c)) 
 - 2 * x ** (2 - a - a) * (1 - x ** (1 - 2 * a - 2 * a - c) - x ** (1 - 2 * a - 2 * a)) # mu, mu'
 - 2 * x ** (2 - a - 2 * a) * (1 - x ** a) * (1 - x ** c - x ** (1 - 2 * a - 2 * a)) # nu, nu'
 - 2 * x ** (2 - 2 * a - a) * (1 - x ** a) * (1 - x ** c - x ** (1 - 2 * a - 2 * a)), # omega, omega'
 16, 0.08, 1, 0.38, 0.01),

("mu and nu and omega meet other petal curves, there is a proper nonempty subset of petal curves met by mu and nu but not omega", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (2 - a - b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)) # mu, mu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)) # nu, nu'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * b - c) - x ** (1 - 2 * a - 2 * b)), # omega, omega'
 15.7, 0.08, 0.08, 0.25, 0.01),

("mu and omega meet other petal curves, nu does not meet other petal curves, there is a proper nonempty subset of petal curves met by mu and omega", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (2 - a - b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)) # mu, mu'
 - 2 * x ** (1 + a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)), # omega, omega'
 16.8, 0.11, 0.08, 0.19, 0.01),
    
("mu and omega meet other petal curves, nu does not meet other petal curves, there is no proper nonempty subset of petal curves met by mu and omega", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (2 - a - b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)) # mu, mu'
 - 2 * x ** (1 + a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * b - c) - x ** (1 - 2 * a - 2 * b)), # omega, omega'
 17.5, 0.1, 0.08, 0.26, 0.01),

("mu meets other petal curves, nu and omega do not meet other petal curves", 
 (1 - x) * (1 - x ** a) * (1 - x ** a) * (1 - x ** c - x ** (1 - 2 * a - 2 * a - c)) 
 - 2 * x ** (2 - a - a) * (1 - x ** c - x ** (1 - 2 * a - 2 * a)) # mu, mu'
 - 2 * x ** (1 + a) * (1 - x ** a) * (1 - x ** c - x ** (1 - 2 * a - 2 * a - c)) # nu, nu'
 - 2 * x ** (1 + a) * (1 - x ** a) * (1 - x ** c - x ** (1 - 2 * a - 2 * a - c)), # omega, omega'
 18.5, 0.11, 1, 0.19, 0.01),

("mu does not meet other petal curves, omega meets other petal curves", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a + b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)), # omega, omega'
 18.1, 0.11, 0.13, 0.22, 0.01),

("mu and nu and omega do not meet other petal curves", 
 (1 - x) * (1 - x ** a) * (1 - x ** a) * (1 - x ** c - x ** (1 - 2 * a - 2 * a - c)) 
 - 2 * x ** (1 + a + a) * (1 - x ** c - x ** (1 - 2 * a - 2 * a - c)) # mu, mu'
 - 4 * x ** (2 - a - 2 * a) * (1 - x ** a) * (1 - x ** c - x ** (1 - 2 * a - 2 * a - c)) # nu, nu'
 - 2 * x ** (1 + a + a + c) * (1 - x ** a) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * a - c) - x ** (1 - 2 * a - 2 * a)), # omega, omega'
 17.4, 0.12, 1, 0.31, 0.05),

# If mu and nu meet beta but not gamma, omega and chi meet gamma but not beta

("mu and nu and omega and chi do not meet delta", # then there exists upsilon meeting delta and filaments
 (1 - x) * (1 - x ** a) * (1 - x ** a) * ((1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * a - b)) - x ** c) 
 - 8 * x ** (2 - a - 2 * a - c) * (1 - x ** a) * ((1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * a - b)) - x ** c) # mu, mu', nu, nu', omega, omega', chi, chi'
 - 2 * x ** (2 - 2 * a - 2 * a) * (1 - x ** a) * (1 - x ** a) * (1 - x ** b), # upsilon, upsilon'
 16.9, 0.06, 0.37, 0.34, 0.01),
    
("mu meets delta, nu and omega and chi do not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * ((1 - x ** (1/2 - a - b)) ** 2 - x ** c) 
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) * ((1 - x ** (1/2 - a - b)) ** 2) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * ((1 - x ** (1/2 - a - b)) ** 2 - x ** c) # nu, nu'
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * ((1 - x ** (1/2 - a - b)) ** 2 - x ** c) # omega, omega'
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * ((1 - x ** (1/2 - a - b)) ** 2 - x ** c), # chi, chi'
 17.1, 0.07, 0.07, 0.42, 0.01),

("mu and omega meet delta, nu and chi do not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** a) * ((1 - x ** (1/2 - a - a)) ** 2 - x ** c) 
 - 4 * x ** (2 - a - 2 * a) * (1 - x ** a) * ((1 - x ** (1/2 - a - a)) ** 2) # mu, mu', nu, nu'
 - 4 * x ** (2 - a - 2 * a - c) * (1 - x ** a) * ((1 - x ** (1/2 - a - a)) ** 2 - x ** c), # omega, omega', chi, chi'
 16.8, 0.07, 1, 0.5, 0.01),

("mu and nu meet delta, omega and chi do not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * ((1 - x ** (1/2 - a - b)) ** 2 - x ** c) 
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) * ((1 - x ** (1/2 - a - b)) ** 2) # mu, mu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) * ((1 - x ** (1/2 - a - b)) ** 2) # nu, nu'
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * ((1 - x ** (1/2 - a - b)) ** 2 - x ** c) # omega, omega'
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * ((1 - x ** (1/2 - a - b)) ** 2 - x ** c), # chi, chi'
 16.6, 0.06, 0.09, 0.53, 0.01),
    
("mu and nu and omega meet delta, chi does not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * ((1 - x ** (1/2 - a - b)) ** 2 - x ** c) 
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) * ((1 - x ** (1/2 - a - b)) ** 2) # mu, mu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) * ((1 - x ** (1/2 - a - b)) ** 2) # nu, nu'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) * ((1 - x ** (1/2 - a - b)) ** 2) # omega, omega'
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * ((1 - x ** (1/2 - a - b)) ** 2 - x ** c), # chi, chi'
 15.2, 0.06, 0.09, 0.66, 0.01),

# # If mu and nu and omega and chi meet delta
# # let epsilon be one of the petal curves met by delta
    
("mu and nu and omega and chi do not meet epsilon", 
 (1 - x) * (1 - x ** a) * (1 - x ** a) * (1 - x ** c - x ** (1 - 2 * a - 2 * a - c)) 
 - 8 * x ** (2 - a - 2 * a - c) * (1 - x ** a) * (1 - x ** c), # mu, mu', nu, nu', omega, omega', chi, chi'
 16.7, 0.05, 1, 0.14, 0.01),

("chi meets epsilon, mu and nu and omega do not meet epsilon", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b - c)) 
 - 4 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # mu, mu', nu, nu'
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) # omega, omega'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * b - c)), # chi, chi'
 17.2, 0.05, 0.06, 0.19, 0.01),

("omega and chi meet epsilon, mu and nu do not meet epsilon", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b - c)) 
 - 4 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # mu, mu', nu, nu'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * b - c)) # omega, omega'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * b - c)), # chi, chi'
 17.1, 0.06, 0.05, 0.25, 0.01),

("nu and chi meet epsilon, mu and omega do not meet epsilon", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) # omega, omega'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * b - c)), # chi, chi'
 17.1, 0.06, 0.06, 0.24, 0.01),

("nu and omega and chi meet epsilon, mu does not meet epsilon", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 4 * x ** (2 - 2 * a - b) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * b - c)), # omega, omega', chi, chi'
 16.3, 0.06, 0.05, 0.33, 0.01),
    
("mu and nu and omega and chi meet all petal curves met by delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** a) * (1 - 2 * x ** (1/2 - a - a)) 
 - 8 * x ** (2 - a - a) * (1 - x ** a), # mu, mu', nu, nu', omega, omega', chi, chi'
 15.9, 0.05, 1, 1, 0.01),
]

In [85]:
for i, (name, function, zero_start, a_start, b_start, c_start, margin_x) in enumerate(Lemma_A_8_3):
    print(f"Evaluating {name}: {function} ...")
    MinPFRootApprox(function, starting_x = 1/zero_start, margin_x = margin_x, a_start = a_start, b_start = b_start, c_start = c_start, a_range = 0.005, b_range = 0.005, c_range = 0.01)

Evaluating mu meets other petal curves: -2*x**(2 - 2*a)*(-x**c - x**(1 - 4*a) + 1) - 2*x**(2 - 2*a)*(-x**c - x**(-4*a - c + 1) + 1) + (1 - x)*(1 - x**a)**2*(-x**c - x**(-4*a - c + 1) + 1) ...
The minimum PF zero is 15.1357699607505 within margin of 0.01, attained around a=0.09, b=1, c=0.2, d=0.5 within margin of 0.001

Evaluating mu and nu do not meet other petal curves: -4*x**(2*a + 1)*(-x**c - x**(-4*a - c + 1) + 1) - 2*x**(2*a + c + 1)*(1 - x**a)**2*(-x**(1 - 4*a) - x**(-4*a - c + 1) + 1) + (1 - x)*(1 - x**a)**2*(-x**c - x**(-4*a - c + 1) + 1) ...
The minimum PF zero is 20.9075399316982 within margin of 0.05, attained around a=0.12853125, b=1, c=0.28531250000000014, d=0.5 within margin of 0.001

Evaluating mu and nu and omega meet other petal curves, there is a proper nonempty subset of petal curves met by mu and nu and omega: -4*x**(2 - 3*a)*(1 - x**a)*(-x**c - x**(1 - 4*a) + 1) - 2*x**(2 - 2*a)*(-x**c - x**(1 - 4*a) + 1) + (1 - x)*(1 - x**a)**2*(-x**c - x**(-4*a - c + 1) + 1) ...


In [86]:
Lemma_A_8_4_1 = [
# Let mu be a curve that exits delta at the same vertex as epsilon
# If mu meets eta

("mu meets beta and gamma", 
 (1 - x) * (1 - x ** a) * (1 - x ** a) * (1 - 2 * x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * a - c)) 
 - 2 * x ** (2 - a - a) * (1 - x ** (1 - 2 * a - 2 * a - c)), # mu, mu'
 21.9, 0.07, 1, 0.38),

# # If mu meets beta but not gamma
# # then there exists nu, omega exiting gamma at the same vertex
    
("omega meets beta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - 2 * x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c))
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (2 - a - b) * (1 - x ** a) * (1 - 2 * x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 2 * x ** (2 - a - b) * (1 - 2 * x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)), # omega, omega'
 16.4, 0.09, 0.05, 0.38),

("nu and omega do not meet beta, omega meets delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - 2 * x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c))
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) * (1 - 2 * x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)), # omega, omega'
 14.9, 0.05, 0.04, 0.44),

("nu and omega do not meet beta, nu and omega do not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - 2 * x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c))
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - 2 * x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - 2 * x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)), # omega, omega'
 16.7, 0.06, 0.06, 0.37),

# # If mu does not meet beta nor gamma
# # let nu and omega be curves that exit beta at the same vertex

("nu meets gamma", 
 (1 - x) * (1 - x ** a) * (1 - x ** a) * (1 - 2 * x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * a - c))
 - 2 * x ** (2 - 2 * a - 2 * a) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * a - c)) # mu, mu'
 - 2 * x ** (2 - a - a) * (1 - 2 * x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * a - c))
 - 2 * x ** (2 - a - a) * (1 - x ** a) * (1 - 2 * x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * a - c)), 
 17.4, 0.06, 1, 0.39),
    
("nu and omega do not meet gamma", # then there exists curves chi and psi meeting gamma and filaments, we can assume chi and psi do not meet beta
 (1 - x) * (1 - x ** a) * (1 - x ** a) * (1 - 2 * x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * a - c))
 - 2 * x ** (2 - 2 * a - 2 * a) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * a - c)) # mu, mu'
 - 8 * x ** (2 - a - a) * (1 - x ** a) * (1 - 2 * x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * a - c)), 
 15.8, 0.05, 1, 0.43),

# If mu does not meet eta
# let nu be a curve that exits eta at the same vertex as theta
# We can assume nu does not meet delta
    
("mu meets beta and gamma", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - 2 * x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a + b + c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (2 - a - b - c) * (1 - x ** a) * (1 - x ** b) * (1 - 2 * x ** c), # nu, nu'
 21.7, 0.083, 0.083, 0.404),

("mu meets beta but not gamma, nu meets gamma but not beta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - 2 * x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a + c) * (1 - x ** b) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - 2 * x ** c), # nu, nu'
 19.6, 0.06, 0.06, 0.37),

("mu meets beta but not gamma, nu meets beta but not gamma", # then there exists omega meeting gamma and filaments
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - 2 * x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a + c) * (1 - x ** b) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - 2 * x ** c) # nu, nu'
 - 2 * x ** (2 - a - b) * (1 - x ** a) * (1 - 2 * x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)), # omega, omega'
 16.9, 0.11, 0.02, 0.37),

# # If mu meets beta but not gamma, nu does not meet beta nor gamma
# # then there exists omega exiting gamma at the same vertex as mu

("omega meets beta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - 2 * x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a + c) * (1 - x ** b) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (2 - 2 * a - 2 * b - c) * (1 - x ** a) * (1 - x ** b) * (1 - 2 * x ** c) # nu, nu'
 - 2 * x ** (2 - a - b) * (1 - 2 * x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)), # omega, omega'
 17, 0.1, 0.05, 0.39),

("omega does not meet beta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - 2 * x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a + c) * (1 - x ** b) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (2 - 2 * a - 2 * b - c) * (1 - x ** a) * (1 - x ** b) * (1 - 2 * x ** c) # nu, nu'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) * (1 - 2 * x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)), # omega, omega'
 14.5, 0.07, 0.03, 0.45),

# # If mu and nu do not meet beta nor gamma
# # then there exists omega meeting beta and filaments
    
("omega meets gamma", 
 (1 - x) * (1 - x ** a) * (1 - x ** a) * (1 - 2 * x ** (1/2 - a - a)) * (1 - 2 * x ** (1/2 - a - a)) 
 - 4 * x ** (3/2 - a - a) * (1 - x ** a) * (1 - x ** a) * (1 - 2 * x ** (1/2 - a - a)) # mu, mu', nu, nu'
 - 2 * x ** (1 + a + a) * (1 - 2 * x ** (1/2 - a - a)) * (1 - 2 * x ** (1/2 - a - a)), # omega, omega'
 22.7, 0.11, 1, 1),

("omega does not meet gamma", # then there exists chi meeting and filaments, we can assume chi does not meet beta
 (1 - x) * (1 - x ** a) * (1 - x ** a) * (1 - 2 * x ** (1/2 - a - a)) * (1 - 2 * x ** (1/2 - a - a)) 
 - 4 * x ** (3/2 - a - a) * (1 - x ** a) * (1 - x ** a) * (1 - 2 * x ** (1/2 - a - a)) # mu, mu', nu, nu'
 - 4 * x ** (1 + a) * (1 - x ** a) * (1 - 2 * x ** (1/2 - a - a)) * (1 - 2 * x ** (1/2 - a - a)), # omega, omega', chi, chi'
 20.2, 0.09, 1, 1),
]

In [87]:
for i, (name, function, zero_start, a_start, b_start, c_start) in enumerate(Lemma_A_8_4_1):
    print(f"Evaluating {name}: {function} ...")
    MinPFRootApprox(function, starting_x = 1/zero_start, a_start = a_start, b_start = b_start, c_start = c_start, a_range = 0.005, b_range = 0.005, c_range = 0.01)

Evaluating mu meets beta and gamma: -2*x**(2 - 2*a)*(1 - x**(-4*a - c + 1)) + (1 - x)*(1 - x**a)**2*(1 - 2*x**c)*(1 - 2*x**(-4*a - c + 1)) ...
The minimum PF zero is 21.9698021010332 within margin of 0.01, attained around a=0.06812499999999999, b=1, c=0.3793749999999999, d=0.5 within margin of 0.001

Evaluating omega meets beta: -2*x**(-a - 2*b + 2)*(1 - x**b)*(1 - x**(-2*a - 2*b - c + 1)) - 2*x**(-a - b + 2)*(1 - x**a)*(1 - 2*x**c)*(1 - 2*x**(-2*a - 2*b - c + 1)) - 2*x**(-a - b + 2)*(1 - 2*x**c)*(1 - 2*x**(-2*a - 2*b - c + 1)) + (1 - x)*(1 - x**a)*(1 - x**b)*(1 - 2*x**c)*(1 - 2*x**(-2*a - 2*b - c + 1)) ...
The minimum PF zero is 16.4690167247758 within margin of 0.01, attained around a=0.08843749999999997, b=0.046875, c=0.37625, d=0.5 within margin of 0.001

Evaluating nu and omega do not meet beta, omega meets delta: -2*x**(-2*a - b + 2)*(1 - x**a)*(1 - 2*x**c)*(1 - 2*x**(-2*a - 2*b - c + 1)) - 2*x**(-2*a - b + 2)*(1 - x**a)*(1 - x**c)*(1 - 2*x**(-2*a - 2*b - c + 1)) - 2*x**(-a - 2*b

In [88]:
Lemma_A_8_4_2 = [
    
# Let mu be a curve that meets eta and filaments

("mu meets beta and gamma and delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** a) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * a - c)) 
 - 2 * x ** (2 - a - a), # mu, mu'
 18.4, 0.09, 1, 0.21, 1),

# If mu meet beta and gamma but not delta
# then there exists nu and omega meeting delta and filaments

("nu meets beta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (2 - a - b - c) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - b) * (1 - x ** b) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 2 * x ** (2 - a - b) * (1 - x ** a) * (1 - x ** b) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)), # omega, omega'
 15.6, 0.12, 0.09, 0.08, 1),
    
("nu and omega do not meet beta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (2 - a - b - c) * (1 - x ** c) # mu, mu'
 - 4 * x ** (2 - 2 * a - b) * (1 - x ** a) * (1 - x ** b) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)), # nu, nu', omega, omega'
 14.5, 0.09, 0.1, 0.07, 1),

# If mu meets beta and delta but not gamma
# let nu be a curve exiting beta at the same vertex as mu

# # If nu meets theta
# # we can assume nu does not meet gamma

# # # If nu meets delta
# # # then there exists omega and chi meeting gamma and filaments
    
("omega meets beta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) 
 - 4 * x ** (2 - a - 2 * b) * (1 - x ** b) # mu, mu', nu, nu'
 - 2 * x ** (2 - a - b) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) # omega, omega'
 - 2 * x ** (2 - a - b) * (1 - x ** a) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)), # chi, chi'
 16.2, 0.12, 0.05, 0.21, 1),
    
("omega and chi do not meet beta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) 
 - 4 * x ** (2 - a - 2 * b) * (1 - x ** b) # mu, mu', nu, nu'
 - 4 * x ** (2 - 2 * a - b) * (1 - x ** a) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)), # omega, omega', chi, chi'
 14.6, 0.08, 0.04, 0.25, 1),

# # # If nu does not meet delta
# # # then there exists omega exiting delta at the same vertex as mu, chi meeting gamma and filaments
    
("omega meets gamma, chi meets beta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # nu, nu'
 - 2 * x ** (2 - a - b) * (1 - x ** a) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) # omega, omega'
 - 2 * x ** (2 - a - b) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)), # chi, chi' 
 16.5, 0.11, 0.06, 0.18, 1),
    
("omega meets gamma, chi does not meet beta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # nu, nu'
 - 2 * x ** (2 - a - b) * (1 - x ** a) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) # omega, omega'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)), # chi, chi'
 14.9, 0.09, 0.05, 0.2, 1),

("omega does not meet gamma, chi meets beta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # nu, nu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** a) * (1 - x ** b) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) # omega, omega'
 - 2 * x ** (2 - a - b) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)), # chi, chi'
 15.2, 0.13, 0.05, 0.15, 1),

("omega does not meet gamma, chi does not meet beta, chi meets delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # nu, nu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** a) * (1 - x ** b) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) # omega, omega'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)), # chi, chi'
 14.7, 0.08, 0.04, 0.22, 1),
    
("omega does not meet gamma, chi does not meet beta and delta", # then there exists upsilon exiting gamma at the same vertex as chi, we can assume upsilon does not meet beta and delta
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # nu, nu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** a) * (1 - x ** b) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) # omega, omega'
 - 4 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)), # chi, chi', upsilon, upsilon'
 15.2, 0.09, 0.05, 0.14, 1),
    
# # If nu does not meet theta

("nu meets gamma", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # mu, mu'
 - 2 * x ** (2 - a - b - d) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)), # nu, nu'
 17, 0.14, 0.07, 0.2, 0.34),

("nu meets delta but not gamma", # then there exists omega, chi meeting gamma and filaments
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - d) * (1 - x ** b) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 4 * x ** (2 - a - b) * (1 - x ** a) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)), # omega, omega', chi, chi'
 14.6, 0.12, 0.04, 0.34, 0.45),

# # # If nu does not meet gamma and delta
# # # then there exists omega exiting delta at the same vertex as mu
    
("omega meets gamma", # then there exists chi exiting gamma at same vertex as omega
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - c - d) * (1 - x ** b) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 2 * x ** (2 - a - b) * (1 - x ** a) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) # omega, omega'
 - 2 * x ** (2 - a - b) * (1 - x ** a) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)), # chi, chi'
 15.6, 0.12, 0.05, 0.23, 0.4),

("omega does not meet gamma", # then there exists chi, upsilon meeting gamma and filaments
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c))
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - c - d) * (1 - x ** b) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** a) * (1 - x ** b) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) # omega, omega'
 - 4 * x ** (2 - a - b) * (1 - x ** a) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)), # chi, chi', upsilon, upsilon'
 14.8, 0.13, 0.04, 0.2, 0.42),
]

In [89]:
for i, (name, function, zero_start, a_start, b_start, c_start, d_start) in enumerate(Lemma_A_8_4_2):
    print(f"Evaluating {name}: {function} ...")
    MinPFRootApprox(function, starting_x = 1/zero_start, a_start = a_start, b_start = b_start, c_start = c_start, d_start = d_start, a_range = 0.01, b_range = 0.01, c_range = 0.01, d_range = 0.01)

Evaluating mu meets beta and gamma and delta: -2*x**(2 - 2*a) + (1 - x)*(1 - x**a)**2*(1 - x**c)*(1 - 2*x**(-4*a - c + 1)) ...
The minimum PF zero is 18.4540607836549 within margin of 0.01, attained around a=0.08749999999999998, b=1, c=0.20625000000000002, d=1 within margin of 0.001

Evaluating nu meets beta: -2*x**(-a - b + 2)*(1 - x**a)*(1 - x**b)*(1 - 2*x**(-2*a - 2*b - c + 1)) - 2*x**(-a - b + 2)*(1 - x**b)*(1 - 2*x**(-2*a - 2*b - c + 1)) - 2*x**(-a - b - c + 2)*(1 - x**c) + (1 - x)*(1 - x**a)*(1 - x**b)*(1 - x**c)*(1 - 2*x**(-2*a - 2*b - c + 1)) ...
The minimum PF zero is 15.6108743195992 within margin of 0.01, attained around a=0.12000000000000001, b=0.08937499999999998, c=0.08375000000000002, d=1 within margin of 0.001

Evaluating nu and omega do not meet beta: -4*x**(-2*a - b + 2)*(1 - x**a)*(1 - x**b)*(1 - 2*x**(-2*a - 2*b - c + 1)) - 2*x**(-a - b - c + 2)*(1 - x**c) + (1 - x)*(1 - x**a)*(1 - x**b)*(1 - x**c)*(1 - 2*x**(-2*a - 2*b - c + 1)) ...
The minimum PF zero is 14.529051

In [90]:
Lemma_A_8_4_3 = [

# If mu meets beta but not gamma and delta
# let nu be a curve exiting beta at the same vertex as mu
    
# # If nu meets theta
# # we can assume nu does not meet gamma

# # # If nu meets delta
# # # then there exists omega and chi meeting gamma and filaments

("omega meets beta and delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # nu, nu'
 - 2 * x ** (2 - a - b) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)), # omega, omega'
 17.6, 0.12, 0.06, 0.22, 1),

("omega meets beta but not delta, chi meets beta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # nu, nu'
 - 2 * x ** (2 - a - b - c) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) # omega, omega'
 - 2 * x ** (2 - a - b) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)), # chi, chi'
 17.4, 0.14, 0.07, 0.11, 1),
    
("omega meets beta but not delta, chi does not meet beta and theta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # nu, nu'
 - 2 * x ** (2 - a - b - c) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) # omega, omega'
 - 2 * x ** (2 - 2 * a - b - c - d) * (1 - x ** a) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)), # chi, chi'
 19.3, 0.11, 0.08, 0.11, 0.4),

("omega meets beta but not delta, chi does not meet beta, chi meets theta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # nu, nu'
 - 2 * x ** (2 - a - b - c) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) # omega, omega'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) * (1 - x ** c), # chi, chi'
 17.1, 0.11, 0.07, 0.11, 1),

# # # We can now assume that omega does not meet beta, similarly chi does not meet beta
    
("omega and chi meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # nu, nu'
 - 4 * x ** (2 - 2 * a - b) * (1 - x ** a) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)), # omega, omega', chi, chi'
 16.1, 0.07, 0.05, 0.23, 1),   

("omega meets delta, chi does not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # nu, nu'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) # omega, omega'
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)), # chi, chi'
 15.8, 0.08, 0.06, 0.19, 1),   

("omega and chi do not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c))  
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # nu, nu'
 - 4 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)), # omega, omega', chi, chi'
 14.9, 0.09, 0.05, 0.13, 1),
    
# # # If nu does not meet delta
# # # then there exists omega and chi meeting gamma and filaments

("omega meets beta and delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) 
 - 4 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # mu, mu', nu, nu'
 - 2 * x ** (2 - a - b) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)), # omega, omega'
 17, 0.13, 0.07, 0.16, 1), 

("omega meets beta but not delta, chi meets beta", # then there exists upsilon meeting delta and filaments
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) 
 - 4 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # mu, mu', nu, nu'
 - 2 * x ** (2 - a - b - c) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) # omega, omega'
 - 2 * x ** (2 - a - b) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) # chi, chi'
 - 2 * x ** (2 - a - b) * (1 - x ** a) * (1 - x ** b) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)), # upsilon, upsilon'
 15.2, 0.15, 0.09, 0.05, 1),

("omega meets beta but not delta, chi does not meet beta and theta", # then there exists upsilon meeting delta and filaments
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) 
 - 4 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # mu, mu', nu, nu'
 - 2 * x ** (2 - a - b - c) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) # omega, omega'
 - 2 * x ** (2 - 2 * a - b - c - d) * (1 - x ** a) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) # chi, chi'
 - 2 * x ** (2 - a - b) * (1 - x ** a) * (1 - x ** b) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)), # upsilon, upsilon'
 16.4, 0.11, 0.1, 0.04, 0.38),

("omega meets beta but not delta, chi does not meet beta, chi meets theta", # then there exists upsilon meeting delta and filaments
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) 
 - 4 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # mu, mu', nu, nu'
 - 2 * x ** (2 - a - b - c) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) # omega, omega'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) * (1 - x ** c) # chi, chi'
 - 2 * x ** (2 - a - b) * (1 - x ** a) * (1 - x ** b) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)), # upsilon, upsilon'
 14.9, 0.11, 0.09, 0.05, 1),

("omega and chi meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) 
 - 4 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # mu, mu', nu, nu'
 - 4 * x ** (2 - 2 * a - b) * (1 - x ** a) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)), # omega, omega', chi, chi'
 15.5, 0.07, 0.07, 0.16, 1),   

("omega meets delta, chi does not meet delta", # then there exists upsilon meeting delta and filaments
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) 
 - 4 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # mu, mu', nu, nu'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) # omega, omega'
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) # chi, chi'
 - 2 * x ** (2 - 2 * a - 2 * b) * (1 - x ** a) * (1 - x ** b) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)), # upsilon, upsilon'
 14.9, 0.07, 0.07, 0.13, 1),   

# # # # If omega and chi do not meet delta
# # # # then there exists upsilon, psi meeting delta and filaments
    
("omega does not meet theta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) 
 - 4 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # mu, mu', nu, nu'
 - 2 * x ** (2 - 2 * a - b - c - d) * (1 - x ** a) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) # omega, omega'
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) # chi, chi'
 - 4 * x ** (2 - 2 * a - 2 * b) * (1 - x ** a) * (1 - x ** b) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)), # upsilon, upsilon', psi, psi' 
 16.1, 0.07, 0.07, 0.07, 0.38),

("upsilon does not meet theta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) 
 - 4 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # mu, mu', nu, nu'
 - 4 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) # omega, omega', chi, chi'
 - 2 * x ** (2 - 2 * a - 2 * b - d) * (1 - x ** a) * (1 - x ** b) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) # upsilon, upsilon'
 - 2 * x ** (2 - 2 * a - 2 * b) * (1 - x ** a) * (1 - x ** b) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)), # psi, psi'
15.5, 0.07, 0.05, 0.11, 0.44),
    
("omega and chi and upsilon and psi meet theta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * b - c)) 
 - 4 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # mu, mu', nu, nu'
 - 4 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) # omega, omega', chi, chi'
 - 4 * x ** (2 - 2 * a - 2 * b) * (1 - x ** a) * (1 - x ** b), # upsilon, upsilon', psi, psi'
 15, 0.05, 0.05, 0.07, 1),

# # If nu does not meet theta

("nu meets gamma", # then there exists omega meeting delta and filaments
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - b - d) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 2 * x ** (2 - a - b) * (1 - x ** a) * (1 - x ** b) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)), # omega, omega'
 14.9, 0.15, 0.09, 0.06, 0.29),

# # # If nu meets delta
# # # then there exists omega and chi meeting gamma and filaments
    
("omega meets beta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - d) * (1 - x ** b) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c))  # nu, nu'
 - 2 * x ** (2 - a - b) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c))  # omega, omega'
 - 2 * x ** (2 - a - b) * (1 - x ** a) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) , # chi, chi'
 16.1, 0.16, 0.06, 0.17, 0.35),

("omega meets delta but not beta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - d) * (1 - x ** b) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c))  # nu, nu'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c))  # omega, omega'
 - 2 * x ** (2 - a - b) * (1 - x ** a) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) , # chi, chi'
 16.1, 0.11, 0.05, 0.24, 0.38),

("omega and chi do not meet beta and delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c))  
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - d) * (1 - x ** b) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c))  # nu, nu'
 - 4 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)), # omega, omega', chi, chi'
 16.3, 0.11, 0.06, 0.14, 0.28),
    
# # # If nu does not meet delta
# # # then there exists omega and chi meeting gamma and filaments

("omega meets beta and delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - c - d) * (1 - x ** b) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c))  # nu, nu'
 - 2 * x ** (2 - a - b) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c))  # omega, omega'
 - 2 * x ** (2 - a - b) * (1 - x ** a) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) , # chi, chi'
 17.6, 0.16, 0.08, 0.17, 0.4),

("omega meets delta but not beta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - c - d) * (1 - x ** b) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c))  # nu, nu'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c))  # omega, omega'
 - 2 * x ** (2 - a - b) * (1 - x ** a) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) , # chi, chi'
 15.2, 0.1, 0.07, 0.14, 0.36),

# # # We can now assume that omega and chi do not meet delta
# # # then there exists upsilon and psi meeting delta and filaments
# # # We can assume upsilon and psi do not meet gamma
    
("omega meets beta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - c - d) * (1 - x ** b) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c))  # nu, nu'
 - 2 * x ** (2 - a - b - c) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c))  # omega, omega'
 - 2 * x ** (2 - a - b - c) * (1 - x ** a) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) # chi, chi'
 - 4 * x ** (2 - a - 2 * b) * (1 - x ** a) * (1 - x ** b) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)), # upsilon, upsilon', psi, psi'
 15.8, 0.15, 0.07, 0.07, 0.37),

# # # We can now assume that omega and chi do not meet beta
    
("omega and chi do not meet beta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c))  
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - c - d) * (1 - x ** b) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c))  # nu, nu'
 - 4 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) # omega, omega', chi, chi'
 - 4 * x ** (2 - a - 2 * b) * (1 - x ** a) * (1 - x ** b) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)), # upsilon, upsilon', psi, psi'
 14.6, 0.1, 0.06, 0.07, 0.34),

# We can now assume that mu does not meet beta
# similarly mu does not meet gamma
]

In [91]:
for i, (name, function, zero_start, a_start, b_start, c_start, d_start) in enumerate(Lemma_A_8_4_3):
    print(f"Evaluating {name}: {function} ...")
    MinPFRootApprox(function, starting_x = 1/zero_start, a_start = a_start, b_start = b_start, c_start = c_start, d_start = d_start, a_range = 0.01, b_range = 0.01, c_range = 0.01, d_range = 0.01)

Evaluating omega meets beta and delta: -2*x**(-a - 2*b + 2)*(1 - x**b) - 2*x**(-a - b + 2)*(1 - 2*x**(-2*a - 2*b - c + 1)) - 2*x**(-a - 2*b - c + 2)*(1 - x**b)*(1 - x**c) + (1 - x)*(1 - x**a)*(1 - x**b)*(1 - x**c)*(1 - 2*x**(-2*a - 2*b - c + 1)) ...
The minimum PF zero is 17.6116808161958 within margin of 0.01, attained around a=0.12187500000000002, b=0.0596875, c=0.216875, d=1 within margin of 0.001

Evaluating omega meets beta but not delta, chi meets beta: -2*x**(-a - 2*b + 2)*(1 - x**b) - 2*x**(-a - b + 2)*(1 - x**c)*(1 - 2*x**(-2*a - 2*b - c + 1)) - 2*x**(-a - 2*b - c + 2)*(1 - x**b)*(1 - x**c) - 2*x**(-a - b - c + 2)*(1 - x**c)*(1 - 2*x**(-2*a - 2*b - c + 1)) + (1 - x)*(1 - x**a)*(1 - x**b)*(1 - x**c)*(1 - 2*x**(-2*a - 2*b - c + 1)) ...
The minimum PF zero is 17.4524959104342 within margin of 0.01, attained around a=0.14125, b=0.06624999999999999, c=0.11062500000000001, d=1 within margin of 0.001

Evaluating omega meets beta but not delta, chi does not meet beta and theta: -2*x**

In [92]:
Lemma_A_8_4_4 = [

# If mu meets delta
# then there exists nu exiting delta at the same vertex as mu
    
# # If nu meets theta, we can assume that it does not meet beta and gamma
# # then there exists omega and chi meeting beta and filaments

("omega and chi meets gamma", 
 (1 - x) * (1 - x ** a) * (1 - x ** a) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * a - c)) 
 - 4 * x ** (2 - 2 * a - 2 * a) * (1 - x ** a) * (1 - x ** a) # mu, mu', nu, nu'
 - 4 * x ** (2 - a - a) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * a - c)), # omega, omega', chi, chi'
 16.6, 0.09, 1, 0.12, 1),

("omega meets gamma, chi does not meet gamma", # then there exists upsilon meeting gamma and filaments 
 (1 - x) * (1 - x ** a) * (1 - x ** a) * (1 - x ** c) * (1 - 2 * x ** (1 - 2 * a - 2 * a - c)) 
 - 4 * x ** (2 - 2 * a - 2 * a) * (1 - x ** a) * (1 - x ** a) # mu, mu', nu, nu'
 - 2 * x ** (2 - a - 2 * a) * (1 - x ** a) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * a - c)) # omega, omega'
 - 2 * x ** (2 - a - a) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * a - c)) # chi, chi'
 - 2 * x ** (2 - 2 * a - a) * (1 - x ** a) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * a - c)), # upsilon, upsilon'
 15.4, 0.08, 1, 0.13, 1),

("omega and chi do not meet gamma", # then there exists upsilon and psi meeting gamma and filaments 
 (1 - x) * (1 - x ** a) * (1 - x ** a) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * a - c)) 
 - 4 * x ** (2 - 2 * a - 2 * a) * (1 - x ** a) * (1 - x ** a) # mu, mu', nu, nu'
 - 4 * x ** (2 - a - 2 * a - d) * (1 - x ** a) * (1 - x ** c) * (1 - x ** d) # omega, omega', chi, chi'
 - 4 * x ** (2 - 2 * a - a - d) * (1 - x ** a) * (1 - x ** c) * (1 - x ** d), # upsilon, upsilon', psi, psi'
 18.3, 0.05, 1, 0.13, 0.21),

# # If nu does not meet theta

("nu meets beta and gamma", 
 (1 - x) * (1 - x ** a) * (1 - x ** a) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * a - c)) 
 - 2 * x ** (2 - 2 * a - 2 * a) * (1 - x ** a) * (1 - x ** a) # mu, mu'
 - 2 * x ** (2 - a - a - d) * (1 - x ** d - x ** (1 - 2 * a - 2 * a - c)), # nu, nu'
 17.7, 0.09, 1, 0.32, 0.27),

# We can now assume that nu does not meet gamma
# then there exists omega and chi meeting gamma and filaments, we can assume omega and chi do not meet theta
    
("nu meets beta, omega meets beta",
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (2 - 2 * a - 2 * b) * (1 - x ** a) * (1 - x ** b) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - d) * (1 - x ** b) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 2 * x ** (2 - a - b - d) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) # omega, omega'
 - 2 * x ** (2 - a - b - d) * (1 - x ** a) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)), # chi, chi' 
 17.8, 0.1, 0.08, 0.24, 0.22),

("nu meets beta, omega and chi do not meet beta", # then there exists upsilon exiting beta at the same vertex as nu, we can assume upsilon does not meet theta
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (2 - 2 * a - 2 * b) * (1 - x ** a) * (1 - x ** b) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - d) * (1 - x ** b) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 4 * x ** (2 - 2 * a - b - d) * (1 - x ** a) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) # omega, omega', chi, chi' 
 - 2 * x ** (2 - a - b - d) * (1 - x ** b) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)), # upsilon, upsilon'
 15.4, 0.06, 0.05, 0.3, 0.2),

("nu does not meet beta, omega meets beta",
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (2 - 2 * a - 2 * b) * (1 - x ** a) * (1 - x ** b) # mu, mu'
 - 2 * x ** (2 - 2 * a - 2 * b - d) * (1 - x ** a) * (1 - x ** b) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 2 * x ** (2 - a - b - d) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) # omega, omega'
 - 2 * x ** (2 - a - b - d) * (1 - x ** a) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)), # chi, chi' 
 16.2, 0.08, 0.09, 0.22, 0.23),

("nu does not meet beta, omega and chi do not meet beta", # then there exists upsilon and psi meeting beta and filaments, we can assume upsilon and psi do not meet gamma
 (1 - x) * (1 - x ** a) * (1 - x ** a) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * a - c)) 
 - 2 * x ** (2 - 2 * a - 2 * a) * (1 - x ** a) * (1 - x ** a) # mu, mu'
 - 2 * x ** (2 - 2 * a - 2 * a - d) * (1 - x ** a) * (1 - x ** a) * (1 - x ** d - x ** (1 - 2 * a - 2 * a - c)) # nu, nu'
 - 4 * x ** (2 - 2 * a - a - d) * (1 - x ** a) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * a - c)) # omega, omega', chi, chi' 
 - 4 * x ** (2 - a - 2 * a - d) * (1 - x ** a) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * a - c)), # upsilon, upsilon', psi, psi'
 14.9, 0.06, 1, 0.24, 0.19),

# # We can now assume that mu does not meet beta and gamma and delta
# # then there exists nu, omega meeting delta and filaments
# # We can assume nu, omega does not meet theta

("nu meets beta and gamma", 
 (1 - x) * (1 - x ** a) * (1 - x ** a) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * a - c)) 
 - 2 * x ** (2 - 2 * a - 2 * a - c) * (1 - x ** a) * (1 - x ** a) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - a - d) * (1 - x ** d - x ** (1 - 2 * a - 2 * a - c)) # nu, nu'
 - 2 * x ** (2 - a - a - d) * (1 - x ** a) * (1 - x ** a) * (1 - x ** d - x ** (1 - 2 * a - 2 * a - c)), # omega, omega'
 18.4, 0.1, 1, 0.24, 0.23),

("nu meets beta but not gamma, omega meets gamma but not beta", # then there exists chi exiting beta from the same vertex as nu, we can assume beta does not meet theta
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (2 - 2 * a - 2 * b - c) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - d) * (1 - x ** b) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 2 * x ** (2 - 2 * a - b - d) * (1 - x ** a) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) # omega, omega'
 - 2 * x ** (2 - a - b) * (1 - x ** b) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)), # chi, chi'
 15, 0.06, 0.06, 0.29, 0.22),

# # We can now assume that nu and omega do not meet beta
# # then there exists chi and upsilon meeting beta and filaments, we can assume chi and upsilon do not meet gamma and delta and theta
    
("nu meets gamma",
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (2 - 2 * a - 2 * b - c) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - 2 * a - b - d) * (1 - x ** a) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 2 * x ** (2 - 2 * a - b - d) * (1 - x ** a) * (1 - x ** b) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)) # omega, omega'
 - 4 * x ** (2 - a - 2 * b - c - d) * (1 - x ** b) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * b - c)), # chi, chi', upsilon, upsilon'
 16.1, 0.06, 0.06, 0.15, 0.14),

("nu and omega do not meet gamma", # then there exists psi and phi meeting gamma and filaments, we can assume psi and phi do not meet beta and delta and theta
 (1 - x) * (1 - x ** a) * (1 - x ** a) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * a - c)) 
 - 2 * x ** (2 - 2 * a - 2 * a - c) * (1 - x ** a) * (1 - x ** a) * (1 - x ** c) # mu, mu'
 - 4 * x ** (2 - 2 * a - 2 * a - d) * (1 - x ** a) * (1 - x ** a) * (1 - x ** d - x ** (1 - 2 * a - 2 * a - c)) # nu, nu', omega, omega'
 - 4 * x ** (2 - a - 2 * a - c - d) * (1 - x ** a) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * a - c)) # chi, chi', upsilon, upsilon'
 - 4 * x ** (2 - 2 * a - a - c - d) * (1 - x ** a) * (1 - x ** c) * (1 - x ** d - x ** (1 - 2 * a - 2 * a - c)), # psi, psi', phi, phi'
 15.9, 0.05, 1, 0.07, 0.11),
]

In [93]:
for i, (name, function, zero_start, a_start, b_start, c_start, d_start) in enumerate(Lemma_A_8_4_4):
    print(f"Evaluating {name}: {function} ...")
    MinPFRootApprox(function, starting_x = 1/zero_start, a_start = a_start, b_start = b_start, c_start = c_start, d_start = d_start, a_range = 0.01, b_range = 0.01, c_range = 0.01, d_range = 0.01)

Evaluating omega and chi meets gamma: -4*x**(2 - 4*a)*(1 - x**a)**2 - 4*x**(2 - 2*a)*(1 - x**c)*(1 - x**(-4*a - c + 1)) + (1 - x)*(1 - x**a)**2*(1 - x**c)*(1 - 2*x**(-4*a - c + 1)) ...
The minimum PF zero is 16.6734713400880 within margin of 0.01, attained around a=0.08999999999999998, b=1, c=0.1225, d=1 within margin of 0.001

Evaluating omega meets gamma, chi does not meet gamma: -4*x**(2 - 4*a)*(1 - x**a)**2 - 4*x**(2 - 3*a)*(1 - x**a)*(1 - x**c)*(1 - x**(-4*a - c + 1)) - 2*x**(2 - 2*a)*(1 - x**c)*(1 - x**(-4*a - c + 1)) + (1 - x)*(1 - x**a)**2*(1 - x**c)*(1 - 2*x**(-4*a - c + 1)) ...
The minimum PF zero is 15.4851853189814 within margin of 0.01, attained around a=0.08000000000000002, b=1, c=0.13124999999999998, d=1 within margin of 0.001

Evaluating omega and chi do not meet gamma: -4*x**(2 - 4*a)*(1 - x**a)**2 - 8*x**(-3*a - d + 2)*(1 - x**a)*(1 - x**c)*(1 - x**d) + (1 - x)*(1 - x**a)**2*(1 - x**c)*(-x**d - x**(-4*a - c + 1) + 1) ...
The minimum PF zero is 18.3092178054022 within 

In [94]:
Lemma_A_8_4_5 = [
("If there exists mu meeting beta and gamma and delta and eta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (2 - a - b), # mu, mu'
 14.5, 0.11, 0.11, 0.28),

# If there exists mu that meets beta but not gamma and delta and eta, and nu that meets gamma but not beta and delta and eta
# then there exists omega and chi meeting delta and filaments

# # If omega meets eta
    
("omega meets beta and gamma", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (1 + b) * (1 - x ** a) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 2 * x ** (2 - a - b), # omega, omega'
 18.5, 0.14, 0.14, 0.22),
    
("omega meets beta but not gamma", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (1 + b) * (1 - x ** a) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b), # omega, omega'
 15.1, 0.17, 0.11, 0.22),

# # We can now assume that omega does not meet beta and gamma
    
("chi meets eta", # then we can assume that chi does not beta and gamma
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (1 + b) * (1 - x ** a) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 4 * x ** (2 - 2 * a - 2 * b) * (1 - x ** a) * (1 - x ** b), # omega, omega', chi, chi'
 15.1, 0.125, 0.125, 0.25),

("chi does not meet eta", # then there exists upsilon meeting eta and filaments 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (1 + b) * (1 - x ** a) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 2 * x ** (2 - 2 * a - 2 * b) * (1 - x ** a) * (1 - x ** b) # omega, omega'
 - 2 * x ** (1 + a + b + c) * (1 - x ** a) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # chi, chi'
 - 2 * x ** (2 - a - b - c) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c), # upsilon, upsilon'
 14.9, 0.14, 0.14, 0.23),

# We can now assume that omega and chi do not meet eta
# then there exists upsilon and psi meeting eta and filaments
# We can assume that upsilon and psi do not meet delta
    
("omega meets beta and gamma",
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (1 + b) * (1 - x ** a) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 2 * x ** (1 + a + b + c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # omega, omega'
 - 2 * x ** (1 + a + b + c) * (1 - x ** a) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # chi, chi'
 - 2 * x ** (2 - a - b - c) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) # upsilon, upsilon'
 - 2 * x ** (2 - a - b - c) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c), # psi, psi'
 18, 0.17, 0.17, 0.22),

("omega meets beta but not gamma",
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (1 + b) * (1 - x ** a) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 2 * x ** (1 + a + c) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # omega, omega'
  - 2 * x ** (1 + a + b + c) * (1 - x ** a) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # chi, chi'
 - 2 * x ** (2 - a - b - c) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) # upsilon, upsilon'
 - 2 * x ** (2 - a - b - c) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c), # psi, psi'
 16.2, 0.19, 0.13, 0.22),

("omega and chi and upsilon and psi do not meet beta and gamma", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (1 + b) * (1 - x ** a) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 4 * x ** (1 + c) * (1 - x ** a) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # omega, omega', chi, chi'
 - 4 * x ** (2 - 2 * a - 2 * b - c) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c), # upsilon, upsilon', psi, psi'
 17.8, 0.13, 0.13, 0.25),

# If there exists mu that meets beta and gamma but not delta and eta
# then there exists omega and chi meeting delta and filaments

("omega meets eta, omega meets beta and gamma", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a + b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (2 - a - b), # omega, omega'
 18.3, 0.15, 0.15, 0.2),
    
("omega meets eta, omega meets beta but not gamma", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a + b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b), # omega, omega'
 15, 0.21, 0.1, 0.18),

# # We can now assume that omega does not meet beta and gamma
    
("omega meets eta, omega does not meet beta and gamma, chi meets eta", # then we can assume that chi does not beta and gamma
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a + b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 4 * x ** (2 - 2 * a - 2 * b) * (1 - x ** a) * (1 - x ** b), # omega, omega', chi, chi'
 15.7, 0.14, 0.14, 0.22),

("omega meets eta, omega does not meet beta and gamma, chi does not meet eta", # then there exists upsilon meeting eta and filaments
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a + b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (2 - 2 * a - 2 * b) * (1 - x ** a) * (1 - x ** b) # omega, omega'
 - 2 * x ** (1 + a + b + c) * (1 - x ** a) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # chi, chi'
 - 2 * x ** (2 - a - b - c) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c), # upsilon, upsilon'
 15.1, 0.15, 0.15, 0.2),

# We can now assume that omega and chi do not meet eta
# then there exists upsilon and psi meeting eta and filaments
# We can assume that upsilon and psi do not meet delta
    
("omega meets beta and gamma",
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a + b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (1 + a + b + c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # omega, omega'
 - 2 * x ** (1 + a + b + c) * (1 - x ** a) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # chi, chi'
 - 2 * x ** (2 - a - b - c) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) # upsilon, upsilon'
 - 2 * x ** (2 - a - b - c) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c), # psi, psi'
 17.3, 0.17, 0.17, 0.2),

("omega meets beta but not gamma",
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a + b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (1 + a + c) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # omega, omega'
- 2 * x ** (1 + a + b + c) * (1 - x ** a) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # chi, chi'
 - 2 * x ** (2 - a - b - c) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) # upsilon, upsilon'
 - 2 * x ** (2 - a - b - c) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c), # psi, psi'
 15.5, 0.23, 0.12, 0.18),

("omega and chi and upsilon and psi do not meet beta and gamma", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a + b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 4 * x ** (1 + c) * (1 - x ** a) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # omega, omega', chi, chi'
 - 4 * x ** (2 - 2 * a - 2 * b - c) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c), # upsilon, upsilon', psi, psi'
 17.9, 0.15, 0.15, 0.21),
]

In [95]:
for i, (name, function, zero_start, a_start, b_start, c_start) in enumerate(Lemma_A_8_4_5):
    print(f"Evaluating {name}: {function} ...")
    MinPFRootApprox(function, starting_x = 1/zero_start, a_start = a_start, b_start = b_start, c_start = c_start, a_range = 0.01, b_range = 0.01, c_range = 0.01)

Evaluating If there exists mu meeting beta and gamma and delta and eta: -2*x**(-a - b + 2) + (1 - x)*(1 - x**a)*(1 - x**b)*(1 - x**c)*(1 - x**(-2*a - 2*b - c + 1)) ...
The minimum PF zero is 14.5916691844557 within margin of 0.01, attained around a=0.11250000000000002, b=0.11250000000000002, c=0.2750000000000001, d=0.5 within margin of 0.001

Evaluating omega meets beta and gamma: -2*x**(a + 1)*(1 - x**b)*(1 - x**c)*(1 - x**(-2*a - 2*b - c + 1)) - 2*x**(b + 1)*(1 - x**a)*(1 - x**c)*(1 - x**(-2*a - 2*b - c + 1)) - 2*x**(-a - b + 2) + (1 - x)*(1 - x**a)*(1 - x**b)*(1 - x**c)*(1 - x**(-2*a - 2*b - c + 1)) ...
The minimum PF zero is 18.5596986623487 within margin of 0.01, attained around a=0.14, b=0.14, c=0.22, d=0.5 within margin of 0.001

Evaluating omega meets beta but not gamma: -2*x**(a + 1)*(1 - x**b)*(1 - x**c)*(1 - x**(-2*a - 2*b - c + 1)) - 2*x**(b + 1)*(1 - x**a)*(1 - x**c)*(1 - x**(-2*a - 2*b - c + 1)) - 2*x**(-a - 2*b + 2)*(1 - x**b) + (1 - x)*(1 - x**a)*(1 - x**b)*(1 - x**c)*(

In [96]:
Lemma_A_8_4_6 = [
    
# If there exists mu that meets beta but not gamma and delta and eta, nu that meets beta and delta and eta but not gamma
# then there exists omega and chi meeting gamma and filaments
# We can assume that omega and chi each meet one of delta and eta
    
("omega meets delta and eta", # then we can assume that omega does not meet beta
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # nu, nu'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a), # omega, omega'
 14.8, 0.12, 0.07, 0.31),

("omega meets delta but not eta, chi meets delta but not eta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # nu, nu'
 - 4 * x ** (1 + a + b + c) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * b - c)), # omega, omega', chi, chi'
 15, 0.15, 0.08, 0.41),

("omega meets delta but not eta, chi meets eta but not delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # nu, nu'
 - 2 * x ** (1 + a + b + c) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * b - c)) # omega, omega'
 - 2 * x ** (2 - a - b - c) * (1 - x ** a) * (1 - x ** c), # chi, chi'
 16.4, 0.15, 0.1, 0.25),
    
# If there exists mu that meets beta but not gamma and delta and eta, nu that meets beta and delta but not gamma and eta 
# then there exists omega and chi meeting gamma and filaments
# We can assume that omega and chi each meet one of delta and eta

("omega meets delta and eta", # then we can assume that omega does not meet beta
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (1 + a + c) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a), # omega, omega' 
 14.8, 0.12, 0.09, 0.37),

# # If omega meets delta but not eta, chi meets delta but not eta
# # then there exists upsilon and psi meeting eta and filaments

("upsilon and psi meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (1 + a + c) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 4 * x ** (1 + a + b + c) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * b - c)) # omega, omega', chi, chi'
 - 4 * x ** (2 - a - b) * (1 - x ** a) * (1 - x ** b), # upsilon, upsilon', psi, psi'
 14.7, 0.14, 0.08, 0.46),
    
("upsilon meets delta, psi does not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (1 + a + c) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 4 * x ** (1 + a + b + c) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * b - c)) # omega, omega', chi, chi'
 - 2 * x ** (2 - a - b) * (1 - x ** a) * (1 - x ** b) # upsilon, upsilon'
 - 2 * x ** (2 - a - b - c) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c), # psi, psi'
 15.5, 0.15, 0.09, 0.38),

("upsilon and psi do not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (1 + a + c) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 4 * x ** (1 + a + b + c) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * b - c)) # omega, omega', chi, chi'
 - 4 * x ** (2 - a - b - c) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c), # upsilon, upsilon', psi, psi'
 16, 0.16, 0.1, 0.33),

# #
    
("omega meets delta but not eta, chi meets eta but not delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (1 + a + c) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 2 * x ** (1 + a + b + c) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * b - c)) # omega, omega'
 - 2 * x ** (2 - a - b - c) * (1 - x ** a) * (1 - x ** c), # chi, chi'
 16, 0.17, 0.13, 0.24),

("omega meets eta but not delta, chi meets eta but not delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (1 + a + c) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 4 * x ** (2 - a - b - c) * (1 - x ** a) * (1 - x ** c), # omega, omega', chi, chi'
 16, 0.18, 0.11, 0.17),

# If there exists mu that meets beta but not gamma and delta and eta, nu that meets beta and gamma and delta but not eta
# then there exists omega and chi meeting gamma and filaments
# We can assume that omega and chi each meet one of delta and eta
    
("omega meets delta and eta", # then we can assume that omega does not meet beta
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (1 + a + b + c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a), # omega, omega'
 17.6, 0.12, 0.13, 0.36),

# # If omega meets delta but not eta, chi meets delta but not eta
# # then there exists upsilon and psi meeting eta and filaments

("upsilon meets delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (1 + a + b + c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 4 * x ** (1 + a + b + c) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * b - c)) # omega, omega', chi, chi'
 - 2 * x ** (2 - a - b) * (1 - x ** a) * (1 - x ** b) # upsilon, upsilon'
 - 2 * x ** (2 - a - b) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c), # psi, psi'
 16.5, 0.15, 0.12, 0.39),
    
("upsilon and psi do not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (1 + a + b + c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 4 * x ** (1 + a + b + c) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * b - c)) # omega, omega', chi, chi'
 - 4 * x ** (2 - a - b - c) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c), # upsilon, upsilon', chi, chi'
 17.9, 0.16, 0.14, 0.29),

# We can now assume that there are no curves that meet beta but not gamma and delta and eta
# similarly there are no curves that meet gamma but not beta and delta and eta
    
# If there exists mu that meets beta and gamma and delta but not eta, nu that meets beta and gamma and delta but not eta
# then there exists omega and chi meeting eta and filaments

("omega meets beta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 4 * x ** (1 + a + b + c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu', nu, nu'
 - 2 * x ** (2 - a - b) * (1 - x ** b) * (1 - x ** c), # omega, omega'
 15.7, 0.16, 0.12, 0.36),

("omega and chi do not meet beta and gamma", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 4 * x ** (1 + a + b + c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu', nu, nu'
 - 4 * x ** (2 - 2 * a - 2 * b) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c), # omega, omega', chi, chi'
 16.2, 0.12, 0.12, 0.44),
    
# If there exists mu that meets beta and gamma and delta but not eta, nu that meets beta and gamma and eta but not delta
# then there exists omega and chi meeting eta and filaments

("omega meets beta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a + b + c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (2 - a - b - c) * (1 - x ** c) # nu, nu'
 - 2 * x ** (2 - a - b) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # omega, omega'
 - 2 * x ** (2 - a - b) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c), # chi, chi'
 19, 0.17, 0.15, 0.19),

("omega and chi do not meet beta and gamma", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a + b + c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (2 - a - b - c) * (1 - x ** c) # nu, nu'
 - 2 * x ** (2 - 2 * a - 2 * b) * (1 - x ** a) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # omega, omega'
 - 2 * x ** (2 - 2 * a - 2 * b) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c), # chi, chi'
 19.4, 0.15, 0.15, 0.2),

# If there exists mu that meets beta and gamma and delta but not eta, nu that meets beta and delta and eta but not gamma
# then there exists omega meeting gamma and filaments, we can assume that omega does not meet beta

("omega meets delta and eta",
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a + b + c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # nu, nu'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a), # omega, omega'
 17, 0.1, 0.1, 0.42),

("omega meets eta but not delta",
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a + b + c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # nu, nu'
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c), # omega, omega'
 18.3, 0.12, 0.13, 0.3),

("omega meets delta but not eta", # then there exists chi exiting eta at the same vertex as nu
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a + b + c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # nu, nu'
 - 2 * x ** (1 + b + c) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * b - c)) # omega, omega'
 - 2 * x ** (2 - a - b) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c), # chi, chi'
 16.1, 0.12, 0.09, 0.45),
    
# If there exists mu that meets beta and gamma and delta but not eta, nu that meets beta and delta but not gamma and eta
# then there exists omega meeting gamma and filaments, we can assume that omega does not meet beta
    
("omega meets delta and eta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a + b + c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (1 + a + c) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a), # omega, omega'
 15.8, 0.09, 0.13, 0.45),

("omega meets eta but not delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a + b + c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (1 + a + c) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c), # omega, omega'
 17.5, 0.12, 0.16, 0.3),

("omega meets delta but not eta", # then there exists chi and upsilon meeting eta and filaments
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a + b + c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (1 + a + c) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 2 * x ** (1 + b + c) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * b - c)) # omega, omega'
 - 4 * x ** (2 - a - b) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c), # chi, chi', upsilon, upsilon'
 14.5, 0.11, 0.11, 0.48),
    
# If there exists mu that meets beta and gamma and delta but not eta, nu that meets beta and eta but not gamma and delta
# then there exists omega leaving delta at the same vertex as mu

("omega meets eta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a + b + c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # nu, nu'
 - 2 * x ** (2 - 2 * a - 2 * b) * (1 - x ** a) * (1 - x ** b), # omega, omega'
 16.7, 0.16, 0.09, 0.31),
    
("omega meets beta", # then there exists chi leaving eta at the same vertex as mu 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a + b + c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # nu, nu'
 - 2 * x ** (2 - a - b) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # omega, omega'
 - 2 * x ** (2 - a - b) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c), # chi, chi'
 16.3, 0.19, 0.1, 0.27),

("omega meets gamma", # then there exists chi leaving eta at the same vertex as mu 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a + b + c) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # nu, nu'
 - 2 * x ** (2 - a - b) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * b - c)) # omega, omega'
 - 2 * x ** (2 - a - b) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c), # chi, chi'
 16.7, 0.17, 0.11, 0.28),

("omega does not meet beta and gamma and eta", # then there exists chi leaving eta at the same vertex as mu
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (1 + a + b + c) * (1 - x ** (1 - 2 * a - 2 * b - c))
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c)
 - 2 * x ** (1 + c) * (1 - x ** a) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c))
 - 2 * x ** (2 - 2 * a - 2 * b) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c), 16.9, 0.17, 0.09, 0.31),
]

In [97]:
for i, (name, function, zero_start, a_start, b_start, c_start) in enumerate(Lemma_A_8_4_6):
    print(f"Evaluating {name}: {function} ...")
    MinPFRootApprox(function, starting_x = 1/zero_start, a_start = a_start, b_start = b_start, c_start = c_start, a_range = 0.01, b_range = 0.01, c_range = 0.01)

Evaluating omega meets delta and eta: -2*x**(a + 1)*(1 - x**b)*(1 - x**c)*(1 - x**(-2*a - 2*b - c + 1)) - 2*x**(-2*a - b + 2)*(1 - x**a) - 2*x**(-a - 2*b + 2)*(1 - x**b) + (1 - x)*(1 - x**a)*(1 - x**b)*(1 - x**c)*(1 - x**(-2*a - 2*b - c + 1)) ...
The minimum PF zero is 14.8763822420696 within margin of 0.01, attained around a=0.11937500000000002, b=0.07312500000000001, c=0.3075000000000001, d=0.5 within margin of 0.001

Evaluating omega meets delta but not eta, chi meets delta but not eta: -2*x**(a + 1)*(1 - x**b)*(1 - x**c)*(1 - x**(-2*a - 2*b - c + 1)) - 2*x**(-a - 2*b + 2)*(1 - x**b) - 4*x**(a + b + c + 1)*(1 - x**a)*(1 - x**(-2*a - 2*b - c + 1)) + (1 - x)*(1 - x**a)*(1 - x**b)*(1 - x**c)*(1 - x**(-2*a - 2*b - c + 1)) ...
The minimum PF zero is 15.0162916891870 within margin of 0.01, attained around a=0.15375000000000003, b=0.07625000000000001, c=0.4100000000000001, d=0.5 within margin of 0.001

Evaluating omega meets delta but not eta, chi meets eta but not delta: -2*x**(a + 1)*(1 

In [98]:
Lemma_A_8_4_7 = [

# Let mu and nu be curves meeting beta and filaments
# By the analysis above, the remaining options for mu and nu are: 
# (meet delta and eta but not gamma, meet delta and eta but not gamma),
# (meet delta and eta but not gamma, meet delta but not gamma and eta),
# (meet delta but not gamma and eta, meet eta but not gamma and delta),
# (meet delta but not gamma and eta, meet delta but not gamma and eta)

# If mu meets delta and eta but not gamma, nu meets delta and eta but not gamma
# there exists omega and chi meeting gamma and filaments, we can assume omega and chi do not meet beta
# We can assume that omega and chi each meet one of delta and eta
    
("omega meets delta and eta, chi meets delta and eta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 4 * x ** (2 - a - 2 * b) * (1 - x ** b) # mu, mu', nu, nu'
 - 4 * x ** (2 - 2 * a - b) * (1 - x ** a), # omega, omega', chi, chi'
 14.9, 0.06, 0.06, 0.37),

("omega meets delta and eta, chi meets delta but not eta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 4 * x ** (2 - a - 2 * b) * (1 - x ** b) # mu, mu', nu, nu'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) # omega, omega'
 - 2 * x ** (1 + b + c) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * b - c)), # chi, chi'
 15.5, 0.07, 0.07, 0.45),

("omega meets delta but not eta, chi meets eta but not delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 4 * x ** (2 - a - 2 * b) * (1 - x ** b) # mu, mu', nu, nu'
 - 2 * x ** (1 + b + c) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * b - c)) # omega, omega'
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c), # chi, chi'
 16.7, 0.08, 0.09, 0.33),

("omega meets delta but not eta, chi meets delta but not delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 4 * x ** (2 - a - 2 * b) * (1 - x ** b) # mu, mu', nu, nu'
 - 4 * x ** (1 + b + c) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * b - c)), # omega, omega', chi, chi'
 15.2, 0.08, 0.07, 0.51),

# If mu meets delta and eta but not gamma, nu meets delta but not gamma and eta
# there exists omega and chi meeting gamma and filaments, we can assume omega and chi do not meet beta
# We can assume that omega and chi each meet one of delta and eta
# We can assume that omega and chi do not both meet delta and eta

("omega meets delta and eta, chi meets delta but not eta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # mu, mu'
 - 2 * x ** (1 + a + c) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) # omega, omega'
 - 2 * x ** (1 + b + c) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * b - c)), # chi, chi'
 15.3, 0.08, 0.08, 0.5),

("omega meets delta and eta, chi meets eta but not delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # mu, mu'
 - 2 * x ** (1 + a + c) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) # omega, omega'
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c), # chi, chi'
 16.8, 0.09, 0.09, 0.33),

("omega meets delta but not eta, chi meets eta but not delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # mu, mu'
 - 2 * x ** (1 + a + c) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 2 * x ** (1 + b + c) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * b - c)) # omega, omega'
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c), # chi, chi'
 17.2, 0.09, 0.1, 0.37),

("omega meets delta but not eta, chi meets delta but not eta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # mu, mu'
 - 2 * x ** (1 + a + c) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 4 * x ** (1 + b + c) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * b - c)) # omega, omega'
 - 2 * x ** (2 - a - b) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c), # chi, chi'
 14.6, 0.09, 0.07, 0.55),
    
("omega meets eta but not delta, chi meets eta but not delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c)) 
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # mu, mu'
 - 2 * x ** (1 + a + c) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 4 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c), # omega, omega', chi, chi'
 17.2, 0.1, 0.09, 0.25),

# If mu meets delta but not eta, nu meets eta but not delta
# there exists omega and chi meeting gamma and filaments, we can assume omega and chi do not meet beta
# We can assume that omega and chi each meet one of delta and eta
# We can assume that omega and chi each do not meet both delta and eta

("omega meets delta but not eta, chi meets eta but not delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c))
 - 2 * x ** (1 + a + c) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # nu, nu'
 - 2 * x ** (1 + b + c) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * b - c)) # omega, omega'
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c), # chi, chi'
 18, 0.11, 0.11, 0.28),
    
("omega meets delta but not eta, chi meets delta but not eta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c))
 - 2 * x ** (1 + a + c) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # nu, nu'
 - 4 * x ** (1 + b + c) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * b - c)), # omega, omega', chi, chi'
 17, 0.13, 0.09, 0.37),

# If mu meets delta but not eta, nu meets delta but not eta
# there exists omega and chi meeting gamma and filaments, we can assume omega and chi do not meet beta
# We can assume that omega and chi each meet one of delta and eta
# We can assume that omega and chi each do not meet both delta and eta

("omega meets eta but not delta, chi meets eta but not delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c))
 - 4 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu', nu, nu'
 - 4 * x ** (1 + b + c) * (1 - x ** a) * (1 - x ** c), # omega, omega', chi, chi'
 18, 0.11, 0.11, 0.28),

# We can assume that omega meets delta but not eta, chi meets delta but not eta
# then there exists upsilon and psi meeting eta and filaments
    
("upsilon and psi do not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c))
 - 4 * x ** (1 + a + c) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu', nu, nu'
 - 4 * x ** (1 + b + c) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * b - c)) # omega, omega', chi, chi'
 - 4 * x ** (2 - 2 * a - 2 * b - c) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c), # upsilon, upsilon', psi, psi'
 17, 0.08, 0.08, 0.51),
    
("upsilon meets delta, psi does not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c))
 - 4 * x ** (1 + a + c) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu', nu, nu'
 - 4 * x ** (1 + b + c) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * b - c)) # omega, omega', chi, chi'
 - 2 * x ** (2 - 2 * a - 2 * b) * (1 - x ** a) * (1 - x ** b) # upsilon, upsilon'
 - 2 * x ** (2 - 2 * a - 2 * b - c) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c), # psi, psi'
 15.7, 0.07, 0.07, 0.56),
]

In [99]:
for i, (name, function, zero_start, a_start, b_start, c_start) in enumerate(Lemma_A_8_4_7):
    print(f"Evaluating {name}: {function} ...")
    MinPFRootApprox(function, starting_x = 1/zero_start, a_start = a_start, b_start = b_start, c_start = c_start, a_range = 0.01, b_range = 0.01, c_range = 0.01)

Evaluating omega meets delta and eta, chi meets delta and eta: -4*x**(-2*a - b + 2)*(1 - x**a) - 4*x**(-a - 2*b + 2)*(1 - x**b) + (1 - x)*(1 - x**a)*(1 - x**b)*(1 - x**c)*(1 - x**(-2*a - 2*b - c + 1)) ...
The minimum PF zero is 14.9950161002401 within margin of 0.01, attained around a=0.06375, b=0.06375, c=0.3725000000000001, d=0.5 within margin of 0.001

Evaluating omega meets delta and eta, chi meets delta but not eta: -2*x**(-2*a - b + 2)*(1 - x**a) - 4*x**(-a - 2*b + 2)*(1 - x**b) - 2*x**(b + c + 1)*(1 - x**a)*(1 - x**(-2*a - 2*b - c + 1)) + (1 - x)*(1 - x**a)*(1 - x**b)*(1 - x**c)*(1 - x**(-2*a - 2*b - c + 1)) ...
The minimum PF zero is 15.5098343940618 within margin of 0.01, attained around a=0.07, b=0.07187500000000001, c=0.4475000000000001, d=0.5 within margin of 0.001

Evaluating omega meets delta but not eta, chi meets eta but not delta: -4*x**(-a - 2*b + 2)*(1 - x**b) - 2*x**(b + c + 1)*(1 - x**a)*(1 - x**(-2*a - 2*b - c + 1)) - 2*x**(-2*a - b - c + 2)*(1 - x**a)*(1 - x**c) 

In [100]:
Lemma_A_8_4_8 = [
    
# Remaining case: mu and nu meet beta and delta but not gamma and eta, omega and chi meet gamma and delta but not beta and eta, upsilon and psi meet delta and eta but not beta and gamma
    
("there exists phi other than upsilon and psi meeting eta", # then we can assume that phi meet delta and eta but not beta and gamma
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c))
 - 4 * x ** (1 + a + c) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu', nu, nu'
 - 4 * x ** (1 + b + c) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * b - c)) # omega, omega', chi, chi'
 - 6 * x ** (2 - 2 * a - 2 * b) * (1 - x ** a) * (1 - x ** b), # upsilon, upsilon', psi, psi', phi, phi'
 14.5, 0.06, 0.06, 0.66, 1),

# Thus there is only one pair of edges exiting eta, leading to upsilon and psi
# By Proposition A.2.1, if the pair of edges exiting eta enters a filament, equivalently if upsilon and psi are oriented from delta to eta,
# then there is a curve zeta that only passes through filaments (and it has a double)
    
("mu and nu and omega and chi meet zeta", 
 (1 - 3 * x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c))
 - 4 * x ** (1 + a + c) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu', nu, nu'
 - 4 * x ** (1 + b + c) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * b - c)) # omega, omega', chi, chi'
 - 4 * x ** (2 - 2 * a - 2 * b) * (1 - x ** a) * (1 - x ** b), # upsilon, upsilon', psi, psi'
 14.8, 0.06, 0.06, 0.66, 1),

("mu does not meet zeta", 
 (1 - 2 * x ** d - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c))
 - 2 * x ** (1 + a + c - d) * (1 - 2 * x ** d) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu'
 - 2 * x ** (1 + a + c) * (1 - 2 * x ** d) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # nu, nu'
 - 4 * x ** (1 + b + c) * (1 - 2 * x ** d) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * b - c)) # omega, omega', chi, chi'
 - 4 * x ** (2 - 2 * a - 2 * b) * (1 - x ** a) * (1 - x ** b), # upsilon, upsilon', psi, psi'
 18.1, 0.07, 0.05, 0.67, 0.49),

# Thus we can assume that upsilon and psi are oriented from eta to delta
# The pair of edges exiting delta that is used to construct upsilon and psi must land in different filament curves, 
# otherwise there are more curves passing through eta
# In particular there is a filament curve zeta disjoint from upsilon and psi
# We divide into cases depending on whether mu and nu and omega and chi meet these filament curves

("mu and nu and omega and chi meet zeta", 
 (1 - x ** d) * (1 - x ** (1 - d)) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c))
 - 4 * x ** (1 + a + c) * (1 - x ** (1 - d)) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu', nu, nu'
 - 4 * x ** (1 + b + c) * (1 - x ** (1 - d)) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * b - c)) # omega, omega', chi, chi'
 - 4 * x ** (2 - 2 * a - 2 * b - d) * (1 - x ** d) * (1 - x ** a) * (1 - x ** b), # upsilon, upsilon', psi, psi'
 18, 0.05, 0.05, 0.67, 0.35),

("mu and nu meet zeta, omega and chi do not meet zeta", 
 (1 - x ** d) * (1 - x ** (1 - d)) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c))
 - 4 * x ** (1 + a + c) * (1 - x ** (1 - d)) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu', nu, nu'
 - 4 * x ** (1 + b + c - d) * (1 - x ** d) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * b - c)) # omega, omega', chi, chi'
 - 4 * x ** (2 - 2 * a - 2 * b - d) * (1 - x ** d) * (1 - x ** a) * (1 - x ** b), # upsilon, upsilon', psi, psi'
 18.6, 0.067, 0.056, 0.676, 0.169),

("mu and nu and omega and chi do not meet zeta", 
 (1 - x ** d - x ** (1 - d)) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c) * (1 - x ** (1 - 2 * a - 2 * b - c))
 - 4 * x ** (1 + a + c - d) * (1 - x ** d) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b - c)) # mu, mu', nu, nu'
 - 4 * x ** (1 + b + c - d) * (1 - x ** d) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * b - c)) # omega, omega', chi, chi'
 - 4 * x ** (2 - 2 * a - 2 * b - d) * (1 - x ** d) * (1 - x ** a) * (1 - x ** b), # upsilon, upsilon', psi, psi'
 17.9, 0.06, 0.06, 0.674, 0.094),
]

In [101]:
for i, (name, function, zero_start, a_start, b_start, c_start, d_start) in enumerate(Lemma_A_8_4_8):
    print(f"Evaluating {name}: {function} ...")
    MinPFRootApprox(function, starting_x = 1/zero_start, a_start = a_start, b_start = b_start, c_start = c_start, d_start = d_start, a_range = 0.005, b_range = 0.005, c_range = 0.005, d_range = 0.01)

Evaluating there exists phi other than upsilon and psi meeting eta: -6*x**(-2*a - 2*b + 2)*(1 - x**a)*(1 - x**b) - 4*x**(a + c + 1)*(1 - x**b)*(1 - x**(-2*a - 2*b - c + 1)) - 4*x**(b + c + 1)*(1 - x**a)*(1 - x**(-2*a - 2*b - c + 1)) + (1 - x)*(1 - x**a)*(1 - x**b)*(1 - x**c)*(1 - x**(-2*a - 2*b - c + 1)) ...
The minimum PF zero is 14.5156958170930 within margin of 0.01, attained around a=0.059687500000000004, b=0.059687500000000004, c=0.6584374999999998, d=1 within margin of 0.001

Evaluating mu and nu and omega and chi meet zeta: -4*x**(-2*a - 2*b + 2)*(1 - x**a)*(1 - x**b) - 4*x**(a + c + 1)*(1 - x**b)*(1 - x**(-2*a - 2*b - c + 1)) - 4*x**(b + c + 1)*(1 - x**a)*(1 - x**(-2*a - 2*b - c + 1)) + (1 - 3*x)*(1 - x**a)*(1 - x**b)*(1 - x**c)*(1 - x**(-2*a - 2*b - c + 1)) ...
The minimum PF zero is 14.8123497740230 within margin of 0.01, attained around a=0.0625, b=0.0625, c=0.6618749999999999, d=1 within margin of 0.001

Evaluating mu does not meet zeta: -4*x**(-2*a - 2*b + 2)*(1 - x**a)*(1

In [102]:
Lemma_A_8_5_1 = [

# If there exists mu meeting beta and gamma and eta
# there exists nu exiting beta from the same vertex as mu

# # If nu meets gamma
    
("nu meets eta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - 2 * x ** (1 - 2 * a - 2 * b))
 - 4 * x ** (2 - a - b), # mu, mu', nu, nu'
 14.5, 0.11, 0.11, 1),

("nu meets delta but not eta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - b) # mu, mu'
 - 2 * x ** (2 - a - b - c) * (1 - x ** c), # nu, nu'
 18, 0.13, 0.13, 0.33),

("nu does not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - 2 * x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - b) # mu, mu'
 - 2 * x ** (1 + a + b) * (1 - 2 * x ** (1 - 2 * a - 2 * b)), # nu, nu'
 15.9, 0.15, 0.15, 1),

# # If nu does not meet gamma   
# # there exists omega exiting gamma from the same vertex as mu
# # We can assume omega does not meet beta
    
("nu meets eta, omega meets delta but not eta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - b) # mu, mu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # nu, nu'
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c), # omega, omega' 
 15.8, 0.09, 0.12, 0.48),

("nu meets eta, omega does not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - 2 * x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - b) # mu, mu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # nu, nu'
 - 2 * x ** (1 + b) * (1 - x ** a) * (1 - 2 * x ** (1 - 2 * a - 2 * b)), # omega, omega' 
 15, 0.11, 0.13, 1),

("nu meets delta but not eta, omega meets delta but not eta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - b) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # nu, nu'
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c), # omega, omega' 
 17, 0.1, 0.1, 0.35),

("nu meets delta but not eta, omega does not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - b) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # nu, nu'
 - 2 * x ** (1 + b) * (1 - x ** a) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)), # omega, omega' 
 17.1, 0.13, 0.12, 0.41),

("nu does not meet delta, omega does not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - 2 * x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - b) # mu, mu'
 - 2 * x ** (1 + a) * (1 - x ** b) * (1 - 2 * x ** (1 - 2 * a - 2 * b)) # nu, nu'
 - 2 * x ** (1 + b) * (1 - x ** a) * (1 - 2 * x ** (1 - 2 * a - 2 * b)), # omega, omega' 
 15.9, 0.14, 0.14, 1),
    
# If there exists mu meeting beta and gamma and delta but not eta
# there exists nu exiting beta from the same vertex as mu

("nu meets gamma and delta but not eta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - b - c) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - b - c) * (1 - x ** c) # nu, nu'
 - 2 * x ** (2 - a - b) * (1 - x ** a) * (1 - x ** b), # omega, omega' 
 18.7, 0.11, 0.11, 0.21),

("nu meets gamma but not delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - b - c) * (1 - x ** c) # mu, mu'
 - 2 * x ** (1 + a + b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)) # nu, nu'
 - 2 * x ** (2 - a - b) * (1 - x ** a) * (1 - x ** b), # omega, omega' 
 18.8, 0.15, 0.15, 0.29),

# We can now assume that nu does not meet gamma
# then there exists omega exiting gamma at the same vertex as mu

("nu meets eta, omega meets delta but not eta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - b - c) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # nu, nu'
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c), # omega, omega' 
 17.6, 0.1, 0.1, 0.24),

("nu meets eta, omega does not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - b - c) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # nu, nu'
 - 2 * x ** (1 + b) * (1 - x ** a) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)), # omega, omega' 
 18.3, 0.12, 0.13, 0.29),

# We can now assume that omega does not meet eta
# then there exists chi exiting delta at the same vertex as eta

("nu meets delta but not eta, omega meets delta but not eta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - b - c) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # nu, nu'
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) # omega, omega' 
 - 2 * x ** (2 - a - b) * (1 - x ** a) * (1 - x ** b), # chi, chi'
 17.7, 0.09, 0.09, 0.19),

("nu meets delta but not eta, omega does not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - b - c) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # nu, nu'
 - 2 * x ** (1 + b) * (1 - x ** a) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)) # omega, omega' 
 - 2 * x ** (2 - a - b) * (1 - x ** a) * (1 - x ** b), # chi, chi'
 18.7, 0.11, 0.12, 0.22),

("nu does not meet delta, omega does not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - b - c) * (1 - x ** c) # mu, mu'
 - 2 * x ** (1 + a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)) # nu, nu'
 - 2 * x ** (1 + b) * (1 - x ** a) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)) # omega, omega' 
 - 2 * x ** (2 - a - b) * (1 - x ** a) * (1 - x ** b), # chi, chi'
 18.9, 0.14, 0.14, 0.26),

# If there exists mu meeting beta and gamma but not delta
# there exists nu exiting beta from the same vertex as mu

("Lemma 12.5 - nu meets gamma but not delta", # then there exists omega exiting delta at the same vertex as eta
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - b - c) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)) # mu, mu'
 - 2 * x ** (1 + a + b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)) # nu, nu'
 - 2 * x ** (2 - a - b) * (1 - x ** a) * (1 - x ** b), # omega, omega'
 14.8, 0.18, 0.18, 0.35),

# We can now assume that nu does not meet gamma
# then there exists omega exiting gamma at the same vertex as mu

("Lemma 12.5 - nu meets eta, omega meets delta but not eta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - b - c) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)) # mu, mu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # nu, nu'
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c), # omega, omega'
 15.9, 0.11, 0.11, 0.24),

("Lemma 12.5 - nu meets eta, omega does not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - b - c) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)) # mu, mu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # nu, nu'
 - 2 * x ** (1 + b) * (1 - x ** a) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)), # omega, omega'
 15.8, 0.15, 0.14, 0.33),

# We can now assume that omega does not meet eta
# then there exists chi exiting delta at the same vertex as eta

("Lemma 12.5 - nu meets delta but not eta, omega meets delta but not eta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - b - c) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # nu, nu'
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) # omega, omega'
 - 2 * x ** (2 - a - b) * (1 - x ** a) * (1 - x ** b), # chi, chi'
 15.8, 0.1, 0.1, 0.17),

("Lemma 12.5 - nu meets delta but not eta, omega does not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - b - c) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # nu, nu'
 - 2 * x ** (1 + b) * (1 - x ** a) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)) # omega, omega'
 - 2 * x ** (2 - a - b) * (1 - x ** a) * (1 - x ** b), # chi, chi' 
 16.6, 0.12, 0.12, 0.21),

("Lemma 12.5 - nu does not meet delta, omega does not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - b - c) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)) # mu, mu'
 - 2 * x ** (1 + a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)) # nu, nu'
 - 2 * x ** (1 + b) * (1 - x ** a) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)) # omega, omega'
 - 2 * x ** (2 - a - b) * (1 - x ** a) * (1 - x ** b), # chi, chi' 
 15.4, 0.17, 0.17, 0.31),
]

In [103]:
for i, (name, function, zero_start, a_start, b_start, c_start) in enumerate(Lemma_A_8_5_1):
    print(f"Evaluating {name}: {function} ...")
    MinPFRootApprox(function, starting_x = 1/zero_start, a_start = a_start, b_start = b_start, c_start = c_start, a_range = 0.01, b_range = 0.01)

Evaluating nu meets eta: -4*x**(-a - b + 2) + (1 - x)*(1 - x**a)*(1 - x**b)*(1 - 2*x**(-2*a - 2*b + 1)) ...
The minimum PF zero is 14.6036502075974 within margin of 0.01, attained around a=0.11500000000000002, b=0.11500000000000002, c=1, d=0.5 within margin of 0.001

Evaluating nu meets delta but not eta: -2*x**(-a - b + 2) - 2*x**(-a - b - c + 2)*(1 - x**c) + (1 - x)*(1 - x**a)*(1 - x**b)*(-x**c - x**(-2*a - 2*b + 1) + 1) ...
The minimum PF zero is 18.0153566367178 within margin of 0.01, attained around a=0.1265625, b=0.1265625, c=0.3346875, d=0.5 within margin of 0.001

Evaluating nu does not meet delta: -2*x**(-a - b + 2) - 2*x**(a + b + 1)*(1 - 2*x**(-2*a - 2*b + 1)) + (1 - x)*(1 - x**a)*(1 - x**b)*(1 - 2*x**(-2*a - 2*b + 1)) ...
The minimum PF zero is 15.9197893797854 within margin of 0.01, attained around a=0.15, b=0.15, c=1, d=0.5 within margin of 0.001

Evaluating nu meets eta, omega meets delta but not eta: -2*x**(-a - 2*b + 2)*(1 - x**b) - 2*x**(-a - b + 2) - 2*x**(-2*a - b -

In [104]:
Lemma_A_8_5_2 = [

# There exists curves mu and nu meeting beta and filaments
# By the analysis above, we can assume mu and nu do not meet gamma
# The remaining options for curves mu and nu meeting beta are: 
# (meets eta, meets eta), (meets eta, meets delta but not eta), (meets eta, does not meet delta), (meets delta but not eta, meets delta but not eta), (meets delta but not eta, does not meet delta), (does not meet delta, does not meet delta)

# Similarly, there exists omega and chi meeting gamma and filaments
# We can assume omega and chi do not meet beta
# Omega and chi have the same remaining options as for mu and nu
    
# If mu meets eta, nu meets eta
    
("omega meets delta but not eta, chi meets delta but not eta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # mu, mu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # nu, nu'
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) # omega, omega'
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c), # chi, chi'
 15.2, 0.07, 0.08, 0.33),

("omega meets delta but not eta, chi does not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # mu, mu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # nu, nu'
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) # omega, omega'
 - 2 * x ** (1 + b) * (1 - x ** a) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)), # chi, chi'
 16.1, 0.08, 0.13, 0.44),

("omega does not meet delta, chi does not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - 2 * x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # mu, mu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # nu, nu'
 - 2 * x ** (1 + b) * (1 - x ** a) * (1 - 2 * x ** (1 - 2 * a - 2 * b)) # omega, omega'
 - 2 * x ** (1 + b) * (1 - x ** a) * (1 - 2 * x ** (1 - 2 * a - 2 * b)), # chi, chi'
 15.4, 0.1, 0.15, 1),

# If mu meets eta, nu meets delta but not eta
    
("omega meets delta but not eta, chi meets delta but not eta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # nu, nu'
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) # omega, omega'
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c), # chi, chi'
 15.5, 0.07, 0.07, 0.24),

("omega meets delta but not eta, chi does not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # nu, nu'
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) # omega, omega'
 - 2 * x ** (1 + b) * (1 - x ** a) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)), # chi, chi'
 17.2, 0.09, 0.11, 0.28),

("Lemma 12.5 - mu meets eta, nu meets delta but not eta, omega does not meet delta, chi does not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # nu, nu'
 - 2 * x ** (1 + b) * (1 - x ** a) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)) # omega, omega'
 - 2 * x ** (1 + b) * (1 - x ** a) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)), # chi, chi'
 17.7, 0.12, 0.13, 0.34),

# If mu meets eta, nu does not meet delta
    
("omega meets delta but not eta, chi meets delta but not eta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # mu, mu'
 - 2 * x ** (1 + a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)) # nu, nu'
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) # omega, omega'
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c), # chi, chi'
 17.2, 0.11, 0.09, 0.26),

("omega meets delta but not eta, chi does not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # mu, mu'
 - 2 * x ** (1 + a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)) # nu, nu'
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) # omega, omega'
 - 2 * x ** (1 + b) * (1 - x ** a) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)), # chi, chi'
 17.6, 0.12, 0.14, 0.36),

("omega does not meet delta, chi does not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - 2 * x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # mu, mu'
 - 2 * x ** (1 + a) * (1 - x ** b) * (1 - 2 * x ** (1 - 2 * a - 2 * b)) # nu, nu'
 - 2 * x ** (1 + b) * (1 - x ** a) * (1 - 2 * x ** (1 - 2 * a - 2 * b)) # omega, omega'
 - 2 * x ** (1 + b) * (1 - x ** a) * (1 - 2 * x ** (1 - 2 * a - 2 * b)), # chi, chi'
 16.2, 0.15, 0.15, 1),

# We can now assume that mu and nu and omega and chi do not meet eta
# then there exists upsilon leaving delta at the same vertex as eta
    
# If mu meets delta but not eta, nu meets delta but not eta
    
("omega meets delta but not eta, chi meets delta but not eta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # nu, nu'
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) # omega, omega'
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) # chi, chi'
 - 2 * x ** (2 - 2 * a - 2 * b) * (1 - x ** a) * (1 - x ** b), # upsilon, upsilon'
 15.4, 0.06, 0.06, 0.18),

("omega meets delta but not eta, chi does not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # nu, nu'
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) # omega, omega'
 - 2 * x ** (1 + b) * (1 - x ** a) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)) # chi, chi'
 - 2 * x ** (2 - 2 * a - 2 * b) * (1 - x ** a) * (1 - x ** b), # upsilon, upsilon'
 17.6, 0.08, 0.1, 0.2),

("omega does not meet delta, chi does not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # mu, mu'
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # nu, nu'
 - 2 * x ** (1 + b) * (1 - x ** a) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)) # omega, omega'
 - 2 * x ** (1 + b) * (1 - x ** a) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)) # chi, chi'
 - 2 * x ** (2 - 2 * a - 2 * b) * (1 - x ** a) * (1 - x ** b), # upsilon, upsilon'
 18.9, 0.1, 0.12, 0.23),

# If mu meets delta but not eta, nu does not meet delta
    
("omega meets delta but not eta, chi does not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # mu, mu'
 - 2 * x ** (1 + a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)) # nu, nu'
 - 2 * x ** (2 - 2 * a - b - c) * (1 - x ** a) * (1 - x ** c) # omega, omega'
 - 2 * x ** (1 + b) * (1 - x ** a) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)) # chi, chi'
 - 2 * x ** (2 - 2 * a - 2 * b) * (1 - x ** a) * (1 - x ** b), # upsilon, upsilon'
 19, 0.12, 0.12, 0.24),

("omega does not meet delta, chi does not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - 2 * b - c) * (1 - x ** b) * (1 - x ** c) # mu, mu'
 - 2 * x ** (1 + a) * (1 - x ** b) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)) # nu, nu'
 - 2 * x ** (1 + b) * (1 - x ** a) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)) # omega, omega'
 - 2 * x ** (1 + b) * (1 - x ** a) * (1 - x ** c - x ** (1 - 2 * a - 2 * b)) # chi, chi'
 - 2 * x ** (2 - 2 * a - 2 * b) * (1 - x ** a) * (1 - x ** b), # upsilon, upsilon'
 19.2, 0.15, 0.14, 0.31),

# If mu does not meet delta, nu does not meet delta
    
("omega does not meet delta, chi does not meet delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - 2 * x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (1 + a) * (1 - x ** b) * (1 - 2 * x ** (1 - 2 * a - 2 * b)) # mu, mu'
 - 2 * x ** (1 + a) * (1 - x ** b) * (1 - 2 * x ** (1 - 2 * a - 2 * b)) # nu, nu'
 - 2 * x ** (1 + b) * (1 - x ** a) * (1 - 2 * x ** (1 - 2 * a - 2 * b)) # omega, omega'
 - 2 * x ** (1 + b) * (1 - x ** a) * (1 - 2 * x ** (1 - 2 * a - 2 * b)) # chi, chi'
 - 2 * x ** (2 - 2 * a - 2 * b) * (1 - x ** a) * (1 - x ** b), # upsilon, upsilon'
 17.3, 0.16, 0.16, 1),

]

In [105]:
for i, (name, function, zero_start, a_start, b_start, c_start) in enumerate(Lemma_A_8_5_2):
    print(f"Evaluating {name}: {function} ...")
    MinPFRootApprox(function, starting_x = 1/zero_start, a_start = a_start, b_start = b_start, c_start = c_start, a_range = 0.01, b_range = 0.01)

Evaluating omega meets delta but not eta, chi meets delta but not eta: -4*x**(-a - 2*b + 2)*(1 - x**b) - 4*x**(-2*a - b - c + 2)*(1 - x**a)*(1 - x**c) + (1 - x)*(1 - x**a)*(1 - x**b)*(-x**c - x**(-2*a - 2*b + 1) + 1) ...
The minimum PF zero is 15.2902189351949 within margin of 0.01, attained around a=0.06875000000000002, b=0.08316406250000002, c=0.33078124999999997, d=0.5 within margin of 0.001

Evaluating omega meets delta but not eta, chi does not meet delta: -2*x**(b + 1)*(1 - x**a)*(-x**c - x**(-2*a - 2*b + 1) + 1) - 4*x**(-a - 2*b + 2)*(1 - x**b) - 2*x**(-2*a - b - c + 2)*(1 - x**a)*(1 - x**c) + (1 - x)*(1 - x**a)*(1 - x**b)*(-x**c - x**(-2*a - 2*b + 1) + 1) ...
The minimum PF zero is 16.1863531899966 within margin of 0.01, attained around a=0.08359375000000002, b=0.1296875, c=0.436875, d=0.5 within margin of 0.001

Evaluating omega does not meet delta, chi does not meet delta: -4*x**(b + 1)*(1 - x**a)*(1 - 2*x**(-2*a - 2*b + 1)) - 4*x**(-a - 2*b + 2)*(1 - x**b) + (1 - x)*(1 - x**

In [106]:
Lemma_A_8_6 = [

# There exists omega exiting beta at the same vertex as mu
# # If omega meets gamma

("omega meets beta' - 1", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - 2 * x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # mu, mu'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) # nu, nu'
 - 2 * x ** (2 - a - b) * (1 - x ** (1 - 2 * a - 2 * b)), # omega, omega'
 12.9, 0.1, 0.1, 1),

("omega meets beta' - 2", # Using the inequality p' \leq p
 (1 - x) * (1 - x ** a) * (1 - x ** (1/2 - 3/2 * a)) * (1 - 2 * x ** a)
 - 2 * x ** (2 - a - 2 * (1/2 - 3/2 * a)) * (1 - x ** (1/2 - 3/2 * a)) # mu, mu'
 - 2 * x ** (2 - 2 * a - (1/2 - 3/2 * a)) * (1 - x ** a) # nu, nu'
 - 2 * x ** (2 - a - (1/2 - 3/2 * a)) * (1 - x ** a), # omega, omega'
 24.8, 0.28, 1, 1),

("omega does not meet beta'", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - 2 * x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # mu, mu'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) # nu, nu'
 - 2 * x ** (1 + a + b) * (1 - 2 * x ** (1 - 2 * a - 2 * b)), # omega, omega'
 15.9, 0.14, 0.14, 1),

# We can now assume that omega does not meet gamma
# There exists chi exiting gamma at the same vertex as nu
# We can assume that chi does not meet beta
    
("omega meets beta', chi meets beta' - 1", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - 2 * x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # mu, mu'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) # nu, nu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b)) # omega, omega'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * b)), # chi, chi'
 11.3, 0.08, 0.08, 1),

("omega meets beta', chi meets beta' - 2", # Using the inequality p' \leq p
 (1 - x) * (1 - x ** a) * (1 - x ** (1/2 - 3/2 * a)) * (1 - 2 * x ** a)
 - 2 * x ** (2 - a - 2 * (1/2 - 3/2 * a)) * (1 - x ** (1/2 - 3/2 * a)) # mu, mu'
 - 2 * x ** (2 - 2 * a - (1/2 - 3/2 * a)) * (1 - x ** a) # nu, nu'
 - 2 * x ** (2 - a - 2 * (1/2 - 3/2 * a)) * (1 - x ** (1/2 - 3/2 * a)) * (1 - x ** a) # omega, omega'
 - 2 * x ** (2 - 2 * a - (1/2 - 3/2 * a)) * (1 - x ** a) * (1 - x ** a), # chi, chi'
 26.4, 0.28, 1, 1),
    
("omega meets beta', chi does not meet beta' - 1", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - 2 * x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # mu, mu'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) # nu, nu'
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) * (1 - x ** (1 - 2 * a - 2 * b)) # omega, omega'
 - 2 * x ** (1 + b) * (1 - x ** a) * (1 - 2 * x ** (1 - 2 * a - 2 * b)), # chi, chi'
 13.7, 0.09, 0.13, 1),

("omega meets beta', chi does not meet beta' - 2", # Using the inequality p' \leq p
 (1 - x) * (1 - x ** a) * (1 - x ** (1/2 - 3/2 * a)) * (1 - 2 * x ** a)
 - 2 * x ** (2 - a - 2 * (1/2 - 3/2 * a)) * (1 - x ** (1/2 - 3/2 * a)) # mu, mu'
 - 2 * x ** (2 - 2 * a - (1/2 - 3/2 * a)) * (1 - x ** a) # nu, nu'
 - 2 * x ** (2 - a - 2 * (1/2 - 3/2 * a)) * (1 - x ** (1/2 - 3/2 * a)) * (1 - x ** a) # omega, omega'
 - 2 * x ** (1 + (1/2 - 3/2 * a)) * (1 - x ** a) * (1 - 2 * x ** a), # chi, chi'
 25.7, 0.27, 0.08, 1),

("omega does not meet beta', chi meets beta' - 1", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - 2 * x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # mu, mu'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) # nu, nu'
 - 2 * x ** (1 + a) * (1 - x ** b) * (1 - 2 * x ** (1 - 2 * a - 2 * b)) # omega, omega'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) * (1 - x ** (1 - 2 * a - 2 * b)), # chi, chi'
 13.7, 0.13, 0.09, 1),

("omega does not meet beta', chi meets beta' - 2", # Using the inequality p' \leq p
 (1 - x) * (1 - x ** a) * (1 - x ** (1/2 - 3/2 * a)) * (1 - 2 * x ** a)
 - 2 * x ** (2 - a - 2 * (1/2 - 3/2 * a)) * (1 - x ** (1/2 - 3/2 * a)) # mu, mu'
 - 2 * x ** (2 - 2 * a - (1/2 - 3/2 * a)) * (1 - x ** a) # nu, nu'
 - 2 * x ** (1 + a) * (1 - x ** (1/2 - 3/2 * a)) * (1 - 2 * x ** a) # omega, omega'
 - 2 * x ** (2 - 2 * a - (1/2 - 3/2 * a)) * (1 - x ** a) * (1 - x ** a), # chi, chi'
 26.3, 0.28, 0.08, 1),

("omega does not meet beta', chi does not meet beta'", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - 2 * x ** (1 - 2 * a - 2 * b))
 - 2 * x ** (2 - a - 2 * b) * (1 - x ** b) # mu, mu'
 - 2 * x ** (2 - 2 * a - b) * (1 - x ** a) # nu, nu'
 - 2 * x ** (1 + a) * (1 - x ** b) * (1 - 2 * x ** (1 - 2 * a - 2 * b)) # omega, omega'
 - 2 * x ** (1 + b) * (1 - x ** a) * (1 - 2 * x ** (1 - 2 * a - 2 * b)), # chi, chi'
 15.6, 0.13, 0.13, 1),
]

In [107]:
for i, (name, function, zero_start, a_start, b_start, c_start) in enumerate(Lemma_A_8_6):
    print(f"Evaluating {name}: {function} ...")
    MinPFRootApprox(function, starting_x = 1/zero_start, a_start = a_start, b_start = b_start, c_start = c_start, a_range = 0.01, b_range = 0.01, c_range = 0.01)

Evaluating omega meets beta' - 1: -2*x**(-2*a - b + 2)*(1 - x**a) - 2*x**(-a - 2*b + 2)*(1 - x**b) - 2*x**(-a - b + 2)*(1 - x**(-2*a - 2*b + 1)) + (1 - x)*(1 - x**a)*(1 - x**b)*(1 - 2*x**(-2*a - 2*b + 1)) ...
The minimum PF zero is 13.0043685064975 within margin of 0.01, attained around a=0.10437499999999997, b=0.10468749999999996, c=1, d=0.5 within margin of 0.001

Evaluating omega meets beta' - 2: -2*x**(1.5 - 0.5*a)*(1 - x**a) - 2*x**(0.5*a + 1.5)*(1 - x**a) - 2*x**(2.0*a + 1.0)*(1 - x**(0.5 - 1.5*a)) + (1 - x)*(1 - 2*x**a)*(1 - x**a)*(1 - x**(0.5 - 1.5*a)) ...
The minimum PF zero is 24.8749540122516 within margin of 0.01, attained around a=0.28, b=1, c=1, d=0.5 within margin of 0.001

Evaluating omega does not meet beta': -2*x**(-2*a - b + 2)*(1 - x**a) - 2*x**(-a - 2*b + 2)*(1 - x**b) - 2*x**(a + b + 1)*(1 - 2*x**(-2*a - 2*b + 1)) + (1 - x)*(1 - x**a)*(1 - x**b)*(1 - 2*x**(-2*a - 2*b + 1)) ...
The minimum PF zero is 15.9420124350407 within margin of 0.01, attained around a=0.14437

In [108]:
Lemma_A_8_7 = [
("gamma' meets beta and gamma and beta'", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c)
 - 2 * x ** (2 - 4 * a - b - 2 * c), # gamma', (gamma')'
 26.1, 0.07, 0.21, 0.12),
    
("gamma' meets beta and gamma but not beta'", # then there exists mu and nu meeting beta' and filaments
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c)
 - 2 * x ** (2 - 4 * a - b - 2 * c) * (1 - x ** c) # gamma', (gamma')'
 - 4 * x ** (1 + a + b + c) * (1 - x ** a) * (1 - x ** b), # mu, mu', nu, nu'
 15.9, 0.09, 0.36, 0.08),
    
("gamma' meets beta and beta' but not gamma", # then there exists mu and nu meeting gamma and filaments
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c)
 - 2 * x ** (2 - 4 * a - b - 2 * c) * (1 - x ** b) # gamma', (gamma')'
 - 4 * x ** (1 + a + b + c) * (1 - x ** a) * (1 - x ** c), # mu, mu', nu, nu'
 18.4, 0.08, 0.13, 0.16),

("gamma' meets gamma and beta' but not beta - 1", # then there exists mu and nu meeting beta and filaments
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c)
 - 2 * x ** (2 - 4 * a - b - 2 * c) * (1 - x ** a) # gamma', (gamma')'
 - 4 * x ** (1 + a + b + c) * (1 - x ** b) * (1 - x ** c), # mu, mu', nu, nu'
 14, 0.06, 0.44, 0.2),

("gamma' meets gamma and beta' but not beta - 2", # Using the inequality p' \leq p
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** a)
 - 2 * x ** (2 - 4 * a - b - 2 * a) * (1 - x ** a) # gamma', (gamma')'
 - 4 * x ** (1 + a + b + a) * (1 - x ** b) * (1 - x ** a), # mu, mu', nu, nu'
 15.9, 0.09, 0.36, 1),
    
("gamma' meets beta but not gamma and beta'",
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c)
 - 2 * x ** (2 - 4 * a - b - 2 * c) * (1 - x ** b) * (1 - x ** c) # gamma', (gamma')'
 - 2 * x ** (1 + b + c) * (1 - x ** a) # mu, mu'
 - 2 * x ** (1 + a + c) * (1 - x ** b), # nu, nu'
 18.5, 0.13, 0.17, 0.22),
    
("gamma' meets gamma but not beta and beta'", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c)
 - 2 * x ** (2 - 4 * a - b - 2 * c) * (1 - x ** a) * (1 - x ** c) # gamma', (gamma')'
 - 2 * x ** (1 + b + c) * (1 - x ** a) # mu, mu'
 - 2 * x ** (1 + a + c) * (1 - x ** b), # nu, nu'
 15.1, 0.09, 0.33, 0.28),
    
("gamma' meets beta' but not beta and gamma - 1", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c)
 - 2 * x ** (2 - 4 * a - b - 2 * c) * (1 - x ** a) * (1 - x ** b) # gamma', (gamma')'
 - 2 * x ** (1 + b + c) * (1 - x ** a) # mu, mu'
 - 2 * x ** (1 + a + c) * (1 - x ** b), # nu, nu'
 13.9, 0.07, 0.15, 0.43),

("gamma' meets beta' but not beta and gamma - 2", # Using the inequality p' \leq p
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** a)
 - 2 * x ** (2 - 4 * a - b - 2 * a) * (1 - x ** a) * (1 - x ** b) # gamma', (gamma')'
 - 2 * x ** (1 + b + a) * (1 - x ** a) # mu, mu'
 - 2 * x ** (1 + a + a) * (1 - x ** b), # nu, nu'
 19.1, 0.16, 0.2, 1),
]

In [109]:
for i, (name, function, zero_start, a_start, b_start, c_start) in enumerate(Lemma_A_8_7):
    print(f"Evaluating {name}: {function} ...")
    MinPFRootApprox(function, starting_x = 1/zero_start, a_start = a_start, b_start = b_start, c_start = c_start, a_range = 0.01, b_range = 0.01, c_range = 0.01)

Evaluating gamma' meets beta and gamma and beta': -2*x**(-4*a - b - 2*c + 2) + (1 - x)*(1 - x**a)*(1 - x**b)*(1 - x**c) ...
The minimum PF zero is 26.1794494065075 within margin of 0.01, attained around a=0.068125, b=0.21249999999999997, c=0.124375, d=0.5 within margin of 0.001

Evaluating gamma' meets beta and gamma but not beta': -2*x**(-4*a - b - 2*c + 2)*(1 - x**c) - 4*x**(a + b + c + 1)*(1 - x**a)*(1 - x**b) + (1 - x)*(1 - x**a)*(1 - x**b)*(1 - x**c) ...
The minimum PF zero is 15.9356770821643 within margin of 0.01, attained around a=0.08874999999999997, b=0.36031249999999987, c=0.084375, d=0.5 within margin of 0.001

Evaluating gamma' meets beta and beta' but not gamma: -2*x**(-4*a - b - 2*c + 2)*(1 - x**b) - 4*x**(a + b + c + 1)*(1 - x**a)*(1 - x**c) + (1 - x)*(1 - x**a)*(1 - x**b)*(1 - x**c) ...
The minimum PF zero is 18.4783108994529 within margin of 0.01, attained around a=0.08250000000000002, b=0.13124999999999998, c=0.15999999999999998, d=0.5 within margin of 0.001

Evaluat

In [110]:
Lemma_A_8_8 = [

# Case (1): beta' and gamma' are not petal curves
    
("mu does not meet beta' and gamma'", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - 2 * x ** (3/2 - 3/2 * a - 3/2 * b - 3/2 * c) - x ** c)
 - 2 * x ** (1 + a + c - (3/2 - 3/2 * a - 3/2 * b - 3/2 * c)) * (1 - x ** b) * (1 - 2 * x ** (3/2 - 3/2 * a - 3/2 * b - 3/2 * c)) # mu, mu'
 - 2 * x ** (1 + b + c) * (1 - x ** a) * (1 - 2 * x ** (3/2 - 3/2 * a - 3/2 * b - 3/2 * c)), # nu, nu'
 15.9, 0.11, 0.06, 0.54, 1),

("mu meets exactly one of beta' or gamma'", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** d - x ** (3 - 3 * a - 3 * b - 3 * c - d) - x ** c)
 - 2 * x ** (1 + a + c - d) * (1 - x ** b) * (1 - x ** d) # mu, mu'
 - 2 * x ** (1 + b + c) * (1 - x ** a) * (1 - x ** d - x ** (3 - 3 * a - 3 * b - 3 * c - d)), # nu, nu'
 16.7, 0.12, 0.06, 0.51, 0.32),

("mu and nu meet beta' and gamma'", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - 2 * x ** (3/2 - 3/2 * a - 3/2 * b - 3/2 * c) - x ** c)
 - 2 * x ** (1 + a + c) * (1 - x ** b) # mu, mu'
 - 2 * x ** (1 + b + c) * (1 - x ** a), # nu, nu'
 15.7, 0.09, 0.09, 0.43, 1),

# Case (2): beta' is a petal curve

# There exists omega that exits beta at the same vertex as mu
    
("omega meets beta and gamma and gamma' - 1", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** (3 - 3 * a - 3 * b - 4 * c) - x ** c)
 - 2 * x ** (1 + a + c) * (1 - x ** b) # mu, mu'
 - 2 * x ** (1 + b + c) * (1 - x ** a) # nu, nu'
 - 2 * x ** (1 + a + b + c), # omega, omega'
 14, 0.2, 0.2, 0.3, 1),

("omega meets beta and gamma and gamma' - 2", # Using q' \leq p'
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - 2 * x ** (3/5 - 3/5 * a - 3/5 * b))
 - 2 * x ** (1 + a + (3/5 - 3/5 * a - 3/5 * b)) * (1 - x ** b) # mu, mu'
 - 2 * x ** (1 + b + (3/5 - 3/5 * a - 3/5 * b)) * (1 - x ** a) # nu, nu'
 - 2 * x ** (1 + a + b + (3/5 - 3/5 * a - 3/5 * b)), # omega, omega'
 14.9, 0.17, 0.17, 1, 1),
    
("omega meets beta and gamma and beta' but not gamma'", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** (3 - 3 * a - 3 * b - 4 * c) - x ** c)
 - 2 * x ** (1 + a + c) * (1 - x ** b) # mu, mu'
 - 2 * x ** (1 + b + c) * (1 - x ** a) # nu, nu'
 - 2 * x ** (1 + a + b + c - (3 - 3 * a - 3 * b - 4 * c)) * (1 - x ** (3 - 3 * a - 3 * b - 4 * c)), # omega, omega'
 16.4, 0.2, 0.2, 0.36, 1),

# We can now assume that omega does not meet gamma
# then there exists chi that exits gamma at the same vertex as nu
    
("omega meets gamma', chi meets gamma'", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** (3 - 3 * a - 3 * b - 4 * c) - x ** c)
 - 2 * x ** (1 + a + c) * (1 - x ** b) # mu, mu'
 - 2 * x ** (1 + b + c) * (1 - x ** a) # nu, nu'
 - 2 * x ** (1 + a + c) * (1 - x ** b) # omega, omega'
 - 2 * x ** (1 + b + c) * (1 - x ** a), # chi, chi'
 14.5, 0.16, 0.16, 0.36, 1),

("Lemma 12.8 - omega meets gamma', chi does not meet gamma'", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** (3 - 3 * a - 3 * b - 4 * c) - x ** c)
 - 2 * x ** (1 + a + c) * (1 - x ** b) # mu, mu'
 - 2 * x ** (1 + b + c) * (1 - x ** a) # nu, nu'
 - 2 * x ** (1 + a + c) * (1 - x ** b) # omega, omega'
 - 2 * x ** (1 + b + c - (3 - 3 * a - 3 * b - 4 * c)) * (1 - x ** a) * (1 - x ** (3 - 3 * a - 3 * b - 4 * c) - x ** c), # chi, chi'
 15.2, 0.17, 0.18, 0.38, 1),

("Lemma 12.8 - omega meets beta' but not gamma'", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** (3 - 3 * a - 3 * b - 4 * c) - x ** c)
 - 2 * x ** (1 + a + c) * (1 - x ** b) # mu, mu'
 - 2 * x ** (1 + b + c) * (1 - x ** a) # nu, nu'
 - 2 * x ** (1 + a + c - (3 - 3 * a - 3 * b - 4 * c)) * (1 - x ** b) * (1 - x ** (3 - 3 * a - 3 * b - 4 * c)) # omega, omega'
 - 2 * x ** (1 + b + c) * (1 - x ** a) * (1 - x ** (3 - 3 * a - 3 * b - 4 * c) - x ** c), # chi, chi'
 15.3, 0.19, 0.13, 0.42, 1),

("Lemma 12.8 - omega does not meet beta', chi does not meet beta' - 1", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** (3 - 3 * a - 3 * b - 4 * c) - x ** c)
 - 2 * x ** (1 + a + c) * (1 - x ** b) # mu, mu'
 - 2 * x ** (1 + b + c) * (1 - x ** a) # nu, nu'
 - 2 * x ** (1 + a) * (1 - x ** b) * (1 - x ** (3 - 3 * a - 3 * b - 4 * c) - x ** c) # omega, omega'
 - 2 * x ** (1 + b) * (1 - x ** a) * (1 - x ** (3 - 3 * a - 3 * b - 4 * c) - x ** c), # chi, chi'
 14.1, 0.22, 0.22, 0.27, 1),
    
("Lemma 12.8 - omega does not meet beta', chi does not meet beta' - 2", # Using q' \leq p'
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - 2 * x ** (3/5 - 3/5 * a - 3/5 * b))
 - 2 * x ** (1 + a + (3/5 - 3/5 * a - 3/5 * b)) * (1 - x ** b) # mu, mu'
 - 2 * x ** (1 + b + (3/5 - 3/5 * a - 3/5 * b)) * (1 - x ** a) # nu, nu'
 - 2 * x ** (1 + a) * (1 - x ** b) * (1 - 2 * x ** (3/5 - 3/5 * a - 3/5 * b)) # omega, omega'
 - 2 * x ** (1 + b) * (1 - x ** a) * (1 - 2 * x ** (3/5 - 3/5 * a - 3/5 * b)), # chi, chi'
 14.9, 0.19, 0.19, 1, 1),    
]

In [111]:
for i, (name, function, zero_start, a_start, b_start, c_start, d_start) in enumerate(Lemma_A_8_8):
    print(f"Evaluating {name}: {function} ...")
    MinPFRootApprox(function, starting_x = 1/zero_start, a_start = a_start, b_start = b_start, c_start = c_start, d_start = d_start, a_range = 0.01, b_range = 0.01, c_range = 0.01, d_range = 0.01)

Evaluating mu does not meet beta' and gamma': -2*x**(b + c + 1)*(1 - x**a)*(1 - 2*x**(-1.5*a - 1.5*b - 1.5*c + 1.5)) - 2*x**(2.5*a + 1.5*b + 2.5*c - 0.5)*(1 - x**b)*(1 - 2*x**(-1.5*a - 1.5*b - 1.5*c + 1.5)) + (1 - x)*(1 - x**a)*(1 - x**b)*(-x**c - 2*x**(-1.5*a - 1.5*b - 1.5*c + 1.5) + 1) ...
The minimum PF zero is 15.9986548410518 within margin of 0.01, attained around a=0.11125000000000002, b=0.06125, c=0.545, d=1 within margin of 0.001

Evaluating mu meets exactly one of beta' or gamma': -2*x**(b + c + 1)*(1 - x**a)*(-x**d - x**(-3*a - 3*b - 3*c - d + 3) + 1) - 2*x**(a + c - d + 1)*(1 - x**b)*(1 - x**d) + (1 - x)*(1 - x**a)*(1 - x**b)*(-x**c - x**d - x**(-3*a - 3*b - 3*c - d + 3) + 1) ...
The minimum PF zero is 16.7388968157741 within margin of 0.01, attained around a=0.115, b=0.061875, c=0.505625, d=0.32125000000000004 within margin of 0.001

Evaluating mu and nu meet beta' and gamma': -2*x**(a + c + 1)*(1 - x**b) - 2*x**(b + c + 1)*(1 - x**a) + (1 - x)*(1 - x**a)*(1 - x**b)*(-x**c 

In [112]:
Lemma_A_8_10_1 = [
    
("beta' meets beta and gamma, gamma' meets at least one of beta and gamma", 
 (1 - x) * (1 - x ** a) * (1 - x ** b)
 - 2 * x ** d # beta', (beta')'
 - 2 * x ** (2 - a - b - d) * (1 - x ** a - 2 * x ** d), # gamma', (gamma')'
 17.4, 0.24, 0.47, 1, 0.77),  

("beta' meets beta and gamma, gamma' meets delta", 
 (1 - x) * (1 - x ** a) * (1 - x ** a) * (1 - x ** c)
 - 2 * x ** d * (1 - x ** c) # beta', (beta')'
 - 2 * x ** (2 - a - a - 2 * c - d) * ((1 - x ** a) * (1 - x ** a) - 2 * x ** d), # gamma', (gamma')'
 25.8, 0.21, 1, 0.13, 0.7), 

("beta' meets gamma and delta, gamma' meets beta", 
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c)
 - 2 * x ** d * (1 - x ** a) # beta', (beta')'
 - 2 * x ** (2 - a - b - 2 * c - d) * ((1 - x ** b) * (1 - x ** c) - 2 * x ** d), # gamma', (gamma')'
 25.8, 0.21, 0.21, 0.12, 0.83),

("beta' meets delta but not beta and gamma, gamma' meets beta but not gamma", # then there exists mu meeting gamma and filaments
 (1 - x) * (1 - x ** a) * (1 - x ** b) * (1 - x ** c)
 - 2 * x ** d * (1 - x ** a) * (1 - x ** b) # beta', (beta')'
 - 2 * x ** (2 - a - b - 2 * c - d) * (1 - x ** b) * (1 - x ** c - 2 * x ** d) # gamma', (gamma')'
 - 2 * x ** (1 + a + b + c) * (1 - x ** a - 2 * x ** (2 - a - b - 2 * c - d)) * (1 - x ** c - 2 * x ** d), # mu, mu'
 15.7, 0.27, 0.05, 0.15, 0.75),

# If beta' and gamma' meet delta but not beta and gamma
# then there exists mu meeting beta and filaments
    
("mu meets gamma", 
 (1 - x) * (1 - x ** a) * (1 - x ** a) * (1 - x ** c)
 - 2 * x ** (1 - 1/2 * a - 1/2 * a - c) * (1 - x ** a) * (1 - x ** a) # beta', (beta')'
 - 2 * x ** (1 - 1/2 * a - 1/2 * a - c) * (1 - x ** a) * (1 - x ** a) * (1 - 2 * x ** (1 - 1/2 * a - 1/2 * a - c)) # gamma', (gamma')'
 - 2 * x ** (1 + a + a + c) * ((1 - 2 * x ** (1 - 1/2 * a - 1/2 * a - c)) ** 2 - x ** c), # mu, mu'
 19.9, 0.09, 1, 0.29, 1),

("mu does not meet gamma", # then there exists nu meeting gamma and filaments, we can assume that nu does not meet beta
 (1 - x) * (1 - x ** a) * (1 - x ** a) * (1 - x ** c)
 - 2 * x ** (1 - 1/2 * a - 1/2 * a - c) * (1 - x ** a) * (1 - x ** a) # beta', (beta')'
 - 2 * x ** (1 - 1/2 * a - 1/2 * a - c) * (1 - x ** a) * (1 - x ** a) # gamma', (gamma')'
 - 2 * x ** (1 + a + c) * (1 - x ** a) * (1 - x ** c) * ((1 - 2 * x ** (1 - 1/2 * a - 1/2 * a - c)) ** 2 - x ** c) # mu, mu'
 - 2 * x ** (1 + a + c) * (1 - x ** a) * (1 - x ** c) * ((1 - 2 * x ** (1 - 1/2 * a - 1/2 * a - c)) ** 2 - x ** c), # nu, nu'
 20.7, 0.04, 1, 0.22, 1),

# If beta' and gamma' meet gamma but not beta
# then there exists mu meeting beta and filaments

("mu meets gamma and beta' and gamma'", 
 (1 - x) * (1 - x ** a) * (1 - x ** b)
 - 2 * x ** d * (1 - x ** a) # beta', (beta')'
 - 2 * x ** (3 - 3/2 * a - 3 * b - 3/2 * d) * (1 - x ** a) * (1 - 2 * x ** d) # gamma', (gamma')'
 - 2 * x ** (1 + a + b), # mu, mu'
 16.1, 0.17, 0.32, 1, 0.67),

("mu meets gamma and beta' but not gamma'", 
 (1 - x) * (1 - x ** a) * (1 - x ** b)
 - 2 * x ** d * (1 - x ** a) # beta', (beta')'
 - 2 * x ** (3 - 3/2 * a - 3 * b - 3/2 * d) * (1 - x ** a) * (1 - 2 * x ** d) # gamma', (gamma')'
 - 2 * x ** (1 + a + b - (3 - 3/2 * a - 3 * b - 3/2 * d)) * (1 - 2 * x ** (3 - 3/2 * a - 3 * b - 3/2 * d)), # mu, mu'
 20.8, 0.22, 0.4, 1, 0.62),

("mu meets gamma and gamma' but not beta'", 
 (1 - x) * (1 - x ** a) * (1 - x ** b)
 - 2 * x ** d * (1 - x ** a) # beta', (beta')'
 - 2 * x ** (3 - 3/2 * a - 3 * b - 3/2 * d) * (1 - x ** a) * (1 - 2 * x ** d) # gamma', (gamma')'
 - 2 * x ** (1 + a + b - d) * (1 - 2 * x ** d), # mu, mu'
 19.6, 0.21, 0.4, 1, 0.5),

("mu meets gamma but not beta' and gamma', beta' and gamma' meet - 1", # Bounding length of mu from disjointness with beta'
 (1 - x) * (1 - x ** a) * (1 - x ** b)
 - 2 * x ** d * (1 - x ** a) # beta', (beta')'
 - 2 * x ** (3 - 3/2 * a - 3 * b - 3/2 * d) * (1 - x ** a) # gamma', (gamma')'
 - 2 * x ** (1 + a + b - d) * (1 - 2 * x ** d - 2 * x ** (3 - 3/2 * a - 3 * b - 3/2 * d)), # mu, mu'
 21.3, 0.2, 0.36, 1, 0.55),

("mu meets gamma but not beta' and gamma', beta' and gamma' meet - 2", # Bounding length of mu from disjointness with gamma'
 (1 - x) * (1 - x ** a) * (1 - x ** b)
 - 2 * x ** d * (1 - x ** a) # beta', (beta')'
 - 2 * x ** (3 - 3/2 * a - 3 * b - 3/2 * d) * (1 - x ** a) # gamma', (gamma')'
 - 2 * x ** (1 + a + b - (3 - 3/2 * a - 3 * b - 3/2 * d)) * (1 - 2 * x ** d - 2 * x ** (3 - 3/2 * a - 3 * b - 3/2 * d)), # mu, mu'
 22.6, 0.21, 0.36, 1, 0.66),  
    
("mu meets gamma but not beta' and gamma', beta' and gamma' do not meet", 
 (1 - x) * (1 - x ** a) * (1 - x ** b)
 - 2 * x ** (6/5 - 3/5 * a - 6/5 * b) * (1 - x ** a) # beta', (beta')'
 - 2 * x ** (6/5 - 3/5 * a - 6/5 * b) * (1 - x ** a) * (1 - 2 * x ** (6/5 - 3/5 * a - 6/5 * b)) # gamma', (gamma')'
 - 2 * x ** (1 + a + b - 2 * (6/5 - 3/5 * a - 6/5 * b)) * (1 - 2 * x ** (6/5 - 3/5 * a - 6/5 * b)) ** 2, # mu, mu'
 26.1, 0.25, 0.49, 1, 1),  
    
("mu does not meet gamma, mu meets beta' and gamma'", 
 (1 - x) * (1 - x ** a) * (1 - x ** b)
 - 2 * x ** d * (1 - x ** a) # beta', (beta')'
 - 2 * x ** (3 - 3 * a - 3 * b - d) * (1 - x ** a) * (1 - 2 * x ** d) # gamma', (gamma')'
 - 2 * x ** (1 + a) * (1 - x ** b), # mu, mu'
 14.5, 0.15, 0.2, 1, 0.98),

("mu does not meet gamma, mu meets gamma' but not beta'", 
 (1 - x) * (1 - x ** a) * (1 - x ** b)
 - 2 * x ** d * (1 - x ** a) # beta', (beta')'
 - 2 * x ** (3 - 3 * a - 3 * b - d) * (1 - x ** a) * (1 - 2 * x ** d) # gamma', (gamma')'
 - 2 * x ** (1 + a - d) * (1 - x ** b) * (1 - 2 * x ** d), # mu, mu'
 24.9, 0.24, 0.31, 1, 0.52),

("mu does not meet gamma, mu does not meet beta' and gamma', beta' and gamma' meet", 
 (1 - x) * (1 - x ** a) * (1 - x ** b)
 - 2 * x ** d * (1 - x ** a) # beta', (beta')'
 - 2 * x ** (3 - 3 * a - 3 * b - d) * (1 - x ** a) # gamma', (gamma')'
 - 2 * x ** (1 + a - d) * (1 - x ** b) * (1 - 2 * x ** d - 2 * x ** (3 - 3 * a - 3 * b - d)), # mu, mu'
 25.7, 0.23, 0.3, 1, 0.56),
    
("mu does not meet gamma, mu does not meet beta' and gamma', beta' and gamma' do not meet", 
 (1 - x) * (1 - x ** a) * (1 - x ** b)
 - 2 * x ** (3/2 - 3/2 * a - 3/2 * b) * (1 - x ** a) # beta', (beta')'
 - 2 * x ** (3/2 - 3/2 * a - 3/2 * b) * (1 - x ** a) * (1 - 2 * x ** (3/2 - 3/2 * a - 3/2 * b)) # gamma', (gamma')'
 - 2 * x ** (1 + a - 2 * (3/2 - 3/2 * a - 3/2 * b)) * (1 - x ** b) * (1 - 2 * x ** (3/2 - 3/2 * a - 3/2 * b)) ** 2, # mu, mu'
 26.1, 0.3, 0.44, 1, 1),  
    
]

In [113]:
for i, (name, function, zero_start, a_start, b_start, c_start, d_start) in enumerate(Lemma_A_8_10_1):
    print(f"Evaluating {name}: {function} ...")
    MinPFRootApprox(function, starting_x = 1/zero_start, a_start = a_start, b_start = b_start, c_start = c_start, d_start = d_start, a_range = 0.01, b_range = 0.01, c_range = 0.01, d_range = 0.01)

Evaluating beta' meets beta and gamma, gamma' meets at least one of beta and gamma: -2*x**d - 2*x**(-a - b - d + 2)*(-x**a - 2*x**d + 1) + (1 - x)*(1 - x**a)*(1 - x**b) ...
The minimum PF zero is 17.4305590547630 within margin of 0.01, attained around a=0.24250000000000002, b=0.46812499999999996, c=1, d=0.7662499999999999 within margin of 0.001

Evaluating beta' meets beta and gamma, gamma' meets delta: -2*x**d*(1 - x**c) - 2*x**(-2*a - 2*c - d + 2)*(-2*x**d + (1 - x**a)**2) + (1 - x)*(1 - x**a)**2*(1 - x**c) ...
The minimum PF zero is 25.8711012861499 within margin of 0.01, attained around a=0.2125, b=1, c=0.125, d=0.7074999999999998 within margin of 0.001

Evaluating beta' meets gamma and delta, gamma' meets beta: -2*x**d*(1 - x**a) - 2*x**(-a - b - 2*c - d + 2)*(-2*x**d + (1 - x**b)*(1 - x**c)) + (1 - x)*(1 - x**a)*(1 - x**b)*(1 - x**c) ...
The minimum PF zero is 25.8709714561187 within margin of 0.01, attained around a=0.21312499999999995, b=0.21281249999999996, c=0.124687499999999

In [114]:
functions_case_6_2_3 = [

# Suppose beta' meets beta but not gamma, gamma' meet gamma but not beta
# There exists mu exiting beta from the same vertex as beta'
    
("Lemma 12.10 - beta' and gamma' meet", 
 (1 - x) * (1 - x ** a) * (1 - x ** a)
 - 2 * x ** (1 - 1/2 * a - 1/2 * a) * (1 - x ** a) # beta', (beta')'
 - 2 * x ** (1 - 1/2 * a - 1/2 * a) * (1 - x ** a), # gamma', (gamma')'
 17, 0.25, 1, 1, 1),  

("Lemma 12.10 - beta' and gamma' do not meet, mu meets gamma and gamma'", 
 (1 - x) * (1 - x ** a) * (1 - x ** a)
 - 2 * x ** (3/2 - 3/2 * a - 3/2 * a) * (1 - x ** a) # beta', (beta')'
 - 2 * x ** (3/2 - 3/2 * a - 3/2 * a) * (1 - x ** a) * (1 - 2 * x ** (3/2 - 3/2 * a - 3/2 * a)) # gamma', (gamma')'
 - 2 * x ** (1 + a + a), # mu, mu'
 16, 0.2, 1, 1, 1),

("Lemma 12.10 - beta' and gamma' do not meet, mu meets gamma but not gamma'", 
 (1 - x) * (1 - x ** a) * (1 - x ** a)
 - 2 * x ** d * (1 - x ** a) # beta', (beta')'
 - 2 * x ** (3 - 3 * a - 3 * a - d) * (1 - x ** a) * (1 - 2 * x ** d) # gamma', (gamma')'
 - 2 * x ** (1 + a + a - (3 - 3 * a - 3 * a - d)) * (1 - 2 * x ** (3 - 3 * a - 3 * a - d)), # mu, mu'
 23.1, 0.28, 1, 1, 0.78),

# We can now assume that mu does not meet gamma
# then there exists nu exiting gamma from the same vertex as gamma', we can assume nu does not meet beta
    
("Lemma 12.10 - beta' and gamma' do not meet, mu meets gamma', nu meets beta'", 
 (1 - x) * (1 - x ** a) * (1 - x ** a)
 - 2 * x ** (3/2 - 3/2 * a - 3/2 * a) * (1 - x ** a) # beta', (beta')'
 - 2 * x ** (3/2 - 3/2 * a - 3/2 * a) * (1 - x ** a) * (1 - 2 * x ** (3/2 - 3/2 * a - 3/2 * a)) # gamma', (gamma')'
 - 2 * x ** (1 + a) * (1 - x ** a) # mu, mu'
 - 2 * x ** (1 + a) * (1 - x ** a), # nu, nu'
 17.3, 0.18, 1, 1, 1),

("Lemma 12.10 - beta' and gamma' do not meet, mu meets gamma', nu does not meet beta'", 
 (1 - x) * (1 - x ** a) * (1 - x ** b)
 - 2 * x ** d * (1 - x ** b) # beta', (beta')'
 - 2 * x ** (3 - 3 * a - 3 * b - d) * (1 - x ** a) * (1 - 2 * x ** d) # gamma', (gamma')'
 - 2 * x ** (1 + a) * (1 - x ** b) # mu, mu'
 - 2 * x ** (1 + b - d) * (1 - x ** a) * (1 - 2 * x ** d), # nu, nu'
 27.1, 0.27, 0.27, 1, 0.56),

("Lemma 12.10 - beta' and gamma' do not meet, mu does not meet gamma', nu does not meet beta'", 
 (1 - x) * (1 - x ** a) * (1 - x ** a)
 - 2 * x ** (3/2 - 3/2 * a - 3/2 * a) * (1 - x ** a) # beta', (beta')'
 - 2 * x ** (3/2 - 3/2 * a - 3/2 * a) * (1 - x ** a) * (1 - 2 * x ** (3/2 - 3/2 * a - 3/2 * a)) # gamma', (gamma')'
 - 2 * x ** (1 + a - (3/2 - 3/2 * a - 3/2 * a)) * (1 - x ** a) * (1 - 2 * x ** (3/2 - 3/2 * a - 3/2 * a)) # mu, mu'
 - 2 * x ** (1 + a - (3/2 - 3/2 * a - 3/2 * a)) * (1 - x ** a) * (1 - 2 * x ** (3/2 - 3/2 * a - 3/2 * a)), # nu, nu'
 36.3, 0.32, 1, 1, 1),
]

In [115]:
for i, (name, function, zero_start, a_start, b_start, c_start, d_start) in enumerate(functions_case_6_2_3):
    print(f"Evaluating {name}: {function} ...")
    MinPFRootApprox(function, starting_x = 1/zero_start, a_start = a_start, b_start = b_start, c_start = c_start, d_start = d_start, a_range = 0.01, b_range = 0.01, c_range = 0.01, d_range = 0.01)

Evaluating Lemma 12.10 - beta' and gamma' meet: -4*x**(1 - 1.0*a)*(1 - x**a) + (1 - x)*(1 - x**a)**2 ...
The minimum PF zero is 17.0000156760608 within margin of 0.01, attained around a=0.245, b=1, c=1, d=1 within margin of 0.001

Evaluating Lemma 12.10 - beta' and gamma' do not meet, mu meets gamma and gamma': -2*x**(1.5 - 3.0*a)*(1 - x**a)*(1 - 2*x**(1.5 - 3.0*a)) - 2*x**(1.5 - 3.0*a)*(1 - x**a) - 2*x**(2*a + 1) + (1 - x)*(1 - x**a)**2 ...
The minimum PF zero is 16.0096987775689 within margin of 0.01, attained around a=0.203125, b=1, c=1, d=1 within margin of 0.001

Evaluating Lemma 12.10 - beta' and gamma' do not meet, mu meets gamma but not gamma': -2*x**d*(1 - x**a) - 2*x**(-6*a - d + 3)*(1 - x**a)*(1 - 2*x**d) - 2*x**(8*a + d - 2)*(1 - 2*x**(-6*a - d + 3)) + (1 - x)*(1 - x**a)**2 ...
The minimum PF zero is 23.1953943783652 within margin of 0.01, attained around a=0.2750000000000001, b=1, c=1, d=0.7787500000000002 within margin of 0.001

Evaluating Lemma 12.10 - beta' and gamma' d